# 🎨 NoobAI-XL EPS 1.1 – Bộ cài 1-click cho Google Colab

Cài **NoobAI-XL (NAI-XL) Epsilon-pred 1.1** trên nền **Stable Diffusion WebUI Forge** với đầy đủ tính năng,
giao diện Gradio chia sẻ qua **gradio.live** (hoặc ngrok).

| Có sẵn | Chi tiết |
|---|---|
| 🧠 Model | NoobAI-XL-v1.1.safetensors (7,1 GB, kiểm tra kích thước + SHA256) |
| 🎨 VAE | sdxl-vae-fp16-fix (chống ảnh đen/NaN khi chạy fp16) |
| 🔍 Upscaler | 4x-AnimeSharp, 4x-UltraSharp (Hires fix / Extras / Ultimate Upscale) |
| 🕹 ControlNet | ControlNet Union ProMax (1 file đủ canny/depth/openpose/lineart/tile…) hoặc bộ ControlNet riêng của NoobAI |
| 🧩 Extensions | Tag Autocomplete (dùng **file tag Danbooru+e621 của repo này**), ADetailer, Infinite Image Browsing, Ultimate SD Upscale, Dynamic Prompts, Civitai Helper (+ tuỳ chọn WD14 Tagger, Regional Prompter) |
| ⚙️ Mặc định NoobAI | Prompt chất lượng + negative chính thức, Euler a · 28 steps · CFG 6 · 832×1216 · Clip skip 2, 14 preset Styles |
| ☁️ Google Drive | Lưu ảnh ra Drive, tự liên kết thư mục `MyDrive/NoobAI/(models|Lora|embeddings|VAE|ControlNet)` |
| ✅ Tự kiểm tra | Cell kiểm tra môi trường, file model, phiên bản thư viện trước khi chạy |

**Yêu cầu:** Runtime → Change runtime type → **GPU** (T4 trở lên). Dung lượng đĩa cần ~15 GB (mặc định) – ~35 GB (tải hết ControlNet).

> ⚠️ Google Colab **miễn phí** có thể hạn chế/cảnh báo khi chạy WebUI (theo điều khoản của Colab). Khuyến nghị **Colab Pro**. Bộ cài này không vi phạm gì thêm – chỉ cần bạn tự chịu trách nhiệm với tài khoản của mình.

**Cách dùng nhanh:** chỉnh form ở cell 1 (hoặc để mặc định) → **Runtime ▸ Run all** → chờ đến khi hiện dòng `★ GIAO DIỆN ĐÃ SẴN SÀNG → https://xxxx.gradio.live` → bấm vào link.


In [ ]:
#@title 1️⃣ Cấu hình (chỉnh rồi chạy – có thể để mặc định) { display-mode: "form" }
#@markdown ### 🧠 Model & tài nguyên
DOWNLOAD_MODEL = True  #@param {type:"boolean"}
DOWNLOAD_VAE_FP16_FIX = True  #@param {type:"boolean"}
DOWNLOAD_UPSCALERS = True  #@param {type:"boolean"}
#@markdown `none` = không ControlNet · `union_promax` = 1 file 2,5 GB dùng được mọi loại · `noob_core` = 5 ControlNet riêng của NoobAI (12,5 GB) · `noob_all` = 9 file (22 GB)
CONTROLNET = "union_promax"  #@param ["none", "union_promax", "noob_core", "noob_all"]
#@markdown Link tải thêm (HuggingFace / Civitai…), nhiều link cách nhau bằng dấu phẩy. Link trang Civitai dạng `…/models/123?modelVersionId=456` cũng được.
EXTRA_MODEL_URLS = ""  #@param {type:"string"}
EXTRA_LORA_URLS = ""  #@param {type:"string"}
EXTRA_EMBEDDING_URLS = ""  #@param {type:"string"}
CIVITAI_TOKEN = ""  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}
VERIFY_SHA256 = False  #@param {type:"boolean"}

#@markdown ### 🧩 Extensions
EXT_TAGCOMPLETE = True  #@param {type:"boolean"}
USE_REPO_TAG_CSV = True  #@param {type:"boolean"}
EXT_ADETAILER = True  #@param {type:"boolean"}
EXT_IMAGE_BROWSER = True  #@param {type:"boolean"}
EXT_ULTIMATE_UPSCALE = True  #@param {type:"boolean"}
EXT_DYNAMIC_PROMPTS = True  #@param {type:"boolean"}
EXT_CIVITAI_HELPER = True  #@param {type:"boolean"}
EXT_WD14_TAGGER = False  #@param {type:"boolean"}
EXT_REGIONAL_PROMPTER = False  #@param {type:"boolean"}

#@markdown ### ☁️ Google Drive
MOUNT_DRIVE = True  #@param {type:"boolean"}
SAVE_OUTPUTS_TO_DRIVE = True  #@param {type:"boolean"}
LINK_DRIVE_MODELS = True  #@param {type:"boolean"}
DRIVE_FOLDER = "NoobAI"  #@param {type:"string"}

#@markdown ### 🌐 Giao diện / đường hầm
TUNNEL = "gradio.live"  #@param ["gradio.live", "ngrok", "both"]
NGROK_TOKEN = ""  #@param {type:"string"}
#@markdown Đặt mật khẩu cho link công khai (dạng `user:pass`), để trống nếu không cần
GRADIO_AUTH = ""  #@param {type:"string"}
THEME = "dark"  #@param ["dark", "light"]
#@markdown Ngôn ngữ giao diện: `vi` = tiếng Việt (song ngữ, giữ thuật ngữ Prompt/Seed/CFG…), `en` = tiếng Anh gốc
UI_LANGUAGE = "vi"  #@param ["vi", "en"]
USE_XFORMERS = True  #@param {type:"boolean"}
#@markdown Chế độ GPU: **max** = giữ toàn bộ model trên GPU, không hoán đổi ra RAM, VAE fp16, cuda-malloc (nhanh nhất; T4 15 GB vẫn đủ cho SDXL + Hires ×1.5 + ADetailer + 1 ControlNet). **balanced** = mặc định Forge. **lowvram** = ưu tiên ảnh/batch rất lớn (chậm hơn).
GPU_MODE = "max"  #@param ["max", "balanced", "lowvram"]
EXTRA_ARGS = ""  #@param {type:"string"}
#@markdown Commit Forge đã kiểm thử; đặt `latest` để lấy bản mới nhất (có thể phát sinh lỗi mới)
FORGE_COMMIT = "dfdcbab685e57677014f05a3309b48cc87383167"  #@param {type:"string"}

#@markdown ### 🎛 Thông số tạo ảnh
#@markdown **Khuyên dùng** = bộ chuẩn NoobAI (Euler a · 28 steps · CFG 6 · 832×1216 · Clip skip 2 · Hires fix ×1.5/0.4 · ADetailer mặt+mắt+tay) – các ô bên dưới sẽ bị bỏ qua. **Tự chỉnh** = dùng các ô bên dưới. Trong UI vẫn đổi được bằng dropdown ⚙️ phía trên Sampling method.
PARAM_MODE = "Khuyên dùng (NoobAI chuẩn)"  #@param ["Khuyên dùng (NoobAI chuẩn)", "Tự chỉnh (dùng các ô bên dưới)"]
#@markdown Độ phân giải chuẩn (bucket ~1 MP mà NoobAI-XL được huấn luyện). Trong UI vẫn đổi được bằng dropdown 📐 dưới Width/Height.
DEFAULT_RESOLUTION = "832x1216 (dọc 2:3 – chuẩn NoobAI)"  #@param ["832x1216 (dọc 2:3 – chuẩn NoobAI)", "1216x832 (ngang 3:2)", "1024x1024 (vuông)", "896x1152 (dọc 7:9)", "1152x896 (ngang 9:7)", "768x1344 (dọc 9:16)", "1344x768 (ngang 16:9)", "640x1536 (dọc 5:12)", "1536x640 (ngang 12:5)"]
#@markdown Hoặc tự nhập (ví dụ `1024x1536`), để trống để dùng mục trên
CUSTOM_RESOLUTION = ""  #@param {type:"string"}
DEFAULT_STEPS = 28  #@param {type:"integer"}
DEFAULT_CFG = 6.0  #@param {type:"number"}
DEFAULT_SAMPLER = "Euler a"  #@param ["Euler a", "Euler", "DPM++ 2M", "DPM++ 2M SDE", "DPM++ 3M SDE", "DPM++ SDE", "Restart"]
DEFAULT_SCHEDULER = "Automatic"  #@param ["Automatic", "Karras", "Exponential", "SGM Uniform", "Simple", "Normal"]
CLIP_SKIP = 2  #@param {type:"integer"}

#@markdown ### ✨ Tự fix (mở sẵn khi vào UI – tắt được từng ảnh bằng cách bỏ tích)
#@markdown Hires. fix: vẽ ảnh ở độ phân giải chuẩn rồi tự phóng to bằng 4x-AnimeSharp và vẽ thêm chi tiết
AUTO_HIRES_FIX = True  #@param {type:"boolean"}
HIRES_UPSCALE_BY = 1.5  #@param [1.25, 1.5, 1.75, 2.0] {type:"raw"}
HIRES_DENOISE = 0.4  #@param {type:"number"}
#@markdown ADetailer – mức tự sửa sau khi tạo ảnh: **standard** = 3 bộ mặt + mắt + tay (khuyên dùng) · **strong** = mắt/tay lỗi nặng (denoise cao hơn, vùng rộng hơn) · **light** = chỉ mặt (nhanh) · **multi** = ảnh nhiều người (nhận diện nhạy) · **off**. Trong UI đổi được bằng dropdown 🩹.
AUTO_ADETAILER = "standard"  #@param ["standard", "strong", "light", "multi", "off"]

import json, os, re
_res = CUSTOM_RESOLUTION.strip() or DEFAULT_RESOLUTION
_m = re.match(r"\s*(\d+)\s*[xX×]\s*(\d+)", _res)
assert _m, f"Độ phân giải không hợp lệ: {_res!r} (đúng dạng 832x1216)"
DEFAULT_WIDTH, DEFAULT_HEIGHT = (int(_m.group(1)) // 8 * 8, int(_m.group(2)) // 8 * 8)
extensions = [k for k, on in {
    "tagcomplete": EXT_TAGCOMPLETE, "adetailer": EXT_ADETAILER, "image_browser": EXT_IMAGE_BROWSER,
    "ultimate_upscale": EXT_ULTIMATE_UPSCALE, "dynamic_prompts": EXT_DYNAMIC_PROMPTS,
    "civitai_helper": EXT_CIVITAI_HELPER, "wd14_tagger": EXT_WD14_TAGGER, "regional_prompter": EXT_REGIONAL_PROMPTER,
}.items() if on]
config = dict(
    download_model=DOWNLOAD_MODEL, download_vae=DOWNLOAD_VAE_FP16_FIX, download_upscalers=DOWNLOAD_UPSCALERS,
    controlnet=CONTROLNET, extra_model_urls=EXTRA_MODEL_URLS, extra_lora_urls=EXTRA_LORA_URLS,
    extra_embedding_urls=EXTRA_EMBEDDING_URLS, civitai_token=CIVITAI_TOKEN.strip(), hf_token=HF_TOKEN.strip(),
    verify_sha256=VERIFY_SHA256, extensions=extensions, use_repo_tag_csv=USE_REPO_TAG_CSV,
    mount_drive=MOUNT_DRIVE, save_outputs_to_drive=SAVE_OUTPUTS_TO_DRIVE, link_drive_models=LINK_DRIVE_MODELS,
    drive_folder=DRIVE_FOLDER.strip() or "NoobAI", tunnel=TUNNEL, ngrok_token=NGROK_TOKEN.strip(),
    gradio_auth=GRADIO_AUTH.strip(), theme=THEME, ui_language=UI_LANGUAGE, use_xformers=USE_XFORMERS, gpu_mode=GPU_MODE, extra_args=EXTRA_ARGS.strip(),
    forge_commit=FORGE_COMMIT.strip(), default_width=DEFAULT_WIDTH, default_height=DEFAULT_HEIGHT,
    default_steps=DEFAULT_STEPS, default_cfg=DEFAULT_CFG, default_sampler=DEFAULT_SAMPLER,
    default_scheduler=DEFAULT_SCHEDULER, clip_skip=CLIP_SKIP,
    auto_hires_fix=AUTO_HIRES_FIX, hires_upscale_by=float(HIRES_UPSCALE_BY), hires_denoise=float(HIRES_DENOISE),
    auto_adetailer=AUTO_ADETAILER,
    param_mode="recommended" if PARAM_MODE.startswith("Khuyên") else "custom",
)
os.makedirs("/content", exist_ok=True)
with open("/content/noobai_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("✔ Đã lưu cấu hình vào /content/noobai_config.json")
print(json.dumps(config, indent=2, ensure_ascii=False))


In [ ]:
#@title 2️⃣ Ghi thư viện cài đặt (noobai_lib.py) – chỉ cần chạy, không cần sửa
%%writefile /content/noobai_lib.py
# -*- coding: utf-8 -*-
"""
NoobAI-XL EPS 1.1 – bộ cài tự động cho Google Colab (nền Stable Diffusion WebUI Forge).

File này là "nguồn duy nhất" của toàn bộ logic cài đặt. Notebook Colab chỉ:
  1. Ghi cấu hình (form) ra JSON
  2. Ghi file này ra /content/noobai_lib.py
  3. Gọi lần lượt: install() -> download() -> self_test() -> launch()

Có thể chạy độc lập ngoài Colab (dùng để test):
  NOOBAI_ROOT=/tmp/noob python noobai_lib.py all
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Iterable

# --------------------------------------------------------------------------------------
# Hằng số / tài nguyên
# --------------------------------------------------------------------------------------
LIB_VERSION = "1.7.0"

ROOT = Path(os.environ.get("NOOBAI_ROOT", "/content")).resolve()
CONFIG_PATH = Path(os.environ.get("NOOBAI_CONFIG", str(ROOT / "noobai_config.json")))

FORGE_REPO = "https://github.com/lllyasviel/stable-diffusion-webui-forge"
# Commit đã được kiểm thử với bộ cài này (26/06/2025). Đặt FORGE_COMMIT="" trong config để lấy bản mới nhất.
FORGE_COMMIT_TESTED = "dfdcbab685e57677014f05a3309b48cc87383167"

PYTHON_VERSION = "3.10"  # Forge chỉ hỗ trợ 3.7–3.11; Colab hiện là 3.12/3.13 nên phải tạo env riêng.
TORCH_SPEC_GPU = "torch==2.3.1 torchvision==0.18.1"
TORCH_INDEX_GPU = "https://download.pytorch.org/whl/cu121"
XFORMERS_SPEC = "xformers==0.0.27"
CLIP_PKG = "https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip"
OPENCLIP_PKG = "https://github.com/mlfoundations/open_clip/archive/bb6e834e9c70d9c27d0dc3ecedeebeaeb1ffad6b.zip"

TAG_CSV_NAME = "danbooru_e621_merged_2026-10-01_pt20-ia-dd-ed-spc.csv"
TAG_CSV_URLS = [
    f"https://github.com/manhlee1196-boop/anime2/raw/main/{TAG_CSV_NAME}",
    f"https://raw.githubusercontent.com/manhlee1196-boop/anime2/main/{TAG_CSV_NAME}",
    # GitHub API trả nội dung raw (hoạt động cả khi raw.githubusercontent.com bị chặn)
    f"https://api.github.com/repos/manhlee1196-boop/anime2/contents/{TAG_CSV_NAME}?ref=main",
]

# Bản dịch tiếng Việt cho giao diện Forge (file nằm cạnh noobai_lib.py trong Colab, hoặc tải từ repo)
VI_LOCALE_NAME = "vi_VN"
VI_LOCALE_URLS = [
    f"https://github.com/manhlee1196-boop/anime2/raw/main/colab/{VI_LOCALE_NAME}.json",
    f"https://raw.githubusercontent.com/manhlee1196-boop/anime2/main/colab/{VI_LOCALE_NAME}.json",
    f"https://api.github.com/repos/manhlee1196-boop/anime2/contents/colab/{VI_LOCALE_NAME}.json?ref=main",
]

# build_notebook.py sẽ chèn nội dung colab/vi_VN.json vào đây để notebook tự chứa bản dịch (không cần tải thêm)
VI_LOCALE_EMBEDDED: dict | None = {
 "Txt2img": "Văn bản → Ảnh (txt2img)",
 "Img2img": "Ảnh → Ảnh (img2img)",
 "Spaces": "Spaces (ứng dụng)",
 "Extras": "Công cụ thêm (Extras)",
 "PNG Info": "Thông tin ảnh (PNG Info)",
 "Checkpoint Merger": "Trộn model",
 "Train": "Huấn luyện",
 "Settings": "Cài đặt",
 "Extensions": "Tiện ích mở rộng",
 "UI": "Giao diện",
 "Checkpoint": "Model (Checkpoint)",
 "VAE / Text Encoder": "VAE / Bộ mã hoá văn bản",
 "Diffusion in Low Bits": "Độ chính xác (bits)",
 "Swap Method": "Cách hoán đổi bộ nhớ",
 "Swap Location": "Nơi hoán đổi",
 "GPU Weights (MB)": "Trọng số trên GPU (MB)",
 "Clip skip": "Clip skip",
 "Automatic": "Tự động",
 "Queue": "Hàng đợi",
 "Async": "Bất đồng bộ",
 "CPU": "CPU",
 "Shared": "Chia sẻ",
 "Stable Diffusion checkpoint": "Model Stable Diffusion",
 "SD VAE": "VAE",
 "Generate": "✨ Tạo ảnh",
 "Interrupt": "Dừng",
 "Interrupting...": "Đang dừng...",
 "Skip": "Bỏ qua",
 "Waiting...": "Đang chờ...",
 "Loading...": "Đang tải...",
 "Prompt": "Prompt (mô tả ảnh muốn tạo)",
 "Negative prompt": "Negative prompt (những gì KHÔNG muốn)",
 "Prompt (press Ctrl+Enter or Alt+Enter to generate)": "Prompt – mô tả ảnh muốn tạo (Ctrl+Enter để tạo)",
 "Negative prompt (press Ctrl+Enter or Alt+Enter to generate)": "Negative prompt – những gì KHÔNG muốn (Ctrl+Enter để tạo)",
 "Styles": "Kiểu (Styles)",
 "Apply all selected styles to prompt.": "Áp dụng các style đã chọn vào prompt.",
 "Apply all selected styles from the style selection dropdown in main UI to the prompt.": "Áp dụng toàn bộ style đã chọn vào prompt.",
 "Generation": "Thông số tạo ảnh",
 "Sampling method": "Thuật toán lấy mẫu (Sampler)",
 "Schedule type": "Kiểu lịch nhiễu (Scheduler)",
 "Sampling steps": "Số bước (Steps)",
 "Width": "Chiều rộng",
 "Height": "Chiều cao",
 "Batch count": "Số lượt (Batch count)",
 "Batch size": "Số ảnh mỗi lượt (Batch size)",
 "CFG Scale": "CFG Scale (độ bám prompt)",
 "Distilled CFG Scale": "Distilled CFG Scale",
 "Seed": "Seed (hạt giống ngẫu nhiên)",
 "Extra": "Nâng cao",
 "Variation seed": "Seed biến thể",
 "Variation strength": "Độ mạnh biến thể",
 "Resize seed from width": "Co giãn seed từ chiều rộng",
 "Resize seed from height": "Co giãn seed từ chiều cao",
 "Script": "Script",
 "None": "Không",
 "Refiner": "Refiner (tinh chỉnh)",
 "Switch at": "Chuyển tại",
 "(use model of same architecture)": "(dùng model cùng kiến trúc)",
 "Override settings": "Ghi đè cài đặt",
 "Restore faces": "Phục hồi khuôn mặt",
 "Tiling": "Lặp liền mạch (Tiling)",
 "Hires. fix": "Hires. fix (nâng độ phân giải)",
 "Upscaler": "Bộ phóng to (Upscaler)",
 "Hires steps": "Số bước Hires",
 "Denoising strength": "Mức khử nhiễu (Denoising)",
 "Upscale by": "Phóng to gấp",
 "Upscaled resolution": "Độ phân giải sau phóng to",
 "Resize width to": "Đổi chiều rộng thành",
 "Resize height to": "Đổi chiều cao thành",
 "Hires CFG Scale": "CFG Scale khi Hires",
 "Hires Distilled CFG Scale": "Distilled CFG khi Hires",
 "Hires Checkpoint": "Model cho Hires",
 "Use same checkpoint": "Dùng cùng model",
 "Hires VAE / Text Encoder": "VAE / Text Encoder cho Hires",
 "Hires sampling method": "Sampler cho Hires",
 "Use same sampler": "Dùng cùng sampler",
 "Hires schedule type": "Scheduler cho Hires",
 "Use same scheduler": "Dùng cùng scheduler",
 "Hires prompt": "Prompt cho Hires",
 "Hires negative prompt": "Negative prompt cho Hires",
 "Image for img2img": "Ảnh gốc cho img2img",
 "Sketch": "Vẽ phác (Sketch)",
 "Inpaint": "Vẽ lại vùng (Inpaint)",
 "Inpaint sketch": "Inpaint + vẽ phác",
 "Inpaint upload": "Inpaint tải mask lên",
 "Batch": "Hàng loạt",
 "Upload": "Tải lên",
 "From directory": "Từ thư mục",
 "Mask": "Mặt nạ (Mask)",
 "Files": "Tệp",
 "Input directory": "Thư mục đầu vào",
 "Output directory": "Thư mục đầu ra",
 "Inpaint batch mask directory (required for inpaint batch processing only)": "Thư mục mask cho inpaint hàng loạt (chỉ cần khi inpaint hàng loạt)",
 "Append png info to prompts": "Thêm thông tin PNG vào prompt",
 "PNG info directory": "Thư mục PNG info",
 "Leave empty to use input directory": "Để trống để dùng thư mục đầu vào",
 "Parameters to take from png info": "Thông số lấy từ PNG info",
 "Prompts from png info will be appended to prompts set in ui.": "Prompt từ PNG info sẽ được nối vào prompt trong UI.",
 "Resize mode": "Chế độ đổi kích thước",
 "Just resize": "Chỉ co giãn",
 "Crop and resize": "Cắt và co giãn",
 "Resize and fill": "Co giãn và lấp đầy",
 "Just resize (latent upscale)": "Chỉ co giãn (latent)",
 "Resize to": "Đổi thành kích thước",
 "Resize by": "Đổi theo tỉ lệ",
 "Scale": "Tỉ lệ",
 "Unused": "Không dùng",
 "Image CFG Scale": "CFG Scale của ảnh",
 "Mask blur": "Làm mờ mép mask",
 "Mask transparency": "Độ trong suốt mask",
 "Mask mode": "Chế độ mask",
 "Inpaint masked": "Vẽ lại vùng được tô",
 "Inpaint not masked": "Vẽ lại vùng KHÔNG tô",
 "Masked content": "Nội dung vùng tô",
 "fill": "lấp màu",
 "original": "giữ nguyên gốc",
 "latent noise": "nhiễu latent",
 "latent nothing": "latent trống",
 "Inpaint area": "Vùng inpaint",
 "Whole picture": "Toàn bộ ảnh",
 "Only masked": "Chỉ vùng tô",
 "Only masked padding, pixels": "Đệm quanh vùng tô (pixel)",
 "Soft inpainting": "Inpaint mềm",
 "Soft inpainting enabled": "Bật inpaint mềm",
 "Interrogate CLIP": "Đọc prompt bằng CLIP",
 "Interrogate DeepBooru": "Đọc tag bằng DeepBooru",
 "Interrogate\nCLIP": "Đọc prompt\nbằng CLIP",
 "Interrogate\nDeepBooru": "Đọc tag\nbằng DeepBooru",
 "Copy image to:": "Chép ảnh sang:",
 "img2img": "img2img",
 "sketch": "sketch",
 "inpaint": "inpaint",
 "inpaint sketch": "inpaint sketch",
 "Save": "Lưu",
 "Zip": "Nén Zip",
 "Send to img2img": "Gửi sang img2img",
 "Send to inpaint": "Gửi sang inpaint",
 "Send to extras": "Gửi sang Extras",
 "Send to txt2img": "Gửi sang txt2img",
 "Open images output directory": "Mở thư mục ảnh đầu ra",
 "Save the image to a dedicated directory (log/images).": "Lưu ảnh vào thư mục riêng (log/images).",
 "Save zip archive with images to a dedicated directory (log/images)": "Lưu file zip chứa ảnh vào thư mục riêng (log/images)",
 "Single Image": "Một ảnh",
 "Batch Process": "Xử lý hàng loạt",
 "Batch from Directory": "Hàng loạt từ thư mục",
 "Show result images": "Hiện ảnh kết quả",
 "A directory on the same machine where the server is running.": "Thư mục trên máy đang chạy server.",
 "Leave blank to save images to the default path.": "Để trống để lưu vào đường dẫn mặc định.",
 "Upscale": "Phóng to",
 "Upscaler 1": "Upscaler 1",
 "Upscaler 2": "Upscaler 2",
 "Upscaler 2 visibility": "Mức ảnh hưởng Upscaler 2",
 "Resize": "Kích thước",
 "Max side length": "Cạnh dài tối đa",
 "Scale by": "Phóng theo tỉ lệ",
 "Scale to": "Phóng tới kích thước",
 "Crop to fit": "Cắt cho vừa",
 "GFPGAN": "GFPGAN (phục hồi mặt)",
 "CodeFormer": "CodeFormer (phục hồi mặt)",
 "Visibility": "Mức ảnh hưởng",
 "Weight (0 = maximum effect, 1 = minimum effect)": "Trọng số (0 = tác động mạnh nhất, 1 = yếu nhất)",
 "Auto focal point crop": "Tự cắt theo tiêu điểm",
 "Create debug image": "Tạo ảnh gỡ lỗi",
 "Split oversized images": "Tách ảnh quá khổ",
 "Auto-sized crop": "Cắt tự động kích thước",
 "Create flipped copies": "Tạo bản lật ngang",
 "Caption": "Chú thích",
 "Source": "Nguồn",
 "Output": "Đầu ra",
 "Checkpoints": "Model (Checkpoints)",
 "Lora": "LoRA",
 "Textual Inversion": "Embedding (Textual Inversion)",
 "Hypernetworks": "Hypernetworks",
 "Search...": "Tìm kiếm...",
 "Search": "Tìm kiếm",
 "Sort: ": "Sắp xếp: ",
 "Name": "Tên",
 "Date Created": "Ngày tạo",
 "Date Modified": "Ngày sửa",
 "Path": "Đường dẫn",
 "Refresh": "Làm mới",
 "Close": "Đóng",
 "Cancel": "Huỷ",
 "Apply": "Áp dụng",
 "Delete": "Xoá",
 "Replace preview": "Thay ảnh xem trước",
 "Save preview": "Lưu ảnh xem trước",
 "Preview save filename": "Tên file ảnh xem trước",
 "Edit user metadata": "Sửa thông tin (metadata)",
 "Edit metadata": "Sửa thông tin",
 "Description": "Mô tả",
 "Notes": "Ghi chú",
 "Base model": "Model gốc",
 "Preferred VAE / Text encoder(s)": "VAE / Text encoder ưu tiên",
 "Unknown": "Không rõ",
 "Built in": "Có sẵn",
 "Activation text": "Từ kích hoạt (trigger words)",
 "Will be added to prompt along with Lora": "Sẽ được thêm vào prompt cùng LoRA",
 "Preferred weight": "Trọng số ưu tiên",
 "Set to 0 to disable": "Đặt 0 để tắt",
 "Will be added to negative prompts": "Sẽ được thêm vào negative prompt",
 "Random prompt": "Prompt ngẫu nhiên",
 "Add network to prompt": "Thêm vào prompt",
 "Training dataset tags": "Tag trong tập huấn luyện",
 "Show/hide extra networks": "Hiện/ẩn LoRA & mạng bổ sung",
 "Installed": "Đã cài",
 "Available": "Có thể cài",
 "Install from URL": "Cài từ URL",
 "Backup/Restore": "Sao lưu/Khôi phục",
 "Check for updates": "Kiểm tra cập nhật",
 "Apply and restart UI": "Áp dụng và khởi động lại UI",
 "Disable all extensions": "Tắt tất cả tiện ích",
 "Load from:": "Tải danh sách từ:",
 "Extension index URL": "URL danh mục tiện ích",
 "Extension tags": "Nhãn tiện ích",
 "Hide extensions with tags": "Ẩn tiện ích có nhãn",
 "Order": "Thứ tự",
 "newest first": "mới nhất trước",
 "oldest first": "cũ nhất trước",
 "internal order": "thứ tự nội bộ",
 "update time": "thời gian cập nhật",
 "create time": "thời gian tạo",
 "Showing type": "Hiển thị loại",
 "Filtering type": "Lọc loại",
 "URL for extension's git repository": "URL kho git của tiện ích",
 "URL for extension": "URL tiện ích",
 "Specific branch name": "Tên nhánh cụ thể",
 "Leave empty for default main branch": "Để trống dùng nhánh chính",
 "Local directory name": "Tên thư mục cục bộ",
 "Leave empty for auto": "Để trống để tự động",
 "Install": "Cài",
 "Saved Configs": "Cấu hình đã lưu",
 "Current": "Hiện tại",
 "State to restore": "Trạng thái khôi phục",
 "Restore Selected Config": "Khôi phục cấu hình đã chọn",
 "Config Name": "Tên cấu hình",
 "Save Current Config": "Lưu cấu hình hiện tại",
 "Extension": "Tiện ích",
 "URL": "URL",
 "Branch": "Nhánh",
 "Version": "Phiên bản",
 "Date": "Ngày",
 "Update": "Cập nhật",
 "Use checkbox to enable the extension; it will be enabled or disabled when you click apply button": "Tích để bật tiện ích; thay đổi có hiệu lực khi bấm Áp dụng",
 "Use checkbox to mark the extension for update; it will be updated when you click apply button": "Tích để đánh dấu cập nhật; sẽ cập nhật khi bấm Áp dụng",
 "Apply settings": "Áp dụng cài đặt",
 "Reload UI": "Tải lại UI",
 "Defaults": "Mặc định",
 "Sysinfo": "Thông tin hệ thống",
 "Actions": "Hành động",
 "Licenses": "Giấy phép",
 "Show all pages": "Hiện tất cả trang",
 "Show only one page": "Chỉ hiện một trang",
 "Unload all models": "Giải phóng tất cả model",
 "Load SD checkpoint to VRAM from RAM": "Nạp model từ RAM lên VRAM",
 "Calculate hash for all checkpoint": "Tính hash cho tất cả model",
 "Request browser notifications": "Bật thông báo trình duyệt",
 "Download localization template": "Tải mẫu file dịch",
 "Reload custom script bodies (No ui updates, No restart)": "Nạp lại script tuỳ chỉnh (không đổi UI, không khởi động lại)",
 "Check system info for validity": "Kiểm tra thông tin hệ thống",
 "Localization": "Ngôn ngữ (Localization)",
 "Quicksettings list": "Danh sách cài đặt nhanh (thanh trên cùng)",
 "Gradio theme": "Giao diện Gradio",
 "Saving images/grids": "Lưu ảnh / lưới ảnh",
 "Paths for saving": "Đường dẫn lưu",
 "Saving to a directory": "Lưu vào thư mục",
 "Upscaling": "Phóng to ảnh",
 "Face restoration": "Phục hồi khuôn mặt",
 "System": "Hệ thống",
 "Profiler": "Đo hiệu năng",
 "API": "API",
 "Training": "Huấn luyện",
 "Stable Diffusion": "Stable Diffusion",
 "Stable Diffusion XL": "Stable Diffusion XL",
 "Stable Diffusion 3": "Stable Diffusion 3",
 "VAE": "VAE",
 "Optimizations": "Tối ưu hoá",
 "Compatibility": "Tương thích",
 "Interrogate": "Đọc prompt từ ảnh",
 "Extra Networks": "Mạng bổ sung (LoRA...)",
 "Prompt editing": "Soạn prompt",
 "Gallery": "Thư viện ảnh",
 "UI alternatives": "Tuỳ chọn giao diện",
 "User interface": "Giao diện người dùng",
 "Infotext": "Thông tin sinh ảnh (Infotext)",
 "Live previews": "Xem trước trực tiếp",
 "Sampler parameters": "Tham số sampler",
 "Postprocessing": "Hậu xử lý",
 "Settings in UI": "Cài đặt trong UI",
 "Settings for txt2img": "Cài đặt cho txt2img",
 "Settings for img2img": "Cài đặt cho img2img",
 "UI defaults 'sd'": "Mặc định UI 'sd'",
 "UI defaults 'xl'": "Mặc định UI 'xl'",
 "UI defaults 'flux'": "Mặc định UI 'flux'",
 "txt2img sampler": "Sampler txt2img",
 "txt2img scheduler": "Scheduler txt2img",
 "img2img sampler": "Sampler img2img",
 "img2img scheduler": "Scheduler img2img",
 "txt2img width": "Chiều rộng txt2img",
 "txt2img height": "Chiều cao txt2img",
 "txt2img CFG": "CFG txt2img",
 "txt2img HiRes CFG": "CFG Hires txt2img",
 "img2img width": "Chiều rộng img2img",
 "img2img height": "Chiều cao img2img",
 "img2img CFG": "CFG img2img",
 "Always save all generated images": "Luôn lưu mọi ảnh đã tạo",
 "File format for images": "Định dạng file ảnh",
 "Images filename pattern": "Mẫu tên file ảnh",
 "Add number to filename when saving": "Thêm số thứ tự vào tên file",
 "Always save all generated image grids": "Luôn lưu lưới ảnh",
 "File format for grids": "Định dạng file lưới",
 "Do not save grids consisting of one picture": "Không lưu lưới chỉ có 1 ảnh",
 "Save a copy of image before applying highres fix.": "Lưu bản sao ảnh trước khi Hires fix.",
 "Save a copy of image before doing face restoration.": "Lưu bản sao ảnh trước khi phục hồi mặt.",
 "Quality for saved jpeg and avif images": "Chất lượng ảnh JPEG/AVIF",
 "Output directory for images; if empty, defaults to three directories below": "Thư mục lưu ảnh; để trống sẽ dùng 3 thư mục bên dưới",
 "Output directory for grids; if empty, defaults to two directories below": "Thư mục lưu lưới; để trống sẽ dùng 2 thư mục bên dưới",
 "Save images to a subdirectory": "Lưu ảnh vào thư mục con",
 "Save grids to a subdirectory": "Lưu lưới vào thư mục con",
 "Directory name pattern": "Mẫu tên thư mục",
 "Upscaler for img2img": "Upscaler cho img2img",
 "Face restoration model": "Model phục hồi mặt",
 "CodeFormer weight": "Trọng số CodeFormer",
 "Show warnings in console.": "Hiện cảnh báo trong console.",
 "Emphasis mode": "Chế độ nhấn mạnh (từ:1.2)",
 "Prompt word wrap length limit": "Giới hạn xuống dòng prompt",
 "Random number generator source.": "Nguồn sinh số ngẫu nhiên.",
 "Noise multiplier for img2img": "Hệ số nhiễu img2img",
 "Apply color correction to img2img results to match original colors.": "Hiệu chỉnh màu img2img cho khớp ảnh gốc.",
 "Cross attention optimization": "Tối ưu cross attention",
 "Token merging ratio": "Tỉ lệ gộp token",
 "Show grid in gallery": "Hiện lưới trong thư viện",
 "Gallery height": "Chiều cao thư viện",
 "Compact prompt layout": "Bố cục prompt gọn",
 "Hires fix: show hires checkpoint and sampler selection": "Hires fix: hiện chọn model & sampler",
 "Hires fix: show hires prompt and negative prompt": "Hires fix: hiện prompt & negative riêng",
 "Show generation progress in window title.": "Hiện tiến độ trên tiêu đề cửa sổ.",
 "Write infotext to metadata of the generated image": "Ghi infotext vào metadata ảnh",
 "Create a text file with infotext next to every generated image": "Tạo file .txt infotext cạnh mỗi ảnh",
 "Add model name to infotext": "Thêm tên model vào infotext",
 "Add model hash to infotext": "Thêm hash model vào infotext",
 "Show progressbar": "Hiện thanh tiến độ",
 "Show live previews of the created image": "Hiện xem trước ảnh đang tạo",
 "Live preview display period": "Chu kỳ cập nhật xem trước",
 "Live preview method": "Phương pháp xem trước",
 "Progressbar and preview update period": "Chu kỳ cập nhật tiến độ",
 "Hide samplers in user interface": "Ẩn sampler trong UI",
 "Eta noise seed delta": "ENSD (Eta noise seed delta)",
 "Always discard next-to-last sigma": "Luôn bỏ sigma áp chót",
 "Enable postprocessing operations in txt2img and img2img tabs": "Bật hậu xử lý trong txt2img/img2img",
 "Disable these extensions": "Tắt các tiện ích này",
 "Disable all extensions (preserves the list of disabled extensions)": "Tắt tất cả tiện ích (giữ danh sách đã tắt)",
 "Maximum number of checkpoints loaded at the same time": "Số model tối đa nạp cùng lúc",
 "Checkpoints to cache in RAM": "Số model cache trong RAM",
 "Number of Lora networks to keep cached in memory": "Số LoRA giữ trong bộ nhớ",
 "Add Lora hashes to infotext": "Thêm hash LoRA vào infotext",
 "Default multiplier for extra networks": "Trọng số mặc định cho LoRA",
 "Card width for Extra Networks": "Chiều rộng thẻ LoRA",
 "Card height for Extra Networks": "Chiều cao thẻ LoRA",
 "Show description on card": "Hiện mô tả trên thẻ",
 "Enable": "Bật",
 "Low VRAM": "VRAM thấp",
 "Pixel Perfect": "Pixel Perfect (tự tính độ phân giải)",
 "Allow Preview": "Cho phép xem trước",
 "Use Mask": "Dùng mask",
 "Preview as Input": "Dùng ảnh xem trước làm đầu vào",
 "Upload independent control image": "Tải ảnh điều khiển riêng",
 "Control Type": "Loại điều khiển",
 "All": "Tất cả",
 "Preprocessor": "Bộ tiền xử lý",
 "Model": "Model",
 "Control Weight": "Trọng số điều khiển",
 "Timestep Range": "Khoảng bước áp dụng",
 "Starting Control Step": "Bước bắt đầu",
 "Ending Control Step": "Bước kết thúc",
 "Preprocessor resolution": "Độ phân giải tiền xử lý",
 "Threshold A": "Ngưỡng A",
 "Threshold B": "Ngưỡng B",
 "Control Mode": "Chế độ điều khiển",
 "Balanced": "Cân bằng",
 "My prompt is more important": "Ưu tiên prompt",
 "ControlNet is more important": "Ưu tiên ControlNet",
 "Resize Mode": "Chế độ co giãn",
 "Just Resize": "Chỉ co giãn",
 "Crop and Resize": "Cắt và co giãn",
 "Resize and Fill": "Co giãn và lấp đầy",
 "Hires-Fix Option": "Tuỳ chọn Hires-Fix",
 "Both": "Cả hai",
 "Low res only": "Chỉ độ phân giải thấp",
 "High res only": "Chỉ độ phân giải cao",
 "ControlNet Integrated": "ControlNet (tích hợp)",
 "ControlNet Unit 0": "ControlNet Đơn vị 0",
 "ControlNet Unit 1": "ControlNet Đơn vị 1",
 "ControlNet Unit 2": "ControlNet Đơn vị 2",
 "ControlNet Unit 0 [Canny]": "ControlNet Đơn vị 0 [Canny]",
 "Image": "Ảnh",
 "Batch Folder": "Thư mục hàng loạt",
 "Batch Upload": "Tải hàng loạt",
 "Input Directory": "Thư mục đầu vào",
 "Mask Directory": "Thư mục mask",
 "Masks": "Mask",
 "Open New Canvas": "Mở khung vẽ mới",
 "New Canvas Width": "Chiều rộng khung vẽ",
 "New Canvas Height": "Chiều cao khung vẽ",
 "Create New Canvas": "Tạo khung vẽ",
 "Close Preview": "Đóng xem trước",
 "Run preprocessor": "Chạy tiền xử lý",
 "Preprocessor Preview": "Xem trước tiền xử lý",
 "Set the preprocessor to [invert] If your image has white background and black lines.": "Chọn tiền xử lý [invert] nếu ảnh nền trắng nét đen.",
 "Canny": "Canny (viền)",
 "Depth": "Depth (chiều sâu)",
 "OpenPose": "OpenPose (dáng người)",
 "Lineart": "Lineart (nét vẽ)",
 "SoftEdge": "SoftEdge (viền mềm)",
 "Scribble": "Scribble (nguệch ngoạc)",
 "Tile": "Tile (chi tiết/phóng to)",
 "Reference": "Reference (tham chiếu)",
 "ADetailer": "ADetailer (tự sửa mặt/tay)",
 "Enable ADetailer": "Bật ADetailer",
 "Skip img2img": "Bỏ qua img2img",
 "ADetailer detector": "Model phát hiện",
 "Select a model to use for detection.": "Chọn model phát hiện.",
 "ADetailer detector classes": "Lớp đối tượng cần phát hiện",
 "ADetailer prompt": "Prompt cho ADetailer",
 "ADetailer negative prompt": "Negative prompt cho ADetailer",
 "Detection": "Phát hiện",
 "Detection model confidence threshold": "Ngưỡng tin cậy phát hiện",
 "Mask only the top k (0 to disable)": "Chỉ giữ k mask tốt nhất (0 = tắt)",
 "Mask min area ratio": "Tỉ lệ diện tích mask tối thiểu",
 "Mask max area ratio": "Tỉ lệ diện tích mask tối đa",
 "Mask Preprocessing": "Tiền xử lý mask",
 "Mask x(→) offset": "Dịch mask theo x (→)",
 "Mask y(↑) offset": "Dịch mask theo y (↑)",
 "Mask erosion (-) / dilation (+)": "Co (-) / nở (+) mask",
 "Mask merge mode": "Chế độ gộp mask",
 "Inpainting": "Vẽ lại (Inpainting)",
 "Inpaint mask blur": "Làm mờ mép mask",
 "Inpaint denoising strength": "Mức khử nhiễu khi vẽ lại",
 "Inpaint only masked": "Chỉ vẽ lại vùng mask",
 "Inpaint only masked padding, pixels": "Đệm quanh vùng mask (pixel)",
 "Use separate width/height": "Dùng kích thước riêng",
 "inpaint width": "chiều rộng inpaint",
 "inpaint height": "chiều cao inpaint",
 "Use separate steps": "Dùng số bước riêng",
 "ADetailer steps": "Số bước ADetailer",
 "Use separate CFG scale": "Dùng CFG riêng",
 "ADetailer CFG scale": "CFG ADetailer",
 "Use separate checkpoint": "Dùng model riêng",
 "ADetailer checkpoint": "Model ADetailer",
 "Use separate VAE": "Dùng VAE riêng",
 "ADetailer VAE": "VAE ADetailer",
 "Use separate sampler": "Dùng sampler riêng",
 "ADetailer sampler": "Sampler ADetailer",
 "ADetailer scheduler": "Scheduler ADetailer",
 "Use separate noise multiplier": "Dùng hệ số nhiễu riêng",
 "Use separate CLIP skip": "Dùng CLIP skip riêng",
 "ADetailer CLIP skip": "CLIP skip ADetailer",
 "Restore faces after ADetailer": "Phục hồi mặt sau ADetailer",
 "ControlNet model": "Model ControlNet",
 "ControlNet module": "Module ControlNet",
 "ControlNet weight": "Trọng số ControlNet",
 "ControlNet guidance start": "ControlNet bắt đầu",
 "ControlNet guidance end": "ControlNet kết thúc",
 "Ultimate SD upscale": "Ultimate SD Upscale (phóng to theo ô)",
 "Target size type": "Kiểu kích thước đích",
 "From img2img2 settings": "Theo cài đặt img2img",
 "Custom size": "Kích thước tuỳ chỉnh",
 "Scale from image size": "Theo tỉ lệ ảnh gốc",
 "Custom width": "Chiều rộng tuỳ chỉnh",
 "Custom height": "Chiều cao tuỳ chỉnh",
 "Type": "Kiểu",
 "Tile width": "Chiều rộng ô",
 "Tile height": "Chiều cao ô",
 "Padding": "Đệm",
 "Seams fix": "Sửa đường nối",
 "Denoise": "Khử nhiễu",
 "Upscaled": "Đã phóng to",
 "Save options": "Tuỳ chọn lưu",
 "X/Y/Z plot": "Biểu đồ X/Y/Z (so sánh thông số)",
 "X type": "Trục X",
 "X values": "Giá trị X",
 "Y type": "Trục Y",
 "Y values": "Giá trị Y",
 "Z type": "Trục Z",
 "Z values": "Giá trị Z",
 "Draw legend": "Vẽ chú thích",
 "Keep -1 for seeds": "Giữ seed -1",
 "Include Sub Images": "Kèm ảnh con",
 "Include Sub Grids": "Kèm lưới con",
 "Use text inputs instead of dropdowns": "Nhập văn bản thay vì chọn",
 "Grid margins (px)": "Lề lưới (px)",
 "Swap X/Y axes": "Đổi trục X/Y",
 "Swap Y/Z axes": "Đổi trục Y/Z",
 "Swap X/Z axes": "Đổi trục X/Z",
 "Prompt matrix": "Ma trận prompt",
 "Prompts from file or textbox": "Prompt từ file hoặc ô nhập",
 "List of prompt inputs": "Danh sách prompt",
 "Upload prompt inputs": "Tải file prompt lên",
 "Iterate seed every line": "Tăng seed mỗi dòng",
 "Use same random seed for all lines": "Dùng cùng seed cho mọi dòng",
 "Loopback": "Lặp lại (Loopback)",
 "Loops": "Số vòng lặp",
 "Final denoising strength": "Khử nhiễu vòng cuối",
 "Denoising strength curve": "Đường cong khử nhiễu",
 "Outpainting mk2": "Mở rộng ảnh (Outpainting mk2)",
 "Poor man's outpainting": "Mở rộng ảnh đơn giản",
 "Pixels to expand": "Số pixel mở rộng",
 "Outpainting direction": "Hướng mở rộng",
 "left": "trái",
 "right": "phải",
 "up": "trên",
 "down": "dưới",
 "SD upscale": "SD upscale",
 "Tile overlap": "Chồng lấn ô",
 "Scale Factor": "Hệ số phóng",
 "Put variable parts at start of prompt": "Đặt phần biến đổi ở đầu prompt",
 "Use different seed for each picture": "Seed khác nhau cho mỗi ảnh",
 "FreeU Integrated": "FreeU (tích hợp)",
 "Kohya HRFix Integrated": "Kohya HRFix (tích hợp)",
 "MultiDiffusion Integrated": "MultiDiffusion (tích hợp)",
 "Self Attention Guidance Integrated": "Self Attention Guidance (tích hợp)",
 "PerturbedAttentionGuidance Integrated": "Perturbed Attention Guidance (tích hợp)",
 "DynamicThresholding (CFG-Fix) Integrated": "Dynamic Thresholding (CFG-Fix, tích hợp)",
 "LatentModifier Integrated": "Latent Modifier (tích hợp)",
 "StyleAlign Integrated": "StyleAlign (tích hợp)",
 "Never OOM Integrated": "Never OOM (tích hợp)",
 "Enabled for UNet (always maximize offload)": "Bật cho UNet (luôn tối đa offload)",
 "Enabled for VAE (always tiled)": "Bật cho VAE (luôn chia ô)",
 "Method": "Phương pháp",
 "Tile Width": "Chiều rộng ô",
 "Tile Height": "Chiều cao ô",
 "Tile Overlap": "Chồng lấn ô",
 "Tile Batch Size": "Số ô mỗi lượt",
 "Dynamic Prompts": "Dynamic Prompts (prompt động)",
 "Dynamic Prompts enabled": "Bật Dynamic Prompts",
 "Wildcards Manager": "Quản lý Wildcards",
 "Drop Image Here": "Kéo thả ảnh vào đây",
 "Drop File Here": "Kéo thả file vào đây",
 "- or -": "- hoặc -",
 "Click to Upload": "Bấm để tải lên",
 "Drop Image Here - or - Click to Upload": "Kéo thả ảnh vào đây - hoặc - Bấm để tải lên",
 "Clear": "Xoá",
 "Submit": "Gửi",
 "Edit": "Sửa",
 "Undo": "Hoàn tác",
 "Download": "Tải xuống",
 "Remove": "Gỡ",
 "Use brush": "Dùng cọ",
 "Use eraser": "Dùng tẩy",
 "Brush": "Cọ",
 "Erase": "Tẩy",
 "Color": "Màu",
 "Size": "Kích thước",
 "Error": "Lỗi",
 "Read generation parameters from prompt or last generation if prompt is empty into user interface.": "Đọc thông số từ prompt (hoặc lần tạo gần nhất nếu prompt trống) vào giao diện.",
 "Paste available values into the field": "Dán giá trị có sẵn vào ô",
 "Clear prompt": "Xoá prompt",
 "Save style": "Lưu style",
 "Switch width/height": "Đổi chiều rộng/cao",
 "Set seed to -1, which will cause a new random number to be used every time": "Đặt seed = -1 để mỗi lần tạo dùng số ngẫu nhiên mới",
 "Reuse seed from last generation, mostly useful if it was randomized": "Dùng lại seed của lần tạo trước (hữu ích khi seed ngẫu nhiên)",
 "Read generation parameters from prompt": "Đọc thông số từ prompt",
 "Open folder": "Mở thư mục",
 "How many times to improve the generated image iteratively; higher values take longer; very low values can produce bad results": "Số lần tinh chỉnh ảnh; càng cao càng lâu; quá thấp cho kết quả xấu",
 "Which algorithm to use to produce the image": "Thuật toán tạo ảnh",
 "How many batches of images to create (has no impact on generation performance or VRAM usage)": "Số lượt tạo (không ảnh hưởng tốc độ/VRAM)",
 "How many image to create in a single batch (increases generation performance at cost of higher VRAM usage)": "Số ảnh mỗi lượt (nhanh hơn nhưng tốn VRAM hơn)",
 "Classifier Free Guidance Scale - how strongly the image should conform to prompt - lower values produce more creative results": "Độ bám prompt – thấp hơn thì sáng tạo hơn, cao hơn thì bám sát hơn",
 "A value that determines the output of random number generator - if you create an image with same parameters and seed as another image, you'll get the same result": "Giá trị quyết định ngẫu nhiên – cùng thông số + cùng seed sẽ cho cùng ảnh",
 "Determines how little respect the algorithm should have for image's content. At 0, nothing will change, and at 1 you'll get an unrelated image. With values below 1.0, processing will take less steps than the Sampling Steps slider specifies.": "Mức thay đổi so với ảnh gốc. 0 = giữ nguyên, 1 = ảnh hoàn toàn khác. Dưới 1.0 sẽ chạy ít bước hơn thanh Steps.",
 "Produce an image that can be tiled.": "Tạo ảnh có thể lặp liền mạch.",
 "Use a two step process to partially create an image at smaller resolution, upscale, and then improve details in it without changing composition": "Tạo ảnh nhỏ trước, phóng to rồi thêm chi tiết mà không đổi bố cục",
 "Stop processing current image and continue processing.": "Bỏ qua ảnh hiện tại, tiếp tục ảnh sau.",
 "Stop processing images and return any results accumulated so far.": "Dừng toàn bộ và trả về kết quả đã có.",
 "Write image to a directory (default - log/images) and generation parameters into csv file.": "Lưu ảnh vào thư mục (mặc định log/images) và thông số vào file csv.",
 "Seed of a different picture to be mixed into the generation.": "Seed của ảnh khác để trộn vào.",
 "How strong of a variation to produce. At 0, there will be no effect. At 1, you will get the complete picture with variation seed (except for ancestral samplers, where you will just get something).": "Độ mạnh biến thể. 0 = không đổi, 1 = hoàn toàn theo seed biến thể.",
 "Make an attempt to produce a picture similar to what would have been produced with same seed at specified resolution": "Cố tạo ảnh giống như khi dùng cùng seed ở độ phân giải chỉ định",
 "Separate values for X axis using commas.": "Các giá trị trục X cách nhau bằng dấu phẩy.",
 "Separate values for Y axis using commas.": "Các giá trị trục Y cách nhau bằng dấu phẩy.",
 "Do not do anything special": "Không làm gì đặc biệt",
 "Separate prompts into parts using vertical pipe character (|) and the script will create a picture for every combination of them (except for the first part, which will be present in all combinations)": "Tách prompt bằng dấu | – script tạo ảnh cho mọi tổ hợp (phần đầu luôn có mặt)",
 "Create a grid where images will have different parameters. Use inputs below to specify which parameters will be shared by columns and rows": "Tạo lưới ảnh với các thông số khác nhau theo hàng/cột",
 "Resize the image so that entirety of target resolution is filled with the image. Crop parts that stick out.": "Co giãn lấp đầy kích thước đích, cắt phần thừa.",
 "Resize the image so that entirety of image is inside target resolution. Fill empty space with image's colors.": "Co giãn để toàn bộ ảnh nằm trong kích thước đích, lấp chỗ trống bằng màu ảnh.",
 "Resize image to target resolution. Unless height and width match, you will get incorrect aspect ratio.": "Co giãn về kích thước đích. Sai tỉ lệ nếu không khớp.",
 "How much to blur the mask before processing, in pixels.": "Độ mờ mép mask trước khi xử lý (pixel).",
 "What to put inside the masked area before processing it with Stable Diffusion.": "Điền gì vào vùng mask trước khi xử lý.",
 "fill it with colors of the image": "lấp bằng màu của ảnh",
 "keep whatever was there originally": "giữ nguyên nội dung gốc",
 "fill it with latent space noise": "lấp bằng nhiễu latent",
 "fill it with latent space zeroes": "lấp bằng latent trống",
 "Upscale masked region to target resolution, do inpainting, downscale back and paste into original image": "Phóng to vùng mask, vẽ lại, thu nhỏ và dán lại vào ảnh gốc",
 "Enable Lora": "Bật LoRA",
 "Open images output directory.": "Mở thư mục ảnh đầu ra.",
 "Set width and height to the size of the image.": "Đặt chiều rộng/cao bằng kích thước ảnh.",
 "Save all generated images": "Lưu tất cả ảnh đã tạo",
 "Eta": "Eta",
 "Images": "Ảnh",
 "Prompt\n(Press Ctrl+Enter to generate, Alt+Enter to skip, Esc to interrupt)": "Prompt – mô tả ảnh muốn tạo bằng tag Danbooru\n(Ctrl+Enter: tạo ảnh, Alt+Enter: bỏ qua, Esc: dừng)",
 "Negative prompt\n(Press Ctrl+Enter to generate, Alt+Enter to skip, Esc to interrupt)": "Negative prompt – những gì KHÔNG muốn xuất hiện\n(Ctrl+Enter: tạo ảnh, Alt+Enter: bỏ qua, Esc: dừng)",
 "Prompt for hires fix pass.\nLeave empty to use the same prompt as in first pass.": "Prompt cho bước Hires fix.\nĐể trống để dùng prompt chính.",
 "Negative prompt for hires fix pass.\nLeave empty to use the same negative prompt as in first pass.": "Negative prompt cho bước Hires fix.\nĐể trống để dùng negative prompt chính.",
 "ADetailer prompt\nIf blank, the main prompt is used.": "Prompt cho ADetailer\nĐể trống sẽ dùng prompt chính.",
 "ADetailer negative prompt\nIf blank, the main negative prompt is used.": "Negative prompt cho ADetailer\nĐể trống sẽ dùng negative prompt chính.",
 "ADetailer prompt 2nd\nIf blank, the main prompt is used.": "Prompt cho ADetailer (bộ 2)\nĐể trống sẽ dùng prompt chính.",
 "ADetailer negative prompt 2nd\nIf blank, the main negative prompt is used.": "Negative prompt cho ADetailer (bộ 2)\nĐể trống sẽ dùng negative prompt chính.",
 "Input directory path to the control images.": "Đường dẫn thư mục chứa ảnh điều khiển.",
 "Model URL or Model ID": "URL hoặc ID model",
 "Model URL": "URL model",
 "Search in wildcard names...": "Tìm trong tên wildcard...",
 "to img2img": "→ img2img",
 "to sketch": "→ sketch",
 "to inpaint": "→ inpaint",
 "to inpaint sketch": "→ inpaint sketch",
 "Scan": "Quét",
 "Launch": "Chạy",
 "Terminate": "Tắt",
 "Uninstall": "Gỡ cài",
 "Merge": "Trộn",
 "Previous Page": "Trang trước",
 "Next Page": "Trang sau",
 "Apply and quit": "Áp dụng và thoát",
 "View changes": "Xem thay đổi",
 "Save UNet": "Lưu UNet",
 "Save Checkpoint": "Lưu Checkpoint",
 "Refresh wildcards": "Làm mới wildcards",
 "Save wildcards": "Lưu wildcards",
 "Delete all wildcards": "Xoá tất cả wildcards",
 "Get Model Info from Civitai": "Lấy thông tin model từ Civitai",
 "Get Model Info by Civitai Url": "Lấy thông tin theo URL Civitai",
 "Check New Version from Civitai": "Kiểm tra phiên bản mới trên Civitai",
 "Download Models": "Tải model",
 "3. Download Model": "3. Tải model",
 "Detect Model Type": "Nhận diện loại model",
 "Add to Batch": "Thêm vào hàng loạt",
 "1st": "Bộ 1",
 "2nd": "Bộ 2",
 "Enable this tab (1st)": "Bật bộ 1",
 "Enable this tab (2nd)": "Bật bộ 2",
 "Civitai Helper": "Civitai Helper (quản lý model)",
 "Civitai Helper Browser": "Duyệt Civitai",
 "Infinite image browsing": "Duyệt ảnh đã tạo",
 "ControlNet": "ControlNet",
 "Tag Autocomplete": "Gợi ý tag (Autocomplete)",
 "Single": "Một model",
 "Batch Download": "Tải hàng loạt",
 "Callbacks": "Callbacks",
 "Method to filter top k masks by (confidence or area)": "Lọc top k mask theo (độ tin cậy hoặc diện tích)",
 "ad_prompt": "Prompt ADetailer",
 "ad_negative_prompt": "Negative prompt ADetailer",
 "Combinatorial generation": "Sinh tổ hợp",
 "Max generations (0 = all combinations - the batch count value is ignored)": "Số ảnh tối đa (0 = tất cả tổ hợp, bỏ qua Batch count)",
 "Combinatorial batches": "Số lượt tổ hợp",
 "Prompt Magic": "Prompt Magic (prompt tự động)",
 "Magic prompt": "Magic prompt",
 "Max magic prompt length": "Độ dài tối đa magic prompt",
 "Magic prompt creativity": "Độ sáng tạo magic prompt",
 "Magic prompt model": "Model magic prompt",
 "Magic prompt blocklist regex": "Regex chặn từ (magic prompt)",
 "I'm feeling lucky": "Thử vận may",
 "Attention grabber": "Tăng nhấn mạnh ngẫu nhiên",
 "Minimum attention": "Nhấn mạnh tối thiểu",
 "Maximum attention": "Nhấn mạnh tối đa",
 "Don't apply to negative prompts": "Không áp dụng cho negative prompt",
 "Need help?": "Cần trợ giúp?",
 "Jinja2 templates": "Mẫu Jinja2",
 "Enable Jinja2 templates": "Bật mẫu Jinja2",
 "Help for Jinja2 templates": "Trợ giúp mẫu Jinja2",
 "Advanced options": "Tuỳ chọn nâng cao",
 "Unlink seed from prompt": "Tách seed khỏi prompt",
 "Fixed seed": "Seed cố định",
 "Don't generate images": "Không tạo ảnh",
 "Wildcards file": "File wildcards",
 "File editor": "Trình soạn file",
 "Overwrite existing": "Ghi đè file có sẵn",
 "Select a collection": "Chọn bộ sưu tập",
 "Collection actions": "Thao tác bộ sưu tập",
 "Copy collection": "Sao chép bộ sưu tập",
 "Enabled": "Bật",
 "Start step": "Bước bắt đầu",
 "End step": "Bước kết thúc",
 "Strength": "Cường độ",
 "FreeU Integrated (SD 1.x, SD 2.x, SDXL)": "FreeU (tích hợp – SD 1.x, SD 2.x, SDXL)",
 "SelfAttentionGuidance Integrated (SD 1.x, SD 2.x, SDXL)": "Self Attention Guidance (tích hợp – SD 1.x, SD 2.x, SDXL)",
 "Blur Sigma": "Sigma làm mờ",
 "Blur mask threshold": "Ngưỡng mask làm mờ",
 "Block Number": "Số block",
 "Downscale Factor": "Hệ số thu nhỏ",
 "Start Percent": "Phần trăm bắt đầu",
 "End Percent": "Phần trăm kết thúc",
 "Downscale After Skip": "Thu nhỏ sau skip",
 "Downscale Method": "Cách thu nhỏ",
 "Upscale Method": "Cách phóng to",
 "Mimic Scale": "Mimic Scale",
 "Threshold Percentile": "Phân vị ngưỡng",
 "Mimic Mode": "Chế độ Mimic",
 "Mimic Scale Min": "Mimic Scale tối thiểu",
 "Cfg Mode": "Chế độ CFG",
 "Cfg Scale Min": "CFG tối thiểu",
 "Sched Val": "Giá trị lịch",
 "Share attention in batch": "Chia sẻ attention trong batch",
 "Sharpness Multiplier": "Hệ số sắc nét",
 "Contrast Multiplier": "Hệ số tương phản",
 "Extra Noise Multiplier": "Hệ số nhiễu thêm",
 "Vary seeds for X": "Đổi seed theo X",
 "Vary seeds for Y": "Đổi seed theo Y",
 "Vary seeds for Z": "Đổi seed theo Z",
 "Select prompt": "Áp dụng cho prompt",
 "Select joining char": "Ký tự nối",
 "positive": "positive",
 "negative": "negative",
 "comma": "dấu phẩy",
 "space": "dấu cách",
 "Insert prompts at the": "Chèn prompt vào",
 "start": "đầu",
 "end": "cuối",
 "Make a combined image containing all outputs (if more than one)": "Ghép tất cả kết quả thành một ảnh (nếu nhiều hơn 1)",
 "Schedule bias": "Độ lệch lịch",
 "Preservation strength": "Mức bảo toàn",
 "Transition contrast boost": "Tăng tương phản vùng chuyển",
 "Mask influence": "Ảnh hưởng của mask",
 "Difference threshold": "Ngưỡng khác biệt",
 "Difference contrast": "Tương phản khác biệt",
 "Help": "Trợ giúp",
 "Fall-off exponent (lower=higher detail)": "Số mũ suy giảm (thấp = chi tiết hơn)",
 "Color variation": "Biến thiên màu",
 "Append interrogated prompt at each iteration": "Thêm prompt đọc từ ảnh ở mỗi vòng",
 "Original prompt": "Prompt gốc",
 "Original negative prompt": "Negative prompt gốc",
 "Decode steps": "Số bước giải mã",
 "Decode CFG scale": "CFG giải mã",
 "Randomness": "Độ ngẫu nhiên",
 "Controlnet input directory": "Thư mục ảnh ControlNet",
 "PNG info": "Thông tin PNG",
 "Focal point face weight": "Trọng số tiêu điểm: mặt",
 "Focal point entropy weight": "Trọng số tiêu điểm: entropy",
 "Focal point edges weight": "Trọng số tiêu điểm: cạnh",
 "Primary model (A)": "Model chính (A)",
 "Secondary model (B)": "Model phụ (B)",
 "Tertiary model (C)": "Model thứ ba (C)",
 "Custom Name (Optional)": "Tên tuỳ chọn",
 "Multiplier (M) - set to 0 to get model A": "Hệ số (M) – 0 = giữ nguyên model A",
 "Interpolation Method": "Phương pháp nội suy",
 "Checkpoint format": "Định dạng checkpoint",
 "Save as float16": "Lưu dạng float16",
 "Copy config from": "Chép config từ",
 "Bake in VAE": "Nhúng VAE",
 "Discard weights with matching name": "Bỏ trọng số có tên khớp",
 "Metadata": "Metadata",
 "Save metadata": "Lưu metadata",
 "Add merge recipe metadata": "Thêm công thức trộn vào metadata",
 "Copy metadata from merged models": "Chép metadata từ các model trộn",
 "Save Current Checkpoint (including all quantization)": "Lưu checkpoint hiện tại (gồm lượng tử hoá)",
 "Filename (will save in /models/Stable-diffusion)": "Tên file (lưu trong /models/Stable-diffusion)",
 "Read metadata from selected checkpoints": "Đọc metadata của các checkpoint đã chọn",
 "Model Types": "Loại model",
 "Model Type": "Loại model",
 "Civitai URL": "URL Civitai",
 "Model Name": "Tên model",
 "Base Model": "Model gốc",
 "Model Version": "Phiên bản model",
 "Sub-folder": "Thư mục con",
 "Subfolder": "Thư mục con",
 "Duplicate File Behavior": "Khi trùng file",
 "Download All Files": "Tải tất cả file",
 "Preview Image": "Ảnh xem trước",
 "Rename Model": "Đổi tên model",
 "All Files": "Tất cả file",
 "All Versions": "Tất cả phiên bản",
 "Models to download": "Model cần tải",
 "Add to Batch Form": "Thêm vào danh sách tải",
 "Move models into category folders": "Chuyển model vào thư mục theo loại",
 "Only Show Models have no Info": "Chỉ hiện model chưa có thông tin",
 "Query": "Từ khoá",
 "Tag": "Tag",
 "Model Age": "Độ mới của model",
 "Sort By": "Sắp xếp theo",
 "Allow NSFW Models": "Cho phép model NSFW",
 "Search Results": "Kết quả tìm kiếm",
 "Number of parallel calculations": "Số phép tính song song",
 "General Image Generation": "Tạo ảnh tổng quát",
 "Image Processing: Matting, Saliency, and Background Removal": "Xử lý ảnh: tách nền, vùng nổi bật",
 "Computer Vision: Image Caption, Object Detection, and Image Segmentation": "Thị giác máy: chú thích ảnh, nhận diện & phân vùng",
 "Computer Vision: Depth, Normal, and Geometry": "Thị giác máy: chiều sâu, pháp tuyến, hình học",
 "Image Processing: Illumination, Shading, and Relighting": "Xử lý ảnh: ánh sáng, đổ bóng, chiếu sáng lại",
 "General Image Processing and Applications": "Xử lý ảnh tổng quát & ứng dụng",
 "Face Swap, Human Identification, and Style Transfer": "Hoán đổi mặt, nhận diện người, chuyển phong cách",
 "Noise multiplier for img2img 2nd": "Hệ số nhiễu img2img (bộ 2)",
 "ADetailer 2nd": "ADetailer (tự sửa mặt/tay) (bộ 2)",
 "Skip img2img 2nd": "Bỏ qua img2img (bộ 2)",
 "ADetailer detector 2nd": "Model phát hiện (bộ 2)",
 "ADetailer detector classes 2nd": "Lớp đối tượng cần phát hiện (bộ 2)",
 "ADetailer prompt 2nd": "Prompt cho ADetailer (bộ 2)",
 "ADetailer negative prompt 2nd": "Negative prompt cho ADetailer (bộ 2)",
 "Detection model confidence threshold 2nd": "Ngưỡng tin cậy phát hiện (bộ 2)",
 "Mask only the top k (0 to disable) 2nd": "Chỉ giữ k mask tốt nhất (0 = tắt) (bộ 2)",
 "Mask min area ratio 2nd": "Tỉ lệ diện tích mask tối thiểu (bộ 2)",
 "Mask max area ratio 2nd": "Tỉ lệ diện tích mask tối đa (bộ 2)",
 "Mask x(→) offset 2nd": "Dịch mask theo x (→) (bộ 2)",
 "Mask y(↑) offset 2nd": "Dịch mask theo y (↑) (bộ 2)",
 "Mask erosion (-) / dilation (+) 2nd": "Co (-) / nở (+) mask (bộ 2)",
 "Mask merge mode 2nd": "Chế độ gộp mask (bộ 2)",
 "Inpaint mask blur 2nd": "Làm mờ mép mask (bộ 2)",
 "Inpaint denoising strength 2nd": "Mức khử nhiễu khi vẽ lại (bộ 2)",
 "Inpaint only masked 2nd": "Chỉ vẽ lại vùng mask (bộ 2)",
 "Inpaint only masked padding, pixels 2nd": "Đệm quanh vùng mask (pixel) (bộ 2)",
 "Use separate width/height 2nd": "Dùng kích thước riêng (bộ 2)",
 "inpaint width 2nd": "chiều rộng inpaint (bộ 2)",
 "inpaint height 2nd": "chiều cao inpaint (bộ 2)",
 "Use separate steps 2nd": "Dùng số bước riêng (bộ 2)",
 "ADetailer steps 2nd": "Số bước ADetailer (bộ 2)",
 "Use separate CFG scale 2nd": "Dùng CFG riêng (bộ 2)",
 "ADetailer CFG scale 2nd": "CFG ADetailer (bộ 2)",
 "Use separate checkpoint 2nd": "Dùng model riêng (bộ 2)",
 "ADetailer checkpoint 2nd": "Model ADetailer (bộ 2)",
 "Use separate VAE 2nd": "Dùng VAE riêng (bộ 2)",
 "ADetailer VAE 2nd": "VAE ADetailer (bộ 2)",
 "Use separate sampler 2nd": "Dùng sampler riêng (bộ 2)",
 "ADetailer sampler 2nd": "Sampler ADetailer (bộ 2)",
 "ADetailer scheduler 2nd": "Scheduler ADetailer (bộ 2)",
 "Use separate noise multiplier 2nd": "Dùng hệ số nhiễu riêng (bộ 2)",
 "Use separate CLIP skip 2nd": "Dùng CLIP skip riêng (bộ 2)",
 "ADetailer CLIP skip 2nd": "CLIP skip ADetailer (bộ 2)",
 "Restore faces after ADetailer 2nd": "Phục hồi mặt sau ADetailer (bộ 2)",
 "ControlNet model 2nd": "Model ControlNet (bộ 2)",
 "ControlNet module 2nd": "Module ControlNet (bộ 2)",
 "ControlNet weight 2nd": "Trọng số ControlNet (bộ 2)",
 "ControlNet guidance start 2nd": "ControlNet bắt đầu (bộ 2)",
 "ControlNet guidance end 2nd": "ControlNet kết thúc (bộ 2)",
 "Method to filter top k masks by (confidence or area) 2nd": "Lọc top k mask theo (độ tin cậy hoặc diện tích) (bộ 2)"
}
# build_notebook.py sẽ chèn mã nguồn colab/noob_tools.py (script Forge: độ phân giải chuẩn + tự fix) vào đây
NOOB_TOOLS_EMBEDDED: str | None = '# -*- coding: utf-8 -*-\n"""NoobAI Tools – script tích hợp cho Stable Diffusion WebUI Forge (được bộ cài NoobAI-XL chép vào <forge>/scripts/).\n\nTính năng:\n  1. 📐 Dropdown "Độ phân giải chuẩn" ngay dưới Width/Height (txt2img + img2img) – các bucket ~1 MP mà\n     NoobAI-XL / SDXL được huấn luyện; chọn là tự điền Width/Height. Hiển thị luôn kích thước sau Hires fix.\n  2. ✨ "Tự fix": tự mở Hires. fix và ADetailer (sửa mặt/tay) khi mở UI, theo <forge>/noob_tools.json\n     do noobai_lib.write_settings() ghi:  {"auto_hires_fix": true, "auto_adetailer": "face+hand", "param_mode": "recommended"}\n  3. ⚙️ Dropdown "Thông số khuyên dùng / tự chỉnh" phía trên Sampling method: một cú chọn đặt toàn bộ\n     sampler, scheduler, steps, CFG, kích thước, batch, Hires fix (bật/tắt, upscaler, ×, steps, denoise, CFG);\n     sửa tay bất kỳ thông số nào → tự chuyển sang "Tuỳ chỉnh".\n  4. 🩹 Dropdown "Tự sửa mặt / mắt / tay (ADetailer)": mức Tắt / Nhẹ / Chuẩn / Mạnh / Nhiều người – đặt model,\n     prompt, confidence, dilation, blur, denoise, padding cho 3 bộ ADetailer (mặt, mắt, tay) trong một cú chọn.\n     Bảng mức lấy từ noob_tools.json ("ad_levels", do noobai_lib.AD_LEVELS ghi); có bảng dự phòng trong file này.\n"""\nimport json\nimport os\nimport re\n\nimport gradio as gr\n\nfrom modules import scripts\n\ntry:\n    from modules.paths_internal import data_path\nexcept Exception:  # noqa: BLE001\n    data_path = os.getcwd()\n\nCONFIG_FILE = os.path.join(data_path, "noob_tools.json")\nCUSTOM = "Tuỳ chỉnh (tự kéo Width/Height)"\n\n# (nhãn, rộng, cao) – bucket chuẩn SDXL ~1 megapixel + vài cỡ lớn hơn cho GPU khoẻ\nPRESETS = [\n    ("Dọc 832×1216 · 2:3 (chuẩn NoobAI, khuyên dùng)", 832, 1216),\n    ("Ngang 1216×832 · 3:2", 1216, 832),\n    ("Vuông 1024×1024 · 1:1", 1024, 1024),\n    ("Dọc 896×1152 · 7:9", 896, 1152),\n    ("Ngang 1152×896 · 9:7", 1152, 896),\n    ("Dọc 768×1344 · 9:16 (điện thoại)", 768, 1344),\n    ("Ngang 1344×768 · 16:9 (màn hình)", 1344, 768),\n    ("Dọc 640×1536 · 5:12 (rất cao)", 640, 1536),\n    ("Ngang 1536×640 · 12:5 (banner)", 1536, 640),\n    ("Dọc 1024×1536 · 2:3 lớn (chậm hơn, GPU ≥ 12 GB)", 1024, 1536),\n    ("Ngang 1536×1024 · 3:2 lớn (chậm hơn, GPU ≥ 12 GB)", 1536, 1024),\n]\n_BY_LABEL = {label: (w, h) for label, w, h in PRESETS}\n_BY_SIZE = {(w, h): label for label, w, h in PRESETS}\n\n\nCUSTOM_P = "✏️ Tuỳ chỉnh (tự đặt từng thông số)"\n\n# Bộ thông số cho txt2img. None = giữ nguyên giá trị hiện tại.\nTXT2IMG_PRESETS = {\n    "✅ Khuyên dùng – NoobAI chuẩn (cân bằng)": dict(\n        sampler="Euler a", scheduler="Automatic", steps=28, cfg=6.0, width=832, height=1216,\n        batch_count=1, batch_size=1,\n        hr=True, hr_upscaler="4x-AnimeSharp", hr_scale=1.5, hr_steps=14, denoise=0.4, hr_cfg=6.0),\n    "⚡ Nháp nhanh – tìm ý tưởng/seed (không Hires)": dict(\n        sampler="Euler a", scheduler="Automatic", steps=20, cfg=5.5, hr=False),\n    "💎 Chất lượng cao – chậm hơn ~2× (Hires ×2)": dict(\n        sampler="DPM++ 2M SDE", scheduler="Karras", steps=32, cfg=6.5,\n        hr=True, hr_upscaler="4x-AnimeSharp", hr_scale=2.0, hr_steps=16, denoise=0.35, hr_cfg=6.5),\n    "🎨 Sáng tạo – bám prompt lỏng, đa dạng hơn": dict(\n        sampler="Euler a", scheduler="Automatic", steps=30, cfg=4.5,\n        hr=True, hr_upscaler="4x-AnimeSharp", hr_scale=1.5, hr_steps=14, denoise=0.45, hr_cfg=4.5),\n    "🎯 Chính xác – bám prompt chặt, ít ngẫu nhiên": dict(\n        sampler="DPM++ 2M", scheduler="Karras", steps=30, cfg=7.5,\n        hr=True, hr_upscaler="4x-AnimeSharp", hr_scale=1.5, hr_steps=15, denoise=0.3, hr_cfg=7.5),\n    CUSTOM_P: None,\n}\n\n# Bộ thông số cho img2img\nIMG2IMG_PRESETS = {\n    "✅ Khuyên dùng – chỉnh vừa phải (denoise 0.5)": dict(sampler="Euler a", scheduler="Automatic", steps=28, cfg=6.0, denoise=0.5),\n    "🪶 Nhẹ – giữ gần như nguyên ảnh (denoise 0.3)": dict(sampler="Euler a", scheduler="Automatic", steps=28, cfg=6.0, denoise=0.3),\n    "🔥 Mạnh – vẽ lại nhiều theo prompt (denoise 0.7)": dict(sampler="Euler a", scheduler="Automatic", steps=30, cfg=6.5, denoise=0.7),\n    "🔍 Phóng to (Ultimate SD Upscale, denoise 0.3)": dict(sampler="DPM++ 2M", scheduler="Karras", steps=24, cfg=6.0, denoise=0.3),\n    CUSTOM_P: None,\n}\n\n# ---------------------------------------------------------------- 🩹 ADetailer\nAD_CUSTOM = "✏️ Tuỳ chỉnh (tự đặt trong mục ADetailer)"\n_AD_P = "[PROMPT], "\n_AD_FACE = dict(models=["face_yolov8s.pt", "face_yolov8n.pt"],\n                prompt=_AD_P + "detailed face, beautiful detailed eyes, symmetrical eyes, eye contact",\n                negative=_AD_P + "bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, blurry, lowres",\n                confidence=0.3, dilate=4, blur=8, denoise=0.4, padding=32)\n_AD_EYES = dict(models=["Anzhc_Eyes_seg_hd.pt", "mediapipe_face_mesh_eyes_only"],\n                prompt=_AD_P + "detailed eyes, beautiful detailed eyes, eye focus, bright pupils, eye reflection, sparkling eyes",\n                negative=_AD_P + "bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, extra pupils, empty eyes, blurry eyes",\n                confidence=0.3, dilate=8, blur=4, denoise=0.35, padding=32)\n_AD_HAND = dict(models=["hand_yolov8s.pt", "hand_yolov8n.pt"],\n                prompt=_AD_P + "detailed hands, five fingers, natural hand pose, fingernails",\n                negative=_AD_P + "bad hands, extra digits, fewer digits, fused fingers, mutated hands, extra fingers, missing fingers, too many fingers",\n                confidence=0.25, dilate=12, blur=8, denoise=0.5, padding=48)\nAD_LEVELS_FALLBACK = {\n    "off": dict(label="⛔ Tắt – không tự sửa", enable=False, tabs=[]),\n    "light": dict(label="🪶 Nhẹ – chỉ sửa mặt (nhanh nhất)", enable=True, tabs=[{**_AD_FACE, "denoise": 0.35}]),\n    "standard": dict(label="✅ Chuẩn – mặt + mắt + tay (khuyên dùng)", enable=True, tabs=[_AD_FACE, _AD_EYES, _AD_HAND]),\n    "strong": dict(label="🔥 Mạnh – mắt/tay lỗi nặng, vẽ lại nhiều hơn", enable=True,\n                   tabs=[{**_AD_FACE, "denoise": 0.45, "dilate": 8}, {**_AD_EYES, "denoise": 0.45, "dilate": 12},\n                         {**_AD_HAND, "denoise": 0.6, "dilate": 16, "padding": 64, "confidence": 0.2}]),\n    "multi": dict(label="👥 Nhiều người – nhận diện nhạy hơn, sửa mặt + mắt + tay", enable=True,\n                  tabs=[{**_AD_FACE, "confidence": 0.2}, {**_AD_EYES, "confidence": 0.2}, {**_AD_HAND, "confidence": 0.15}]),\n}\n# khoá trong tab preset → hậu tố elem_id của ADetailer (script_<tab>_adetailer_<id>[_2nd|_3rd])\nAD_FIELD_IDS = {\n    "enable_tab": "ad_tab_enable", "model": "ad_model", "prompt": "ad_prompt", "negative": "ad_negative_prompt",\n    "confidence": "ad_confidence", "dilate": "ad_dilate_erode", "blur": "ad_mask_blur",\n    "denoise": "ad_denoising_strength", "padding": "ad_inpaint_only_masked_padding",\n}\n_AD_ID_RE = re.compile(r"^script_(txt2img|img2img)_adetailer_(ad_[a-z_]+?)(?:_(\\d+)(?:st|nd|rd|th))?$")\n_AD_DEFAULT_TAB = dict(models=["None"], prompt="", negative="", confidence=0.3, dilate=4, blur=4, denoise=0.4, padding=32)\n\n\ndef _ad_summary(level: dict | None) -> str:\n    if not level:\n        return "<div style=\'font-size:0.85em;opacity:0.8\'>Tự chỉnh trong mục <b>ADetailer</b> bên dưới. Chọn lại một mức để khôi phục.</div>"\n    if not level.get("enable"):\n        return "<div style=\'font-size:0.85em;opacity:0.8\'>ADetailer tắt – ảnh không được tự vẽ lại mặt/mắt/tay.</div>"\n    names = ["mặt", "mắt", "tay"]\n    parts = []\n    for i, t in enumerate(level.get("tabs", [])[:3]):\n        parts.append(f"bộ {i + 1} {names[i] if i < 3 else \'\'}: {t[\'models\'][0]} · denoise {t[\'denoise\']:g} · conf {t[\'confidence\']:g}")\n    return "<div style=\'font-size:0.85em;opacity:0.8\'>Đã đặt: " + " | ".join(parts) + "</div>"\n\n\n# elem_id của các thông số theo tab → khoá trong preset\nPARAM_IDS = {\n    "sampler": "{tab}_sampling", "scheduler": "{tab}_scheduler", "steps": "{tab}_steps", "cfg": "{tab}_cfg_scale",\n    "width": "{tab}_width", "height": "{tab}_height", "batch_count": "{tab}_batch_count", "batch_size": "{tab}_batch_size",\n    "denoise": "{tab}_denoising_strength",\n    # chỉ txt2img\n    "hr": "txt2img_hr-checkbox", "hr_upscaler": "txt2img_hr_upscaler", "hr_scale": "txt2img_hr_scale",\n    "hr_steps": "txt2img_hires_steps", "hr_cfg": "txt2img_hr_cfg",\n}\nPARAM_ORDER = list(PARAM_IDS)\n\n\ndef _choices(comp) -> list:\n    out = []\n    for c in getattr(comp, "choices", None) or []:\n        out.append(c[0] if isinstance(c, (list, tuple)) else c)\n    return out\n\n\ndef _summary(preset: dict | None, tab: str) -> str:\n    if not preset:\n        return "<div style=\'font-size:0.85em;opacity:0.8\'>Tự đặt từng thông số bên dưới. Chọn lại một bộ khuyên dùng để khôi phục.</div>"\n    parts = []\n    if "sampler" in preset:\n        parts.append(f"{preset[\'sampler\']} · {preset.get(\'scheduler\', \'Automatic\')}")\n    if "steps" in preset:\n        parts.append(f"{preset[\'steps\']} steps")\n    if "cfg" in preset:\n        parts.append(f"CFG {preset[\'cfg\']:g}")\n    if "width" in preset:\n        parts.append(f"{preset[\'width\']}×{preset[\'height\']}")\n    if "denoise" in preset and tab == "img2img":\n        parts.append(f"denoise {preset[\'denoise\']:g}")\n    if tab == "txt2img":\n        if preset.get("hr"):\n            parts.append(f"Hires fix {preset.get(\'hr_upscaler\', \'\')} ×{preset.get(\'hr_scale\', 1.5):g}, "\n                         f"{preset.get(\'hr_steps\', 14)} steps, denoise {preset.get(\'denoise\', 0.4):g}")\n        elif "hr" in preset:\n            parts.append("Hires fix tắt")\n    return "<div style=\'font-size:0.85em;opacity:0.8\'>Đã đặt: " + " · ".join(parts) + "</div>"\n\n\ndef load_tool_config() -> dict:\n    cfg = {"auto_hires_fix": True, "auto_adetailer": "standard", "fix_level": None, "param_mode": "recommended",\n           "ad_levels": None}\n    try:\n        with open(CONFIG_FILE, "r", encoding="utf-8") as f:\n            cfg.update(json.load(f))\n    except Exception:  # noqa: BLE001\n        pass\n    if not isinstance(cfg.get("ad_levels"), dict) or not cfg["ad_levels"]:\n        cfg["ad_levels"] = AD_LEVELS_FALLBACK\n    if not cfg.get("fix_level"):\n        legacy = str(cfg.get("auto_adetailer", "standard")).lower()\n        cfg["fix_level"] = {"face+hand": "standard", "face": "light", "off": "off"}.get(legacy, legacy)\n    if cfg["fix_level"] not in cfg["ad_levels"]:\n        cfg["fix_level"] = "standard" if "standard" in cfg["ad_levels"] else next(iter(cfg["ad_levels"]))\n    cfg["auto_adetailer"] = cfg["fix_level"]\n    return cfg\n\n\ndef _ui_config_defaults(tab: str, w_fallback: int, h_fallback: int) -> tuple[int, int, float]:\n    """Width/Height/Upscale by mà Forge sẽ áp SAU khi component được tạo:\n    ui-config.json, rồi preset Forge (config.json: forge_preset=xl → xl_t2i_width/xl_t2i_height…) áp lúc tải trang."""\n    w, h, scale = w_fallback, h_fallback, 1.5\n    try:\n        with open(os.path.join(data_path, "ui-config.json"), "r", encoding="utf-8") as f:\n            ui = json.load(f)\n        w = int(ui.get(f"{tab}/Width/value", w) or w)\n        h = int(ui.get(f"{tab}/Height/value", h) or h)\n        scale = float(ui.get("txt2img/Upscale by/value", scale) or scale)\n    except Exception:  # noqa: BLE001\n        pass\n    try:\n        with open(os.path.join(data_path, "config.json"), "r", encoding="utf-8") as f:\n            opts = json.load(f)\n        preset = opts.get("forge_preset", "xl")\n        kind = "t2i" if tab == "txt2img" else "i2i"\n        if preset in ("sd", "xl", "flux"):\n            w = int(opts.get(f"{preset}_{kind}_width", w) or w)\n            h = int(opts.get(f"{preset}_{kind}_height", h) or h)\n    except Exception:  # noqa: BLE001\n        pass\n    return w, h, scale\n\n\ndef _hint(w: int, h: int, scale: float) -> str:\n    if not w or not h:\n        return ""\n    hw, hh = int(w * scale) // 8 * 8, int(h * scale) // 8 * 8\n    mp = w * h / 1e6\n    return (f"<div style=\'font-size:0.85em;opacity:0.8\'>Ảnh gốc <b>{w}×{h}</b> ({mp:.2f} MP) → sau Hires fix ×{scale:g}: "\n            f"<b>{hw}×{hh}</b>. Tăng thêm nữa: dùng <i>Extras</i> hoặc <i>Ultimate SD Upscale</i> (xem hướng dẫn trong notebook).</div>")\n\n\nclass NoobTools(scripts.Script):\n    def __init__(self):\n        super().__init__()\n        self.cfg = load_tool_config()\n        self.width = {}\n        self.height = {}\n        self.params = {"txt2img": {}, "img2img": {}}   # tab -> khoá preset -> component\n        self.preset_dd = {}\n        self.preset_hint = {}\n        # 🩹 ADetailer: tab -> {"enable": checkbox, "tabs": {1: {field: component}, 2: {...}, 3: {...}}}\n        self.ad = {"txt2img": {"enable": None, "tabs": {}}, "img2img": {"enable": None, "tabs": {}}}\n        self.ad_dd = {}\n        self.ad_hint = {}\n        self.ad_wired = set()\n\n    def title(self):\n        return "NoobAI Tools (độ phân giải chuẩn + tự fix)"\n\n    def show(self, is_img2img):\n        return scripts.AlwaysVisible\n\n    def ui(self, is_img2img):\n        # Lúc này toàn bộ thông số của tab đã được tạo → nối sự kiện cho dropdown ⚙️\n        tab = "img2img" if is_img2img else "txt2img"\n        try:\n            self._wire_presets(tab)\n        except Exception as e:  # noqa: BLE001\n            print(f"[NoobAI Tools] không nối được preset thông số ({tab}): {e}")\n        return []\n\n    # ------------------------------------------------------------------ hooks\n    def before_component(self, component, **kwargs):\n        elem_id = kwargs.get("elem_id") or ""\n        if elem_id in ("sampler_selection_txt2img", "sampler_selection_img2img"):\n            self._build_preset_dropdown(elem_id.rsplit("_", 1)[1])\n\n    def after_component(self, component, **kwargs):\n        elem_id = kwargs.get("elem_id") or getattr(component, "elem_id", None) or ""\n\n        for tab in ("txt2img", "img2img"):\n            for key, pattern in PARAM_IDS.items():\n                if elem_id == pattern.format(tab=tab):\n                    self.params[tab][key] = component\n\n        if elem_id in ("txt2img_width", "img2img_width"):\n            self.width[elem_id.split("_")[0]] = component\n        elif elem_id in ("txt2img_height", "img2img_height"):\n            tab = elem_id.split("_")[0]\n            self.height[tab] = component\n            self._build_preset_ui(tab)\n\n        # ✨ Tự fix: mở sẵn Hires. fix (txt2img) và ADetailer khi vào UI\n        if self.cfg.get("auto_hires_fix") and elem_id == "txt2img_hr":\n            self._open_accordion(component)\n        if self.cfg.get("auto_hires_fix") and elem_id == "txt2img_hr-checkbox":\n            self._check(component)\n        ad_on = self.cfg.get("fix_level", "off") != "off"\n        if elem_id.endswith("adetailer_ad_main_accordion") and ad_on:\n            self._open_accordion(component)\n        if elem_id.endswith("adetailer_ad_main_accordion-checkbox"):\n            if ad_on:\n                self._check(component)\n            for tab in ("txt2img", "img2img"):\n                if elem_id.startswith(f"script_{tab}_"):\n                    self.ad[tab]["enable"] = component\n\n        # 🩹 gom các thông số ADetailer (3 bộ) để dropdown đặt một lượt\n        m = _AD_ID_RE.match(elem_id)\n        if m:\n            tab, field, idx = m.group(1), m.group(2), int(m.group(3) or 1)\n            self.ad[tab]["tabs"].setdefault(idx, {})[field] = component\n\n        # gallery được tạo SAU toàn bộ script (kể cả ADetailer) → lúc này mới nối dropdown 🩹\n        if elem_id in ("txt2img_gallery", "img2img_gallery"):\n            tab = elem_id.split("_")[0]\n            try:\n                self._wire_adetailer(tab)\n            except Exception as e:  # noqa: BLE001\n                print(f"[NoobAI Tools] không nối được mức sửa lỗi ADetailer ({tab}): {e}")\n\n    @staticmethod\n    def _open_accordion(acc):\n        try:\n            acc.open = True\n        except Exception:  # noqa: BLE001\n            pass\n\n    @staticmethod\n    def _check(chk):\n        try:\n            chk.value = True\n        except Exception:  # noqa: BLE001\n            pass\n\n    # ------------------------------------------------------------------ UI\n    def _build_preset_ui(self, tab: str):\n        w, h = self.width.get(tab), self.height.get(tab)\n        if w is None or h is None:\n            return\n        w0, h0, scale = _ui_config_defaults(tab, int(w.value or 0), int(h.value or 0))\n        cur = _BY_SIZE.get((w0, h0), CUSTOM)\n        choices = [label for label, _, _ in PRESETS] + [CUSTOM]\n\n        dd = gr.Dropdown(label="📐 Độ phân giải chuẩn (NoobAI-XL / SDXL) – chọn là tự điền Width/Height",\n                         choices=choices, value=cur, elem_id=f"{tab}_noob_res_preset", interactive=True)\n        dd.do_not_save_to_config = True\n        hint = gr.HTML(value=_hint(w0, h0, scale) if tab == "txt2img" else "", elem_id=f"{tab}_noob_res_hint")\n        hint.do_not_save_to_config = True\n\n        def apply_preset(label):\n            if label in _BY_LABEL:\n                pw, ph = _BY_LABEL[label]\n                return gr.update(value=pw), gr.update(value=ph)\n            return gr.update(), gr.update()\n\n        def sync_dropdown(cw, ch):\n            label = _BY_SIZE.get((int(cw), int(ch)), CUSTOM)\n            return gr.update(value=label), (_hint(int(cw), int(ch), scale) if tab == "txt2img" else "")\n\n        # .input = chỉ khi người dùng chọn (tránh vòng lặp với cập nhật tự động)\n        dd.input(fn=apply_preset, inputs=[dd], outputs=[w, h], show_progress=False)\n        w.change(fn=sync_dropdown, inputs=[w, h], outputs=[dd, hint], show_progress=False)\n        h.change(fn=sync_dropdown, inputs=[w, h], outputs=[dd, hint], show_progress=False)\n\n    # ------------------------------------------------------------------ ⚙️ bộ thông số\n    def _build_preset_dropdown(self, tab: str):\n        presets = TXT2IMG_PRESETS if tab == "txt2img" else IMG2IMG_PRESETS\n        names = list(presets)\n        initial = names[0] if str(self.cfg.get("param_mode", "recommended")).startswith("rec") else CUSTOM_P\n        with gr.Row(elem_id=f"{tab}_noob_param_row"):\n            dd = gr.Dropdown(label="⚙️ Thông số tạo ảnh: chọn bộ khuyên dùng hoặc tự chỉnh",\n                             choices=names, value=initial, elem_id=f"{tab}_noob_param_preset", interactive=True, scale=3)\n            dd.do_not_save_to_config = True\n            hint = gr.HTML(value=_summary(presets[initial], tab), elem_id=f"{tab}_noob_param_hint")\n            hint.do_not_save_to_config = True\n        self.preset_dd[tab] = dd\n        self.preset_hint[tab] = hint\n\n        # 🩹 mức tự sửa mặt / mắt / tay (ADetailer)\n        levels = self.cfg["ad_levels"]\n        labels = [lv["label"] for lv in levels.values()] + [AD_CUSTOM]\n        cur_key = self.cfg.get("fix_level", "standard")\n        cur = levels[cur_key]["label"] if cur_key in levels else AD_CUSTOM\n        with gr.Row(elem_id=f"{tab}_noob_fix_row"):\n            add = gr.Dropdown(label="🩹 Tự sửa mặt / mắt / tay sau khi tạo ảnh (ADetailer)",\n                              choices=labels, value=cur, elem_id=f"{tab}_noob_fix_level", interactive=True, scale=3)\n            add.do_not_save_to_config = True\n            ahint = gr.HTML(value=_ad_summary(levels.get(cur_key)), elem_id=f"{tab}_noob_fix_hint")\n            ahint.do_not_save_to_config = True\n        self.ad_dd[tab] = add\n        self.ad_hint[tab] = ahint\n\n    def _wire_adetailer(self, tab: str):\n        if tab in self.ad_wired:\n            return\n        dd, hint = self.ad_dd.get(tab), self.ad_hint.get(tab)\n        info = self.ad.get(tab) or {}\n        enable, tabs = info.get("enable"), info.get("tabs") or {}\n        if dd is None or not tabs:\n            if dd is not None:\n                hint.value = "<div style=\'font-size:0.85em;opacity:0.8\'>Không tìm thấy ADetailer – mức sửa lỗi không hoạt động.</div>"\n            return\n        self.ad_wired.add(tab)\n        levels = self.cfg["ad_levels"]\n        by_label = {lv["label"]: lv for lv in levels.values()}\n\n        outputs, slots = [], []       # slots: (n, field) tương ứng từng output; n=0 → checkbox bật ADetailer\n        if enable is not None:\n            outputs.append(enable)\n            slots.append((0, "enable"))\n        for n in sorted(tabs):\n            for key, fid in AD_FIELD_IDS.items():\n                comp = tabs[n].get(fid)\n                if comp is not None:\n                    outputs.append(comp)\n                    slots.append((n, key))\n\n        def apply(label):\n            lv = by_label.get(label)\n            if lv is None:\n                return [gr.update() for _ in outputs] + [_ad_summary(None)]\n            updates = []\n            for n, key in slots:\n                if n == 0:\n                    updates.append(gr.update(value=bool(lv["enable"])))\n                    continue\n                t = lv["tabs"][n - 1] if n - 1 < len(lv["tabs"]) else _AD_DEFAULT_TAB\n                if key == "enable_tab":\n                    updates.append(gr.update(value=True))\n                elif key == "model":\n                    comp = tabs[n][AD_FIELD_IDS["model"]]\n                    choices = _choices(comp)\n                    val = next((mdl for mdl in t["models"] if not choices or mdl in choices), "None")\n                    updates.append(gr.update(value=val))\n                else:\n                    updates.append(gr.update(value=t[key]))\n            return updates + [_ad_summary(lv)]\n\n        dd.input(fn=apply, inputs=[dd], outputs=outputs + [hint], show_progress=False)\n\n        def to_custom():\n            return gr.update(value=AD_CUSTOM), _ad_summary(None)\n\n        for comp in outputs:\n            try:\n                comp.input(fn=to_custom, inputs=[], outputs=[dd, hint], show_progress=False)\n            except Exception:  # noqa: BLE001\n                pass\n\n    def _wire_presets(self, tab: str):\n        dd, hint = self.preset_dd.get(tab), self.preset_hint.get(tab)\n        comps = self.params.get(tab, {})\n        if dd is None or not comps:\n            return\n        presets = TXT2IMG_PRESETS if tab == "txt2img" else IMG2IMG_PRESETS\n        keys = [k for k in PARAM_ORDER if k in comps]\n        outputs = [comps[k] for k in keys]\n\n        def apply(name):\n            preset = presets.get(name)\n            applied = dict(preset) if preset else None\n            updates = []\n            for k in keys:\n                if not preset or k not in preset:\n                    updates.append(gr.update())\n                    continue\n                val = preset[k]\n                comp = comps[k]\n                if k in ("sampler", "scheduler", "hr_upscaler"):\n                    choices = _choices(comp)\n                    if choices and val not in choices:\n                        fallback = {"sampler": ["Euler a", "Euler", "DPM++ 2M"],\n                                    "scheduler": ["Automatic", "Karras"],\n                                    "hr_upscaler": ["4x-AnimeSharp", "4x-UltraSharp", "R-ESRGAN 4x+ Anime6B", "Latent"]}[k]\n                        val = next((f for f in fallback if f in choices), choices[0])\n                        applied[k] = val\n                updates.append(gr.update(value=val))\n            return updates + [_summary(applied, tab)]\n\n        dd.input(fn=apply, inputs=[dd], outputs=outputs + [hint], show_progress=False)\n\n        # người dùng sửa tay bất kỳ thông số nào → chuyển dropdown sang "Tuỳ chỉnh"\n        def to_custom():\n            return gr.update(value=CUSTOM_P), _summary(None, tab)\n\n        for k in keys:\n            try:\n                comps[k].input(fn=to_custom, inputs=[], outputs=[dd, hint], show_progress=False)\n            except Exception:  # noqa: BLE001\n                pass\n'
NOOB_TOOLS_NAME = "noob_tools.py"
NOOB_TOOLS_URLS = [
    f"https://github.com/manhlee1196-boop/anime2/raw/main/colab/{NOOB_TOOLS_NAME}",
    f"https://raw.githubusercontent.com/manhlee1196-boop/anime2/main/colab/{NOOB_TOOLS_NAME}",
    f"https://api.github.com/repos/manhlee1196-boop/anime2/contents/colab/{NOOB_TOOLS_NAME}?ref=main",
]

# Độ phân giải chuẩn (bucket ~1 MP của SDXL / NoobAI-XL) – dùng cho form notebook và script noob_tools
STANDARD_RESOLUTIONS = {
    "832x1216 (dọc 2:3 – chuẩn NoobAI)": (832, 1216),
    "1216x832 (ngang 3:2)": (1216, 832),
    "1024x1024 (vuông)": (1024, 1024),
    "896x1152 (dọc 7:9)": (896, 1152),
    "1152x896 (ngang 9:7)": (1152, 896),
    "768x1344 (dọc 9:16)": (768, 1344),
    "1344x768 (ngang 16:9)": (1344, 768),
    "640x1536 (dọc 5:12)": (640, 1536),
    "1536x640 (ngang 12:5)": (1536, 640),
}

HF = "https://huggingface.co"

# (tên file đích, url, kích thước byte, sha256) – lấy từ HuggingFace API, dùng để kiểm tra sau khi tải.
MODEL_FILES = {
    "noobai_xl_eps_1_1": dict(
        kind="Stable-diffusion",
        name="NoobAI-XL-v1.1.safetensors",
        url=f"{HF}/Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors",
        size=7105349958,
        sha256="6681e8e4b134c81f16533acedb0d406d7e5e366e1624b4105178c64d00b05d51",
    ),
}
VAE_FILES = {
    "sdxl_vae_fp16_fix": dict(
        kind="VAE",
        name="sdxl_vae_fp16_fix.safetensors",
        url=f"{HF}/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors",
        size=334641162,
        sha256="235745af8d86bf4a4c1b5b4f529868b37019a10f7c0b2e79ad0abca3a22bc6e1",
    ),
}
UPSCALER_FILES = {
    "4x_ultrasharp": dict(
        kind="ESRGAN", name="4x-UltraSharp.pth",
        url=f"{HF}/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth",
        size=66961958, sha256="a5812231fc936b42af08a5edba784195495d303d5b3248c24489ef0c4021fe01",
    ),
    "4x_animesharp": dict(
        kind="ESRGAN", name="4x-AnimeSharp.pth",
        url=f"{HF}/Kim2091/AnimeSharp/resolve/main/4x-AnimeSharp.pth",
        size=67010245, sha256="e7a7de2dafd7331c1992862bbbcd9e9712a9f9f8e6303f0aaa59b4341d359bab",
    ),
}
# ControlNet: bản fp16 (2.5 GB/model). Forge đọc trực tiếp định dạng diffusers.
_EU = f"{HF}/Eugeoter"
CONTROLNET_FILES = {
    "union_promax": dict(
        kind="ControlNet", name="xinsir_controlnet_union_sdxl_promax.safetensors",
        url=f"{HF}/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model_promax.safetensors",
        size=2513342408, sha256="9fae2e50cb431bfcbe05822b59ec2228df545ef27f711dea8949e9f4ed9f7cdc",
    ),
    "noob_canny": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_canny_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-canny/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="68d4115d0f97ff59f3444aa06bd81a6c1e96c8a3750d8f6b442aa0495cc89508",
    ),
    "noob_openpose": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_openpose.safetensors",
        url=f"{HF}/Laxhar/noob_openpose/resolve/main/openpose_pre.safetensors",
        size=2502140008, sha256="918cfc4d7def165ea7a06bee10841fe525332515f8a0486c5cb2a171ec40b873",
    ),
    "noob_depth_midas": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_depth_midas_v1-1_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-depth_midas-v1-1/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="678b94ef3260744a1e9e50932ef8895e7f7fabf25eec7fd5cc96298f7220fe50",
    ),
    "noob_lineart_anime": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_lineart_anime_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-lineart_anime/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="44eae6a514a60ae426ff966ecdbb9e170ad63d8889456982bb29037df42b86a8",
    ),
    "noob_tile": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_tile_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-tile/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="d6d676f29153357ac8d57727e35b8edc090aa4de57e861b878872ff45c432615",
    ),
    "noob_scribble_pidinet": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_scribble_pidinet_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-scribble_pidinet/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="6ba55d75e34f63d0538c2dc1006415c495f254624c1c826df7dfaf4bc172ee47",
    ),
    "noob_softedge_hed": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_softedge_hed_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-softedge_hed/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="c540e9bb474d7b090cee5f45017ceba2d58775c71bc0f6eafa410b435b1dfd92",
    ),
    "noob_manga_line": dict(
        kind="ControlNet", name="noob_sdxl_controlnet_manga_line_fp16.safetensors",
        url=f"{_EU}/noob-sdxl-controlnet-manga_line/resolve/main/diffusion_pytorch_model.fp16.safetensors",
        size=2502139136, sha256="6f2283231fc00f389b20fd46287185b8e9d8c3e647e056ba48268c45b931d213",
    ),
}
# ADetailer (tải sẵn để dùng được ngay, không chờ HF lúc gen)
ADETAILER_FILES = {
    "face_yolov8n": dict(kind="adetailer", name="face_yolov8n.pt",
                         url=f"{HF}/Bingsu/adetailer/resolve/main/face_yolov8n.pt", size=6230011,
                         sha256="70b640f8f60b1cf0dcc72f30caf3da9495eb2fb6509da48c53374ad6806e6a9c"),
    "face_yolov8s": dict(kind="adetailer", name="face_yolov8s.pt",
                         url=f"{HF}/Bingsu/adetailer/resolve/main/face_yolov8s.pt", size=22507707,
                         sha256="c7237eff25787377de196961140ceaed324d859ee8de5a775d93d33a0e3fab78"),
    "hand_yolov8n": dict(kind="adetailer", name="hand_yolov8n.pt",
                         url=f"{HF}/Bingsu/adetailer/resolve/main/hand_yolov8n.pt", size=6237883,
                         sha256="3991202eb69e9ddcb3b9ba80cdeb41e734ffaf844403d6c9f47d515cd88c6f29"),
    "person_yolov8n_seg": dict(kind="adetailer", name="person_yolov8n-seg.pt",
                               url=f"{HF}/Bingsu/adetailer/resolve/main/person_yolov8n-seg.pt", size=6777003,
                               sha256="38fc8aaae97cb6e70be4ec44770005b26ed473471362afcda62a0037d7ccf432"),
    # tay – bản s nhận diện tay anime tốt hơn hẳn bản n
    "hand_yolov8s": dict(kind="adetailer", name="hand_yolov8s.pt",
                         url=f"{HF}/Bingsu/adetailer/resolve/main/hand_yolov8s.pt", size=22507643,
                         sha256="70b540063fbc385736d8258970744a4afbc4cbf7932134bae3b24cdadeadec06"),
    # mắt anime (segmentation, Anzhc) – dùng cho bộ ADetailer "mắt"
    "anzhc_eyes_seg": dict(kind="adetailer", name="Anzhc_Eyes_seg_hd.pt",
                           url=f"{HF}/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Eyes%20-seg-hd.pt", size=6906493,
                           sha256="6be1c13ca7a51c2425e278e07e7ae3d4c94ee125b874a0104a142f4f5a35a308"),
    # mặt anime (segmentation theo đường viền mặt, không lẹm vào tóc) – lựa chọn thay thế
    "anzhc_face_seg": dict(kind="adetailer", name="Anzhc_Face_seg_1024_v2_y8n.pt",
                           url=f"{HF}/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Face%20seg%201024%20v2%20y8n.pt", size=6948693,
                           sha256="1bbcfd7a9f407c6f6e4389a371dbcc392f9444421cf7f824152e92bf563dc6a3"),
}
CONTROLNET_SETS = {
    "none": [],
    "union_promax": ["union_promax"],
    "noob_core": ["noob_canny", "noob_openpose", "noob_depth_midas", "noob_lineart_anime", "noob_tile"],
    "noob_all": list(CONTROLNET_FILES.keys()),
}

EXTENSIONS = {
    "tagcomplete": ("a1111-sd-webui-tagcomplete", "https://github.com/DominikDoom/a1111-sd-webui-tagcomplete"),
    "adetailer": ("adetailer", "https://github.com/Bing-su/adetailer"),
    "image_browser": ("sd-webui-infinite-image-browsing", "https://github.com/zanllp/sd-webui-infinite-image-browsing"),
    "ultimate_upscale": ("ultimate-upscale-for-automatic1111", "https://github.com/Coyote-A/ultimate-upscale-for-automatic1111"),
    "dynamic_prompts": ("sd-dynamic-prompts", "https://github.com/adieyal/sd-dynamic-prompts"),
    "civitai_helper": ("Stable-Diffusion-Webui-Civitai-Helper", "https://github.com/zixaphir/Stable-Diffusion-Webui-Civitai-Helper"),
    "wd14_tagger": ("stable-diffusion-webui-wd14-tagger", "https://github.com/picobyte/stable-diffusion-webui-wd14-tagger"),
    "regional_prompter": ("sd-webui-regional-prompter", "https://github.com/hako-mikan/sd-webui-regional-prompter"),
}

# Prompt khuyến nghị chính thức của NoobAI-XL (EPS)
NOOB_POSITIVE_PREFIX = "masterpiece, best quality, very awa, newest, absurdres, highres, "
NOOB_NEGATIVE = (
    "worst quality, old, early, low quality, lowres, signature, username, logo, bad hands, "
    "mutated hands, mammal, anthro, furry, ambiguous form, feral, semi-anthro"
)
# Negative mặc định trong ô Negative prompt = NoobAI chuẩn + khối anatomy gọn (mắt / tay / chân)
NOOB_NEGATIVE_FULL = NOOB_NEGATIVE + (
    ", worst aesthetic, bad anatomy, bad proportions, extra digits, fewer digits, extra fingers, missing fingers, "
    "fused fingers, bad feet, extra limbs, cross-eyed, uneven eyes"
)

# Bộ thẻ sửa MẮT / TAY / CHÂN (xem docs/prompt_fix_mat_tay_chan.md)
FIX_EYES_POS = "detailed eyes, beautiful detailed eyes, looking at viewer, eye contact, eye focus"
FIX_EYES_NEG = (
    "bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, extra eyes, missing eye, lazy eye, "
    "empty eyes, deformed eyes, misaligned eyes, blurry eyes, extra pupils"
)
FIX_HANDS_POS = "detailed hands, five fingers, fingernails"
FIX_HANDS_NEG = (
    "bad hands, mutated hands, malformed hands, deformed hands, poorly drawn hands, extra digits, fewer digits, "
    "extra fingers, missing fingers, fused fingers, too many fingers, long fingers, extra hands, extra arms, "
    "missing arms, disconnected limbs, floating limbs"
)
FIX_FEET_POS = "detailed legs, detailed feet, toes"
FIX_FEET_NEG = (
    "bad feet, bad legs, extra legs, missing legs, deformed feet, malformed feet, poorly drawn feet, fused toes, "
    "extra toes, missing toes, twisted legs, bad knees, disconnected legs, extra limbs, bad proportions"
)
FIX_ALL_NEG = (
    "bad anatomy, bad proportions, bad perspective, bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, extra eyes, "
    "extra digits, fewer digits, extra fingers, missing fingers, fused fingers, malformed hands, extra arms, missing arms, "
    "bad feet, bad legs, extra legs, missing legs, fused toes, extra toes, disconnected limbs, extra limbs"
)
# Prompt ADetailer mặc định. "[PROMPT]" = ADetailer tự chèn prompt/negative chính của ảnh vào vị trí đó
# → giữ nguyên màu mắt, kiểu tóc, phong cách… rồi mới thêm thẻ của bộ phận.
AD_FACE_PROMPT = "[PROMPT], detailed face, beautiful detailed eyes, symmetrical eyes, eye contact"
AD_FACE_NEG = "[PROMPT], bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, blurry, lowres"
AD_EYES_PROMPT = "[PROMPT], detailed eyes, beautiful detailed eyes, eye focus, bright pupils, eye reflection, sparkling eyes"
AD_EYES_NEG = "[PROMPT], bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, extra pupils, empty eyes, blurry eyes"
AD_HAND_PROMPT = "[PROMPT], detailed hands, five fingers, natural hand pose, fingernails"
AD_HAND_NEG = "[PROMPT], bad hands, extra digits, fewer digits, fused fingers, mutated hands, extra fingers, missing fingers, too many fingers"

# Model ADetailer theo thứ tự ưu tiên (file nào có trong models/adetailer thì dùng)
AD_FACE_MODELS = ["face_yolov8s.pt", "face_yolov8n.pt", "Anzhc_Face_seg_1024_v2_y8n.pt"]
AD_EYES_MODELS = ["Anzhc_Eyes_seg_hd.pt", "mediapipe_face_mesh_eyes_only"]
AD_HAND_MODELS = ["hand_yolov8s.pt", "hand_yolov8n.pt"]


def _ad_tab(models, prompt, neg, *, conf=0.3, dilate=4, blur=4, denoise=0.4, pad=32) -> dict:
    return dict(models=list(models), prompt=prompt, negative=neg, confidence=conf, dilate=dilate,
                blur=blur, denoise=denoise, padding=pad)


_AD_FACE = _ad_tab(AD_FACE_MODELS, AD_FACE_PROMPT, AD_FACE_NEG, conf=0.3, dilate=4, blur=8, denoise=0.4, pad=32)
_AD_EYES = _ad_tab(AD_EYES_MODELS, AD_EYES_PROMPT, AD_EYES_NEG, conf=0.3, dilate=8, blur=4, denoise=0.35, pad=32)
_AD_HAND = _ad_tab(AD_HAND_MODELS, AD_HAND_PROMPT, AD_HAND_NEG, conf=0.25, dilate=12, blur=8, denoise=0.5, pad=48)

# Mức tự sửa (ADetailer) – dùng chung cho ui-config.json (mặc định khi mở UI) và dropdown 🩹 trong noob_tools.py.
# Bộ 1 = mặt, bộ 2 = mắt, bộ 3 = tay. Bộ không có trong danh sách → model "None".
AD_LEVELS = {
    "off": dict(label="⛔ Tắt – không tự sửa", enable=False, tabs=[]),
    "light": dict(label="🪶 Nhẹ – chỉ sửa mặt (nhanh nhất)", enable=True,
                  tabs=[{**_AD_FACE, "denoise": 0.35}]),
    "standard": dict(label="✅ Chuẩn – mặt + mắt + tay (khuyên dùng)", enable=True,
                     tabs=[_AD_FACE, _AD_EYES, _AD_HAND]),
    "strong": dict(label="🔥 Mạnh – mắt/tay lỗi nặng, vẽ lại nhiều hơn", enable=True,
                   tabs=[{**_AD_FACE, "denoise": 0.45, "dilate": 8},
                         {**_AD_EYES, "denoise": 0.45, "dilate": 12},
                         {**_AD_HAND, "denoise": 0.6, "dilate": 16, "padding": 64, "confidence": 0.2}]),
    "multi": dict(label="👥 Nhiều người – nhận diện nhạy hơn, sửa mặt + mắt + tay", enable=True,
                  tabs=[{**_AD_FACE, "confidence": 0.2}, {**_AD_EYES, "confidence": 0.2},
                        {**_AD_HAND, "confidence": 0.15}]),
}
AD_LEVEL_ALIASES = {"face+hand": "standard", "face": "light", "on": "standard", "true": "standard",
                    "false": "off", "none": "off", "full": "standard"}


def ad_level_key(value) -> str:
    v = str(value or "standard").strip().lower()
    v = AD_LEVEL_ALIASES.get(v, v)
    return v if v in AD_LEVELS else "standard"


# --------------------------------------------------------------------------------------
# Cấu hình
# --------------------------------------------------------------------------------------
@dataclass
class Config:
    # Tài nguyên
    download_model: bool = True
    download_vae: bool = True
    download_upscalers: bool = True
    controlnet: str = "union_promax"          # none | union_promax | noob_core | noob_all
    extra_model_urls: str = ""                # cách nhau bằng dấu phẩy / xuống dòng
    extra_lora_urls: str = ""
    extra_embedding_urls: str = ""
    civitai_token: str = ""
    hf_token: str = ""
    verify_sha256: bool = False
    # Extensions
    extensions: list = field(default_factory=lambda: [
        "tagcomplete", "adetailer", "image_browser", "ultimate_upscale", "dynamic_prompts", "civitai_helper",
    ])
    use_repo_tag_csv: bool = True
    # Google Drive
    mount_drive: bool = True
    save_outputs_to_drive: bool = True
    link_drive_models: bool = True
    drive_folder: str = "NoobAI"
    # Giao diện / tunnel
    tunnel: str = "gradio.live"               # gradio.live | ngrok | both | none
    ngrok_token: str = ""
    gradio_auth: str = ""                     # "user:pass"
    theme: str = "dark"
    ui_language: str = "vi"                   # vi (tiếng Việt) | en (English)
    port: int = 7860
    use_xformers: bool = True
    gpu_mode: str = "max"                     # max (dùng tối đa GPU) | balanced (mặc định Forge) | lowvram (ảnh/batch lớn)
    extra_args: str = ""
    forge_commit: str = FORGE_COMMIT_TESTED
    # Mặc định UI – param_mode: recommended (dùng bộ NoobAI chuẩn, bỏ qua các default_* bên dưới) | custom
    param_mode: str = "recommended"
    default_width: int = 832
    default_height: int = 1216
    default_steps: int = 28
    default_cfg: float = 6.0
    default_sampler: str = "Euler a"
    default_scheduler: str = "Automatic"
    clip_skip: int = 2
    # ✨ Tự fix: mở sẵn Hires. fix + ADetailer khi vào UI
    auto_hires_fix: bool = True
    hires_upscale_by: float = 1.5
    hires_denoise: float = 0.4
    auto_adetailer: str = "standard"           # standard | strong | light | multi | off (cũ: face+hand, face)
    # Chế độ test (CPU, không GPU) – chỉ dùng khi kiểm thử bộ cài
    test_mode: bool = False

    @staticmethod
    def load(path: Path = CONFIG_PATH) -> "Config":
        cfg = Config()
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            for k, v in data.items():
                if hasattr(cfg, k):
                    setattr(cfg, k, v)
        return cfg

    def save(self, path: Path = CONFIG_PATH) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(asdict(self), indent=2, ensure_ascii=False), encoding="utf-8")


# --------------------------------------------------------------------------------------
# Đường dẫn
# --------------------------------------------------------------------------------------
class Paths:
    root = ROOT
    forge = ROOT / "noobai-forge"
    venv = ROOT / "venv-noobai"
    uv_bin = ROOT / "bin" / "uv"
    drive = Path("/content/drive/MyDrive")

    @classmethod
    def python(cls) -> Path:
        return cls.venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

    @classmethod
    def models(cls, kind: str) -> Path:
        return cls.forge / "models" / kind

    @classmethod
    def extensions(cls) -> Path:
        return cls.forge / "extensions"


# --------------------------------------------------------------------------------------
# Tiện ích
# --------------------------------------------------------------------------------------
def banner(msg: str) -> None:
    line = "─" * max(8, min(78, len(msg) + 6))
    print(f"\n┌{line}┐\n│   {msg}\n└{line}┘", flush=True)


def log(msg: str) -> None:
    print(f"  • {msg}", flush=True)


def human(n: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"


def run(cmd: list[str] | str, *, cwd: Path | None = None, env: dict | None = None,
        check: bool = True, quiet: bool = False) -> int:
    """Chạy lệnh, stream output trực tiếp ra cell."""
    shell = isinstance(cmd, str)
    if not quiet:
        log("$ " + (cmd if shell else " ".join(str(c) for c in cmd)))
    full_env = {**os.environ, **(env or {})}
    full_env.setdefault("PYTHONUNBUFFERED", "1")
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=full_env, shell=shell,
                          stdout=subprocess.DEVNULL if quiet else None,
                          stderr=subprocess.STDOUT if quiet else None)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {proc.returncode}): {cmd}")
    return proc.returncode


def capture(cmd: list[str], **kw) -> str:
    return subprocess.run(cmd, capture_output=True, text=True, **kw).stdout.strip()


def is_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "google.colab" in sys.modules


def has_nvidia_gpu() -> bool:
    return shutil.which("nvidia-smi") is not None and subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True).returncode == 0


def gpu_name() -> str:
    if not has_nvidia_gpu():
        return "không có GPU"
    return capture(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]) or "GPU không rõ"


def gpu_vram_mb() -> int:
    """Tổng VRAM (MB) của GPU đầu tiên; 0 nếu không có GPU."""
    if not has_nvidia_gpu():
        return 0
    out = capture(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]) or "0"
    try:
        return int(float(out.splitlines()[0].strip()))
    except Exception:  # noqa: BLE001
        return 0


def gpu_compute_capability() -> float:
    """Compute capability (7.5 = T4/Turing, 8.x = A100/L4/Ampere…); 0 nếu không rõ."""
    if not has_nvidia_gpu():
        return 0.0
    out = capture(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]) or ""
    try:
        return float(out.splitlines()[0].strip())
    except Exception:  # noqa: BLE001
        return 0.0


def gpu_plan(cfg: "Config") -> dict:
    """Quyết định cờ Forge + cài đặt bộ nhớ theo GPU_MODE và GPU thực tế.

    max      : giữ toàn bộ model trên GPU, không hoán đổi ra RAM (nhanh nhất).
               VRAM ≥ 20 GB (L4/A100): --always-gpu ; 12–20 GB (T4 15 GB): --always-high-vram ; < 12 GB: như balanced.
    balanced : mặc định Forge (tự hoán đổi theo VRAM trống).
    lowvram  : ưu tiên ảnh lớn / batch lớn: --always-normal-vram --always-offload-from-vram, chừa 4 GB cho inference.
    """
    mode = str(cfg.gpu_mode or "max").lower()
    vram = gpu_vram_mb()
    cc = gpu_compute_capability()
    flags: list[str] = []
    settings: dict = {"forge_inference_memory": 1024, "forge_async_loading": "Queue", "forge_pin_shared_memory": "CPU"}
    why = []
    if vram <= 0 or cfg.test_mode:
        return {"mode": mode, "flags": flags, "settings": settings, "vram": vram, "cc": cc, "why": ["không có GPU"]}
    if mode.startswith("max"):
        if vram >= 20000:
            flags.append("--always-gpu")
            why.append(f"VRAM {vram/1024:.0f} GB ≥ 20 GB → toàn bộ model + VAE + text encoder nằm hẳn trên GPU")
        elif vram >= 12000:
            flags.append("--always-high-vram")
            why.append(f"VRAM {vram/1024:.0f} GB → UNet nằm hẳn trên GPU (high-vram), không hoán đổi ra RAM")
        else:
            why.append(f"VRAM {vram/1024:.0f} GB < 12 GB → giữ chế độ tự hoán đổi của Forge")
        flags.append("--cuda-malloc")
        why.append("--cuda-malloc: cấp phát bộ nhớ CUDA nhanh hơn")
        if cfg.download_vae:
            flags.append("--vae-in-fp16")
            why.append("--vae-in-fp16: giải mã ảnh bằng fp16 (an toàn vì dùng sdxl-vae-fp16-fix) → nhanh gấp ~2 so với fp32"
                       + (" trên T4/Turing" if cc and cc < 8 else ""))
        settings["forge_inference_memory"] = 1024
        settings["show_progress_every_n_steps"] = 8
    elif mode.startswith("low"):
        flags += ["--always-normal-vram", "--always-offload-from-vram"]
        settings["forge_inference_memory"] = 4096
        why.append("chừa 4 GB VRAM cho inference, giải phóng model sau mỗi bước → ảnh/batch lớn hơn, chậm hơn")
    else:
        why.append("mặc định Forge")
    return {"mode": mode, "flags": flags, "settings": settings, "vram": vram, "cc": cc, "why": why}


def sha256sum(path: Path, bufsize: int = 1 << 24) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(bufsize):
            h.update(chunk)
    return h.hexdigest()


def split_urls(text: str) -> list[str]:
    return [u.strip() for u in re.split(r"[,\n\s]+", text or "") if u.strip()]


# --------------------------------------------------------------------------------------
# Tải file (aria2c nếu có, fallback Python thuần có resume + progress)
# --------------------------------------------------------------------------------------
def _auth_headers(url: str, cfg: Config) -> list[str]:
    headers: list[str] = []
    if "api.github.com" in url:
        headers.append("Accept: application/vnd.github.raw")
    if "civitai.com" in url and cfg.civitai_token:
        headers.append(f"Authorization: Bearer {cfg.civitai_token}")
    if "huggingface.co" in url and cfg.hf_token:
        headers.append(f"Authorization: Bearer {cfg.hf_token}")
    return headers


def _py_download(url: str, dest: Path, headers: list[str]) -> None:
    tmp = dest.with_suffix(dest.suffix + ".part")
    existing = tmp.stat().st_size if tmp.exists() else 0
    req = urllib.request.Request(url)
    for h in headers:
        k, v = h.split(":", 1)
        req.add_header(k.strip(), v.strip())
    req.add_header("User-Agent", "noobai-colab-installer/" + LIB_VERSION)
    if existing:
        req.add_header("Range", f"bytes={existing}-")
    with urllib.request.urlopen(req, timeout=60) as resp:
        if existing and resp.status != 206:
            existing = 0  # server không hỗ trợ resume → tải lại từ đầu
        total = resp.headers.get("Content-Length")
        total = int(total) + existing if total else None
        mode = "ab" if existing else "wb"
        done, last, t0, shown = existing, time.time(), time.time(), False
        with tmp.open(mode) as f:
            while chunk := resp.read(1 << 20):
                f.write(chunk)
                done += len(chunk)
                if time.time() - last > 2:
                    last, shown = time.time(), True
                    speed = (done - existing) / max(1e-6, last - t0)
                    pct = f"{done / total * 100:5.1f}%" if total else ""
                    print(f"\r    {dest.name}: {human(done)} {pct}  {human(speed)}/s   ", end="", flush=True)
        if shown:
            print()
    tmp.rename(dest)


def download_file(url: str, dest: Path, cfg: Config, *, size: int | None = None,
                  sha256: str | None = None) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        if size is None or dest.stat().st_size == size:
            log(f"✔ đã có: {dest.name} ({human(dest.stat().st_size)})")
            return dest
        log(f"⚠ {dest.name} sai kích thước ({dest.stat().st_size} ≠ {size}) → tải lại")
        dest.unlink()

    headers = _auth_headers(url, cfg)
    log(f"⬇ tải {dest.name}  ({human(size) if size else '?'})")
    aria = shutil.which("aria2c")
    ok = False
    if aria:
        cmd = [aria, "--console-log-level=error", "--summary-interval=10", "-c", "-x", "16", "-s", "16",
               "-k", "1M", "--file-allocation=none", "--auto-file-renaming=false", "--allow-overwrite=true",
               "-d", str(dest.parent), "-o", dest.name, url]
        for h in headers:
            cmd.insert(1, f"--header={h}")
        ok = run(cmd, check=False, quiet=False) == 0 and dest.exists()
        if not ok:
            log("aria2c thất bại → dùng trình tải Python")
    if not ok:
        for attempt in range(1, 4):
            try:
                _py_download(url, dest, headers)
                ok = True
                break
            except Exception as e:  # noqa: BLE001
                log(f"lỗi tải (lần {attempt}/3): {e}")
                time.sleep(3)
    if not ok or not dest.exists():
        raise RuntimeError(f"Không tải được {url}")

    actual = dest.stat().st_size
    if size is not None and actual != size:
        raise RuntimeError(f"{dest.name}: kích thước {actual} ≠ {size} (file hỏng / link đổi / cần token)")
    if sha256 and cfg.verify_sha256:
        log(f"kiểm tra SHA256 {dest.name} …")
        got = sha256sum(dest)
        if got != sha256:
            raise RuntimeError(f"{dest.name}: SHA256 không khớp ({got})")
        log("✔ SHA256 khớp")
    log(f"✔ xong: {dest.name} ({human(actual)})")
    return dest


def normalize_url(url: str) -> str:
    """Chuyển link trang Civitai (…/models/123?modelVersionId=456) thành link tải API."""
    m = re.search(r"civitai\.com/models/\d+.*?modelVersionId=(\d+)", url)
    if m:
        return f"https://civitai.com/api/download/models/{m.group(1)}"
    return url


def _guess_filename(url: str) -> str:
    """Đoán tên file từ URL; với link Civitai thì hỏi server (Content-Disposition)."""
    name = url.split("?")[0].rstrip("/").split("/")[-1]
    if "civitai.com" in url or "." not in name:
        try:
            req = urllib.request.Request(url, method="HEAD")
            with urllib.request.urlopen(req, timeout=30) as r:
                cd = r.headers.get("Content-Disposition", "")
                m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)', cd)
                if m:
                    name = urllib.parse.unquote(m.group(1))
        except Exception:  # noqa: BLE001
            pass
    if not name:
        name = f"download_{int(time.time())}"
    if "." not in name:
        name += ".safetensors"
    return name


# --------------------------------------------------------------------------------------
# Bước 1: môi trường Python 3.10 + torch + Forge
# --------------------------------------------------------------------------------------
def ensure_system_tools() -> None:
    if shutil.which("aria2c") is None and shutil.which("apt-get"):
        log("cài aria2 (tải song song nhanh hơn)")
        sudo = [] if os.geteuid() == 0 else (["sudo", "-n"] if shutil.which("sudo") else [])
        run(sudo + ["apt-get", "install", "-y", "-qq", "aria2"], check=False, quiet=True)
    if shutil.which("aria2c") is None:
        log("⚠ không có aria2c → sẽ dùng trình tải Python (chậm hơn một chút)")
    if shutil.which("git") is None:
        raise RuntimeError("Thiếu git")


def ensure_uv() -> Path:
    if Paths.uv_bin.exists():
        return Paths.uv_bin
    found = shutil.which("uv")
    if found:
        return Path(found)
    log("cài uv (trình quản lý Python/pip siêu nhanh)")
    Paths.uv_bin.parent.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "-m", "pip", "install", "-q", "--target", str(Paths.uv_bin.parent / "_uvpkg"), "uv"])
    cand = list((Paths.uv_bin.parent / "_uvpkg").rglob("uv"))
    cand = [c for c in cand if c.is_file() and os.access(c, os.X_OK)]
    if not cand:
        raise RuntimeError("Không tìm thấy binary uv sau khi cài")
    shutil.copy2(cand[0], Paths.uv_bin)
    Paths.uv_bin.chmod(0o755)
    return Paths.uv_bin


def uv_env() -> dict:
    return {
        "UV_PYTHON_INSTALL_DIR": str(ROOT / "uv-python"),
        "UV_CACHE_DIR": str(ROOT / "uv-cache"),
        "UV_SYSTEM_CERTS": "1",  # dùng CA của hệ thống (cần khi mạng đi qua proxy TLS)
        "UV_HTTP_TIMEOUT": "600",
    }


def find_system_python() -> Path | None:
    """Tìm Python 3.10 (ưu tiên) hoặc 3.11 có sẵn trên máy – Forge hỗ trợ 3.7–3.11."""
    candidates = ["python3.10", "python3.11"]
    for name in candidates:
        exe = shutil.which(name)
        if exe:
            return Path(exe)
    if sys.version_info[:2] in ((3, 10), (3, 11)):
        return Path(sys.executable)
    return None


def ensure_python_env(cfg: Config) -> Path:
    py = Paths.python()
    uv = str(ensure_uv())
    env = uv_env()
    if not py.exists():
        log(f"tạo môi trường Python {PYTHON_VERSION} tại {Paths.venv}")
        base = find_system_python()
        if base is None:
            # Colab hiện là Python 3.12/3.13 → tải Python 3.10 độc lập bằng uv
            if run([uv, "python", "install", PYTHON_VERSION], env=env, check=False) == 0:
                base = PYTHON_VERSION
            elif shutil.which("apt-get"):
                log("uv không tải được Python → thử cài python3.10 bằng apt")
                sudo = [] if os.geteuid() == 0 else (["sudo", "-n"] if shutil.which("sudo") else [])
                run(sudo + ["apt-get", "update", "-qq"], check=False, quiet=True)
                run(sudo + ["apt-get", "install", "-y", "-qq", "python3.10", "python3.10-venv", "python3.10-distutils"],
                    check=False, quiet=True)
                base = find_system_python()
        if base is None:
            raise RuntimeError("Không chuẩn bị được Python 3.10/3.11 cho Forge")
        log(f"dùng Python: {base}")
        run([uv, "venv", "--python", str(base), "--seed", str(Paths.venv)], env=env)
    ver = capture([str(py), "-c", "import sys;print('%d.%d.%d'%sys.version_info[:3])"])
    log(f"Python trong venv: {ver}")

    def uv_pip(*args: str) -> None:
        run([uv, "pip", "install", "--python", str(py), *args], env=env)

    have_torch = subprocess.run([str(py), "-c", "import torch, torchvision"], capture_output=True).returncode == 0
    if not have_torch:
        if cfg.test_mode or not has_nvidia_gpu():
            log("không có GPU NVIDIA → cài torch bản mặc định PyPI (CPU) – chỉ để kiểm thử")
            uv_pip(*TORCH_SPEC_GPU.split())
        else:
            log(f"cài {TORCH_SPEC_GPU} (CUDA 12.1)")
            uv_pip(*TORCH_SPEC_GPU.split(), "--index-url", TORCH_INDEX_GPU)
    if cfg.use_xformers and has_nvidia_gpu() and not cfg.test_mode:
        uv_pip(XFORMERS_SPEC, "--no-deps", "--extra-index-url", TORCH_INDEX_GPU)
    return py


def install_forge(cfg: Config) -> None:
    if not (Paths.forge / "launch.py").exists():
        log(f"clone Forge → {Paths.forge}")
        run(["git", "clone", "--filter=blob:none", FORGE_REPO, str(Paths.forge)])
    commit = (cfg.forge_commit or "").strip()
    if commit and commit.lower() != "latest":
        cur = capture(["git", "-C", str(Paths.forge), "rev-parse", "HEAD"])
        if cur != commit:
            log(f"checkout Forge commit {commit[:10]} (đã kiểm thử)")
            run(["git", "-C", str(Paths.forge), "fetch", "--depth", "1", "origin", commit], check=False, quiet=True)
            run(["git", "-C", str(Paths.forge), "checkout", "-q", commit])
    else:
        log("cập nhật Forge lên bản mới nhất")
        run(["git", "-C", str(Paths.forge), "pull", "--ff-only"], check=False)

    py = Paths.python()
    uv = str(ensure_uv())
    env = uv_env()
    log("cài requirements của Forge bằng uv (nhanh hơn pip nhiều lần)")
    run([uv, "pip", "install", "--python", str(py), "-r", str(Paths.forge / "requirements_versions.txt")], env=env)
    # CLIP dùng pkg_resources trong setup.py → build bằng setuptools 69.5.1 đã có trong venv (không isolation)
    run([uv, "pip", "install", "--python", str(py), "wheel", "joblib"], env=env)  # joblib: script soft-inpainting của Forge
    run([uv, "pip", "install", "--python", str(py), "--no-build-isolation", CLIP_PKG], env=env)
    if subprocess.run([str(py), "-c", "import open_clip"], capture_output=True).returncode != 0:
        run([uv, "pip", "install", "--python", str(py), "--no-build-isolation", OPENCLIP_PKG], env=env)
    if cfg.tunnel in ("ngrok", "both"):
        run([uv, "pip", "install", "--python", str(py), "ngrok"], env=env)
    for d in ("Stable-diffusion", "VAE", "Lora", "ControlNet", "ESRGAN", "embeddings"):
        (Paths.forge / "models" / d).mkdir(parents=True, exist_ok=True)
    (Paths.forge / "embeddings").mkdir(exist_ok=True)
    (Paths.forge / "outputs").mkdir(exist_ok=True)


def write_constraints() -> Path:
    """Tạo file constraints từ requirements_versions.txt của Forge.

    Đặt PIP_CONSTRAINT khi chạy install.py của các extension để chúng KHÔNG nâng cấp
    numpy / httpx / pillow / torch … lên bản phá vỡ Forge (ví dụ ADetailer → ultralytics → numpy 2.x).
    """
    req = Paths.forge / "requirements_versions.txt"
    pins = [ln.split("#")[0].strip() for ln in req.read_text().splitlines()]
    pins = [p for p in pins if "==" in p and not p.lower().startswith("torch")]
    pins += TORCH_SPEC_GPU.split()
    out = ROOT / "noobai_constraints.txt"
    out.write_text("\n".join(pins) + "\n", encoding="utf-8")
    return out


def forge_env() -> dict:
    """Biến môi trường an toàn khi chạy Forge từ trong notebook Colab/Jupyter."""
    return {
        # Colab đặt MPLBACKEND=module://matplotlib_inline.backend_inline → venv của Forge không có module này
        "MPLBACKEND": "Agg",
        "PYTHONUNBUFFERED": "1",
        "PYTHONIOENCODING": "utf-8",
        "GRADIO_ANALYTICS_ENABLED": "False",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "PYTHONWARNINGS": "ignore::DeprecationWarning",
    }


def pip_env() -> dict:
    return {
        "PIP_CONSTRAINT": str(write_constraints()),
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PIP_NO_INPUT": "1",
        "IIB_SKIP_OPTIONAL_DEPS": "1",  # image browser: bỏ qua hnswlib (tùy chọn, cần compiler)
    }


def prepare_forge_environment(cfg: Config) -> None:
    """Chạy launch.py --exit: clone repo phụ + chạy install.py của extension (có ràng buộc phiên bản)."""
    log("chuẩn bị môi trường Forge (repos phụ + installer của extension)…")
    args = ["--exit", "--skip-torch-cuda-test", "--skip-python-version-check"]
    if cfg.use_xformers and has_nvidia_gpu() and not cfg.test_mode:
        args.append("--xformers")
    env = {**pip_env(), **forge_env(), "COMMANDLINE_ARGS": " ".join(args)}
    rc = run([str(Paths.python()), "launch.py"], cwd=Paths.forge, env=env, check=False)
    if rc != 0:
        log(f"⚠ launch.py --exit trả về mã {rc} – xem log phía trên")
    # Ghim lại các gói cốt lõi phòng khi installer nào đó vẫn đổi phiên bản
    uv = str(ensure_uv())
    run([uv, "pip", "install", "--python", str(Paths.python()), "-r", str(Paths.forge / "requirements_versions.txt")],
        env=uv_env(), quiet=True)
    fix_opencv_if_no_libgl()


def fix_opencv_if_no_libgl() -> None:
    """Máy không có libGL (hiếm – Colab có sẵn) → chuyển opencv sang bản headless để import cv2 không lỗi."""
    import ctypes.util
    if ctypes.util.find_library("GL"):
        return
    py = str(Paths.python())
    if subprocess.run([py, "-c", "import cv2"], capture_output=True).returncode == 0:
        return
    log("không có libGL → chuyển opencv sang bản headless")
    uv = str(ensure_uv())
    installed = capture([uv, "pip", "freeze", "--python", py], env={**os.environ, **uv_env()})
    pkgs = [ln.split("==")[0] for ln in installed.splitlines() if ln.lower().startswith("opencv")]
    if pkgs:
        run([uv, "pip", "uninstall", "--python", py, *pkgs], env=uv_env(), quiet=True)
    want = ["opencv-python-headless"] + (["opencv-contrib-python-headless"] if any("contrib" in p for p in pkgs) else [])
    run([uv, "pip", "install", "--python", py, "--constraint", str(write_constraints()), *want], env=uv_env())


def install_extensions(cfg: Config) -> None:
    ext_dir = Paths.extensions()
    ext_dir.mkdir(parents=True, exist_ok=True)
    for key in cfg.extensions:
        if key not in EXTENSIONS:
            log(f"⚠ bỏ qua extension lạ: {key}")
            continue
        folder, url = EXTENSIONS[key]
        target = ext_dir / folder
        if (target / ".git").exists():
            log(f"✔ extension đã có: {folder}")
            continue
        log(f"cài extension {folder}")
        run(["git", "clone", "--depth", "1", "-q", url, str(target)])


def install_tag_csv(cfg: Config) -> Path | None:
    """Đưa file tag Danbooru+e621 (từ repo anime2) vào Tag Autocomplete."""
    if "tagcomplete" not in cfg.extensions:
        return None
    tags_dir = Paths.extensions() / EXTENSIONS["tagcomplete"][0] / "tags"
    tags_dir.mkdir(parents=True, exist_ok=True)
    dest = tags_dir / TAG_CSV_NAME
    if not cfg.use_repo_tag_csv:
        return None
    if dest.exists() and dest.stat().st_size > 1_000_000:
        log(f"✔ file tag đã có: {dest.name}")
        return dest
    local = Path(__file__).resolve().parent.parent / TAG_CSV_NAME  # khi chạy từ repo
    if local.exists():
        shutil.copy2(local, dest)
        log(f"✔ copy file tag từ repo local ({human(dest.stat().st_size)})")
        return dest
    for url in TAG_CSV_URLS:
        try:
            download_file(url, dest, cfg)
            if dest.stat().st_size > 1_000_000:
                return dest
            dest.unlink()
        except Exception as e:  # noqa: BLE001
            log(f"không tải được từ {url}: {e}")
    log("⚠ không lấy được file tag của repo → dùng danbooru_e621_merged.csv có sẵn của extension")
    return None


def install_localization(cfg: Config) -> Path | None:
    """Cài bản dịch tiếng Việt vào <forge>/localizations/vi_VN.json (Forge tự nạp qua cài đặt `localization`)."""
    dest_dir = Paths.forge / "localizations"
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{VI_LOCALE_NAME}.json"
    here = Path(__file__).resolve().parent
    candidates = [here / f"{VI_LOCALE_NAME}.json",                 # Colab: /content/vi_VN.json
                  here.parent / "colab" / f"{VI_LOCALE_NAME}.json",  # chạy từ repo
                  Path("/content") / f"{VI_LOCALE_NAME}.json"]
    for local in candidates:
        if local.exists() and local.stat().st_size > 1000:
            try:
                json.loads(local.read_text(encoding="utf-8"))
                shutil.copy2(local, dest)
                log(f"✔ bản dịch tiếng Việt: {dest} ({human(dest.stat().st_size)})")
                return dest
            except Exception as e:  # noqa: BLE001
                log(f"⚠ file dịch {local} lỗi: {e}")
    if VI_LOCALE_EMBEDDED:
        dest.write_text(json.dumps(VI_LOCALE_EMBEDDED, ensure_ascii=False, indent=2), encoding="utf-8")
        log(f"✔ bản dịch tiếng Việt (nhúng sẵn, {len(VI_LOCALE_EMBEDDED)} chuỗi) → {dest}")
        return dest
    if dest.exists() and dest.stat().st_size > 1000:
        log(f"✔ bản dịch tiếng Việt đã có: {dest.name}")
        return dest
    for url in VI_LOCALE_URLS:
        try:
            download_file(url, dest, cfg)
            json.loads(dest.read_text(encoding="utf-8"))
            log(f"✔ tải bản dịch tiếng Việt từ {url}")
            return dest
        except Exception as e:  # noqa: BLE001
            log(f"không tải được từ {url}: {e}")
            if dest.exists():
                dest.unlink()
    log("⚠ không có bản dịch tiếng Việt → giao diện sẽ hiển thị tiếng Anh")
    return None


def install_noob_tools(cfg: Config) -> Path | None:
    """Chép script noob_tools.py (dropdown độ phân giải chuẩn + tự bật Hires fix/ADetailer) vào <forge>/scripts/."""
    dest = Paths.forge / "scripts" / NOOB_TOOLS_NAME
    dest.parent.mkdir(parents=True, exist_ok=True)
    here = Path(__file__).resolve().parent
    for local in (here / NOOB_TOOLS_NAME, here.parent / "colab" / NOOB_TOOLS_NAME, Path("/content") / NOOB_TOOLS_NAME):
        if local.exists() and local.stat().st_size > 500:
            shutil.copy2(local, dest)
            log(f"✔ NoobAI Tools script: {dest}")
            return dest
    if NOOB_TOOLS_EMBEDDED:
        dest.write_text(NOOB_TOOLS_EMBEDDED, encoding="utf-8")
        log(f"✔ NoobAI Tools script (nhúng sẵn) → {dest}")
        return dest
    if dest.exists():
        log(f"✔ NoobAI Tools script đã có: {dest.name}")
        return dest
    for url in NOOB_TOOLS_URLS:
        try:
            download_file(url, dest, cfg)
            compile(dest.read_text(encoding="utf-8"), str(dest), "exec")
            log(f"✔ tải NoobAI Tools script từ {url}")
            return dest
        except Exception as e:  # noqa: BLE001
            log(f"không tải được từ {url}: {e}")
            if dest.exists():
                dest.unlink()
    log("⚠ không có noob_tools.py → không có dropdown độ phân giải chuẩn / tự fix")
    return None


def setup_drive(cfg: Config) -> dict:
    """Mount Google Drive (chỉ trong Colab) và liên kết thư mục outputs / models."""
    info = {"mounted": False, "outputs": str(Paths.forge / "outputs")}
    if not cfg.mount_drive:
        return info
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        log("không phải Colab → bỏ qua Google Drive")
        return info
    if not Paths.drive.exists():
        drive.mount("/content/drive")
    if not Paths.drive.exists():
        log("⚠ mount Drive thất bại")
        return info
    info["mounted"] = True
    base = Paths.drive / cfg.drive_folder
    if cfg.save_outputs_to_drive:
        out = base / "outputs"
        out.mkdir(parents=True, exist_ok=True)
        info["outputs"] = str(out)
        log(f"✔ ảnh sẽ lưu vào Drive: {out}")
    if cfg.link_drive_models:
        for sub, kind in (("models", "Stable-diffusion"), ("Lora", "Lora"), ("embeddings", "embeddings"),
                          ("VAE", "VAE"), ("ControlNet", "ControlNet")):
            src = base / sub
            src.mkdir(parents=True, exist_ok=True)
            link = (Paths.forge / "embeddings" / "drive") if kind == "embeddings" else (Paths.models(kind) / "drive")
            link.parent.mkdir(parents=True, exist_ok=True)
            if link.is_symlink() or link.exists():
                continue
            link.symlink_to(src, target_is_directory=True)
            log(f"✔ liên kết Drive/{cfg.drive_folder}/{sub} → {link.relative_to(Paths.forge)}")
    return info


# --------------------------------------------------------------------------------------
# Bước 2: tải tài nguyên
# --------------------------------------------------------------------------------------
def planned_downloads(cfg: Config) -> list[dict]:
    items: list[dict] = []
    if cfg.download_model:
        items += MODEL_FILES.values()
    if cfg.download_vae:
        items += VAE_FILES.values()
    if cfg.download_upscalers:
        items += UPSCALER_FILES.values()
    for key in CONTROLNET_SETS.get(cfg.controlnet, []):
        items.append(CONTROLNET_FILES[key])
    if "adetailer" in cfg.extensions:
        items += ADETAILER_FILES.values()
    for url in map(normalize_url, split_urls(cfg.extra_model_urls)):
        items.append(dict(kind="Stable-diffusion", name=_guess_filename(url), url=url, size=None, sha256=None))
    for url in map(normalize_url, split_urls(cfg.extra_lora_urls)):
        items.append(dict(kind="Lora", name=_guess_filename(url), url=url, size=None, sha256=None))
    for url in map(normalize_url, split_urls(cfg.extra_embedding_urls)):
        items.append(dict(kind="embeddings", name=_guess_filename(url), url=url, size=None, sha256=None))
    return items


def dest_for(item: dict) -> Path:
    if item["kind"] == "embeddings":
        return Paths.forge / "embeddings" / item["name"]
    return Paths.models(item["kind"]) / item["name"]


def download_resources(cfg: Config) -> None:
    items = planned_downloads(cfg)
    total = sum(i["size"] or 0 for i in items)
    banner(f"Tải tài nguyên: {len(items)} file (~{human(total)})")
    free = shutil.disk_usage(str(ROOT)).free
    log(f"dung lượng trống: {human(free)}")
    if total and free < total * 1.05:
        raise RuntimeError("Không đủ dung lượng đĩa cho các file đã chọn – bớt ControlNet hoặc dùng runtime khác")
    failed: list[tuple[dict, str]] = []
    for item in items:
        try:
            download_file(item["url"], dest_for(item), cfg, size=item["size"], sha256=item["sha256"])
        except Exception as e:  # noqa: BLE001
            log(f"❌ {item['name']}: {e}")
            failed.append((item, str(e)))
    if failed:
        print()
        log(f"⚠ {len(failed)}/{len(items)} file tải thất bại (chạy lại cell này để thử lại – tải tiếp từ chỗ dở):")
        for item, err in failed:
            log(f"   - {item['kind']}/{item['name']}")
        if any(f[0]["kind"] == "Stable-diffusion" and f[0]["name"] == MODEL_FILES["noobai_xl_eps_1_1"]["name"]
               for f in failed):
            raise RuntimeError("Model chính NoobAI-XL-v1.1 chưa tải được – kiểm tra mạng/dung lượng rồi chạy lại.")
    else:
        log("✔ đã tải đủ tất cả tài nguyên")


# --------------------------------------------------------------------------------------
# Bước 3: cấu hình mặc định cho NoobAI (config.json / ui-config.json / styles.csv)
# --------------------------------------------------------------------------------------
def _merge_json(path: Path, updates: dict) -> None:
    data = {}
    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception:  # noqa: BLE001
            data = {}
    data.update(updates)
    path.write_text(json.dumps(data, indent=4, ensure_ascii=False), encoding="utf-8")


def _ordinal(n: int) -> str:
    return {1: "1st", 2: "2nd", 3: "3rd"}.get(n, f"{n}th")


def adetailer_ui_config(level: str, max_tabs: int = 3) -> dict:
    """Khoá ui-config.json của ADetailer (txt2img + img2img) cho một mức AD_LEVELS.
    Khoá = <tab>/<label>/value theo nhãn gốc tiếng Anh của ADetailer (bản dịch chỉ đổi hiển thị)."""
    lv = AD_LEVELS[ad_level_key(level)]
    ui: dict = {}
    for tab in ("txt2img", "img2img"):
        ui[f"customscript/!adetailer.py/{tab}/ADetailer/value"] = bool(lv["enable"])
        for n in range(1, max_tabs + 1):
            suf = "" if n == 1 else f" {_ordinal(n)}"
            t = lv["tabs"][n - 1] if n - 1 < len(lv["tabs"]) else None
            ui[f"{tab}/Enable this tab ({_ordinal(n)})/value"] = True
            ui[f"{tab}/ADetailer detector{suf}/value"] = t["models"][0] if t else "None"
            ui[f"{tab}/ad_prompt{suf}/value"] = t["prompt"] if t else ""
            ui[f"{tab}/ad_negative_prompt{suf}/value"] = t["negative"] if t else ""
            ui[f"{tab}/Detection model confidence threshold{suf}/value"] = t["confidence"] if t else 0.3
            ui[f"{tab}/Mask erosion (-) / dilation (+){suf}/value"] = t["dilate"] if t else 4
            ui[f"{tab}/Inpaint mask blur{suf}/value"] = t["blur"] if t else 4
            ui[f"{tab}/Inpaint denoising strength{suf}/value"] = t["denoise"] if t else 0.4
            ui[f"{tab}/Inpaint only masked padding, pixels{suf}/value"] = t["padding"] if t else 32
    return ui


RECOMMENDED_PARAMS = dict(default_width=832, default_height=1216, default_steps=28, default_cfg=6.0,
                          default_sampler="Euler a", default_scheduler="Automatic", clip_skip=2,
                          auto_hires_fix=True, hires_upscale_by=1.5, hires_denoise=0.4, auto_adetailer="standard")


def write_settings(cfg: Config, drive_info: dict | None = None) -> None:
    banner("Ghi cấu hình mặc định cho NoobAI-XL")
    if str(cfg.param_mode).lower().startswith("rec"):
        for k, v in RECOMMENDED_PARAMS.items():
            setattr(cfg, k, v)
        log("thông số: bộ KHUYÊN DÙNG NoobAI (Euler a · 28 steps · CFG 6 · 832×1216 · Clip skip 2 · Hires ×1.5/0.4 · ADetailer mặt+mắt+tay)")
    else:
        log(f"thông số: TỰ CHỈNH ({cfg.default_sampler} · {cfg.default_steps} steps · CFG {cfg.default_cfg} · "
            f"{cfg.default_width}×{cfg.default_height} · Hires {'on' if cfg.auto_hires_fix else 'off'} · ADetailer {cfg.auto_adetailer})")
    drive_info = drive_info or {}
    outputs = Path(drive_info.get("outputs") or (Paths.forge / "outputs"))
    vae_path = Paths.models("VAE") / VAE_FILES["sdxl_vae_fp16_fix"]["name"]
    model_name = MODEL_FILES["noobai_xl_eps_1_1"]["name"]

    settings = {
        "sd_model_checkpoint": model_name,
        "forge_preset": "xl",
        "forge_additional_modules": [str(vae_path)] if cfg.download_vae else [],
        "forge_unet_storage_dtype": "Automatic",
        "CLIP_stop_at_last_layers": cfg.clip_skip,
        "xl_t2i_width": cfg.default_width,
        "xl_t2i_height": cfg.default_height,
        "xl_t2i_cfg": cfg.default_cfg,
        "xl_t2i_hr_cfg": cfg.default_cfg,
        "xl_t2i_sampler": cfg.default_sampler,
        "xl_t2i_scheduler": cfg.default_scheduler,
        "xl_i2i_width": 1024,
        "xl_i2i_height": 1024,
        "xl_i2i_cfg": cfg.default_cfg,
        "xl_i2i_sampler": cfg.default_sampler,
        "xl_i2i_scheduler": cfg.default_scheduler,
        "samples_format": "png",
        "save_images_add_number": True,
        "samples_filename_pattern": "[seed]-[prompt_spaces]",
        "outdir_txt2img_samples": str(outputs / "txt2img-images"),
        "outdir_img2img_samples": str(outputs / "img2img-images"),
        "outdir_extras_samples": str(outputs / "extras-images"),
        "outdir_txt2img_grids": str(outputs / "txt2img-grids"),
        "outdir_img2img_grids": str(outputs / "img2img-grids"),
        "outdir_save": str(outputs / "saved"),
        "live_previews_enable": True,
        "show_progress_every_n_steps": 5,
        "show_progressbar": True,
        "quicksettings_list": ["sd_model_checkpoint", "sd_vae", "CLIP_stop_at_last_layers"],
        "upscaler_for_img2img": "4x-AnimeSharp" if cfg.download_upscalers else "None",
        "hires_fix_show_sampler": True,
        "hires_fix_show_prompts": True,
        "token_merging_ratio": 0.0,
        "emphasis": "Original",
        "always_discard_next_to_last_sigma": False,
        "sdxl_crop_top": 0,
        "sdxl_crop_left": 0,
        "sdxl_refiner_low_aesthetic_score": 2.5,
        "sdxl_refiner_high_aesthetic_score": 6.0,
        "disable_all_extensions": "none",
        # Bộ nhớ GPU (theo GPU_MODE) – xem gpu_plan()
        **gpu_plan(cfg)["settings"],
        # Ngôn ngữ giao diện (Settings ▸ User interface ▸ Localization)
        "localization": VI_LOCALE_NAME if (str(cfg.ui_language).lower().startswith("vi") and
                                           (Paths.forge / "localizations" / f"{VI_LOCALE_NAME}.json").exists())
                        else "None",
        # Tag Autocomplete
        "tac_tagFile": TAG_CSV_NAME if (cfg.use_repo_tag_csv and
                                         (Paths.extensions() / EXTENSIONS["tagcomplete"][0] / "tags" / TAG_CSV_NAME).exists())
                       else "danbooru_e621_merged.csv",
        "tac_active": True,
        "tac_maxResults": 8,
        "tac_showAllResults": False,
        "tac_replaceUnderscores": False,
        "tac_escapeParentheses": True,
        "tac_appendComma": True,
        "tac_appendSpace": True,
        "tac_alias.searchByAlias": True,
        "tac_useWildcards": True,
        "tac_useEmbeddings": True,
        "tac_useLoras": True,
        "tac_useLycos": True,
        "tac_extra.extraFile": "extra-quality-tags.csv",
        "tac_chantFile": "noob_characters-chants.json",
        # ADetailer mặc định: 3 bộ (mặt / mắt / tay); vùng inpaint khớp bucket SDXL theo tỉ lệ bbox (nét hơn)
        "ad_max_models": 3,
        "ad_save_previews": False,
        "ad_match_inpaint_bbox_size": "Strict (SDXL only)",
        "ad_same_seed_for_each_tab": False,
    }
    _merge_json(Paths.forge / "config.json", settings)
    log(f"✔ config.json (ngôn ngữ giao diện: {settings['localization']})")

    ui = {
        "txt2img/Prompt/value": NOOB_POSITIVE_PREFIX,
        "txt2img/Negative prompt/value": NOOB_NEGATIVE_FULL,
        "img2img/Prompt/value": NOOB_POSITIVE_PREFIX,
        "img2img/Negative prompt/value": NOOB_NEGATIVE_FULL,
        "customscript/sampler.py/txt2img/Sampling steps/value": cfg.default_steps,
        "customscript/sampler.py/img2img/Sampling steps/value": cfg.default_steps,
        "txt2img/Upscaler/value": "4x-AnimeSharp" if cfg.download_upscalers else "Latent",
        "txt2img/Upscale by/value": float(cfg.hires_upscale_by),
        "txt2img/Hires steps/value": 14,
        "txt2img/Denoising strength/value": float(cfg.hires_denoise),
        "txt2img/Batch count/value": 1,
        "txt2img/Batch size/value": 1,
        "img2img/Denoising strength/value": 0.5,
    }
    # ✨ Tự fix: ADetailer – bộ 1 mặt, bộ 2 mắt, bộ 3 tay theo mức AD_LEVELS (dropdown 🩹 trong UI đổi được)
    ad = ad_level_key(cfg.auto_adetailer) if "adetailer" in cfg.extensions else "off"
    if "adetailer" in cfg.extensions:
        ui.update(adetailer_ui_config(ad))
    _merge_json(Paths.forge / "ui-config.json", ui)
    log(f"✔ ui-config.json (prompt / steps / hires / ADetailer mức '{ad}': {AD_LEVELS[ad]['label']})")

    tools = {
        "auto_hires_fix": bool(cfg.auto_hires_fix),
        "auto_adetailer": ad,
        "fix_level": ad,
        "ad_levels": AD_LEVELS,
        "param_mode": "recommended" if str(cfg.param_mode).lower().startswith("rec") else "custom",
    }
    (Paths.forge / "noob_tools.json").write_text(json.dumps(tools, ensure_ascii=False, indent=2), encoding="utf-8")
    log(f"✔ noob_tools.json (tự fix: hires={tools['auto_hires_fix']}, adetailer={tools['auto_adetailer']})")

    styles = [
        ("Noob ✦ Quality (chuẩn)", NOOB_POSITIVE_PREFIX + "{prompt}", NOOB_NEGATIVE),
        ("Noob ✦ Quality + Safe", NOOB_POSITIVE_PREFIX + "general, {prompt}", NOOB_NEGATIVE + ", nsfw, explicit, sensitive"),
        ("Noob ✦ Năm: newest", "{prompt}, newest", "old, early"),
        ("Noob ✦ Năm: recent", "{prompt}, recent", "old, early"),
        ("Noob ✦ Năm: mid", "{prompt}, mid", ""),
        ("Noob ✦ Năm: early", "{prompt}, early", "newest"),
        ("Noob ✦ Năm: old", "{prompt}, old", "newest"),
        ("Noob ✦ Phong cách: sketch", "{prompt}, sketch, lineart, monochrome, greyscale", "colored"),
        ("Noob ✦ Phong cách: flat color", "{prompt}, flat color, limited palette, clean lines", "gradient, photorealistic"),
        ("Noob ✦ Phong cách: watercolor", "{prompt}, watercolor \\(medium\\), traditional media, soft colors", ""),
        ("Noob ✦ Phong cách: cinematic", "{prompt}, cinematic lighting, depth of field, volumetric lighting, dramatic shadows", "flat lighting"),
        ("Noob ✦ Phong cách: 90s anime", "{prompt}, 1990s \\(style\\), retro artstyle, film grain", ""),
        ("Noob ✦ Negative: tay/anatomy", "{prompt}", NOOB_NEGATIVE + ", extra fingers, fewer fingers, extra digits, missing fingers, malformed limbs, extra arms, extra legs, deformed, disfigured"),
        ("Noob ✦ Fix mắt", "{prompt}, " + FIX_EYES_POS, FIX_EYES_NEG),
        ("Noob ✦ Fix tay", "{prompt}, " + FIX_HANDS_POS, FIX_HANDS_NEG),
        ("Noob ✦ Fix chân", "{prompt}, " + FIX_FEET_POS, FIX_FEET_NEG),
        ("Noob ✦ Fix mắt+tay+chân", "{prompt}, " + FIX_EYES_POS + ", " + FIX_HANDS_POS + ", " + FIX_FEET_POS, FIX_ALL_NEG),
        ("Noob ✦ Fix mắt+tay+chân (gọn)", "{prompt}, detailed eyes, detailed hands, detailed feet", "bad anatomy, bad hands, bad feet, extra digits, fewer digits, extra limbs, cross-eyed, uneven eyes"),
        ("Noob ✦ Negative: chữ/watermark", "{prompt}", "text, watermark, signature, artist name, logo, patreon username, web address, speech bubble"),
    ]
    styles_path = Paths.forge / "styles.csv"
    existing = styles_path.read_text(encoding="utf-8") if styles_path.exists() else ""
    import csv
    with styles_path.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not existing:
            w.writerow(["name", "prompt", "negative_prompt"])
        for name, p, n in styles:
            if name not in existing:
                w.writerow([name, p, n])
    log(f"✔ styles.csv ({len(styles)} preset NoobAI)")


# --------------------------------------------------------------------------------------
# Bước 4: tự kiểm tra
# --------------------------------------------------------------------------------------
def self_test(cfg: Config) -> bool:
    banner("Tự kiểm tra bộ cài")
    results: list[tuple[bool, str]] = []

    def check(ok: bool, msg: str) -> None:
        results.append((ok, msg))
        print(("  ✅ " if ok else "  ❌ ") + msg, flush=True)

    py = Paths.python()
    check(py.exists(), f"Python venv: {py}")
    if py.exists():
        ver = capture([str(py), "-c", "import sys;print('%d.%d'%sys.version_info[:2])"])
        check(ver in ("3.10", "3.11"), f"Phiên bản Python venv = {ver} (Forge cần 3.10/3.11)")
        probe = capture([str(py), "-c",
                         "import torch,json;print(json.dumps({'torch':torch.__version__,'cuda':torch.cuda.is_available(),"
                         "'dev':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}))"])
        try:
            t = json.loads(probe)
            check(True, f"torch {t['torch']} – thiết bị: {t['dev']}")
            if has_nvidia_gpu() and not cfg.test_mode:
                check(t["cuda"], "torch nhìn thấy GPU CUDA")
        except Exception:  # noqa: BLE001
            check(False, f"import torch thất bại: {probe}")
        for mod in ("gradio", "transformers", "diffusers", "safetensors", "clip", "open_clip", "cv2"):
            rc = subprocess.run([str(py), "-c", f"import {mod}"], capture_output=True).returncode
            check(rc == 0, f"module {mod}")
        rc = subprocess.run([str(py), "-c", "import pytorch_lightning, matplotlib"], capture_output=True,
                            env={**os.environ, "MPLBACKEND": "module://matplotlib_inline.backend_inline", **forge_env()}).returncode
        check(rc == 0, "import pytorch_lightning/matplotlib với env kiểu Colab (MPLBACKEND)")
        npv = capture([str(py), "-c", "import numpy;print(numpy.__version__)"])
        check(npv.startswith("1."), f"numpy {npv} (Forge cần numpy 1.x – extension không được nâng lên 2.x)")
        hx = capture([str(py), "-c", "import httpx;print(httpx.__version__)"])
        check(hx == "0.24.1", f"httpx {hx} (Forge ghim 0.24.1)")
        if "adetailer" in cfg.extensions:
            rc = subprocess.run([str(py), "-c", "import ultralytics"], capture_output=True).returncode
            check(rc == 0, "ultralytics (ADetailer)")
        if cfg.use_xformers and has_nvidia_gpu() and not cfg.test_mode:
            rc = subprocess.run([str(py), "-c", "import xformers"], capture_output=True).returncode
            check(rc == 0, "xformers")

    check((Paths.forge / "launch.py").exists(), f"Forge tại {Paths.forge}")
    commit = capture(["git", "-C", str(Paths.forge), "rev-parse", "--short", "HEAD"]) if Paths.forge.exists() else "?"
    log(f"Forge commit: {commit}")
    for key in cfg.extensions:
        folder = EXTENSIONS.get(key, (key,))[0]
        check((Paths.extensions() / folder).exists(), f"extension {folder}")

    for item in planned_downloads(cfg):
        p = dest_for(item)
        if p.exists():
            size_ok = item["size"] is None or p.stat().st_size == item["size"]
            check(size_ok, f"{item['kind']}/{item['name']} ({human(p.stat().st_size)})")
        else:
            check(False, f"{item['kind']}/{item['name']} CHƯA tải")

    if "tagcomplete" in cfg.extensions and cfg.use_repo_tag_csv:
        csv_path = Paths.extensions() / EXTENSIONS["tagcomplete"][0] / "tags" / TAG_CSV_NAME
        if csv_path.exists():
            n_lines = sum(1 for _ in csv_path.open("rb"))
            bad = 0
            import csv as _csv
            with csv_path.open(encoding="utf-8", newline="") as f:
                for i, row in enumerate(_csv.reader(f)):
                    if len(row) < 3 or not row[1].isdigit() or not row[2].isdigit():
                        bad += 1
                    if i > 5000:
                        break
            check(bad == 0, f"file tag autocomplete hợp lệ ({n_lines:,} tag, định dạng tag,category,count,aliases)")
        else:
            check(False, "file tag của repo (sẽ dùng file mặc định của extension)")

    for name in ("config.json", "ui-config.json", "styles.csv"):
        p = Paths.forge / name
        ok = p.exists()
        if ok and name.endswith(".json"):
            try:
                json.loads(p.read_text(encoding="utf-8"))
            except Exception:  # noqa: BLE001
                ok = False
        check(ok, f"{name}")

    plan = gpu_plan(cfg)
    check(True, f"chế độ GPU '{plan['mode']}': {' '.join(plan['flags']) or 'mặc định Forge'}"
                f" | GPU Weights ≈ {max(plan['vram'] - plan['settings']['forge_inference_memory'], 0)} MB")

    tools_py = Paths.forge / "scripts" / NOOB_TOOLS_NAME
    tools_ok = tools_py.exists()
    if tools_ok:
        try:
            compile(tools_py.read_text(encoding="utf-8"), str(tools_py), "exec")
        except Exception:  # noqa: BLE001
            tools_ok = False
    check(tools_ok and (Paths.forge / "noob_tools.json").exists(),
          f"NoobAI Tools (độ phân giải chuẩn + tự fix: hires={cfg.auto_hires_fix}, adetailer={cfg.auto_adetailer})")

    if str(cfg.ui_language).lower().startswith("vi"):
        loc = Paths.forge / "localizations" / f"{VI_LOCALE_NAME}.json"
        n_keys = 0
        try:
            n_keys = len(json.loads(loc.read_text(encoding="utf-8")))
        except Exception:  # noqa: BLE001
            pass
        try:
            applied = json.loads((Paths.forge / "config.json").read_text(encoding="utf-8")).get("localization")
        except Exception:  # noqa: BLE001
            applied = None
        check(n_keys > 100 and applied == VI_LOCALE_NAME,
              f"giao diện tiếng Việt ({n_keys} chuỗi dịch, localization={applied})")

    free = shutil.disk_usage(str(ROOT)).free
    check(free > 3 * 1024 ** 3, f"dung lượng trống còn {human(free)}")

    failed = [m for ok, m in results if not ok]
    print()
    if failed:
        print(f"⚠ {len(failed)} mục chưa đạt:", flush=True)
        for m in failed:
            print("   -", m)
    else:
        print("🎉 Tất cả kiểm tra đều đạt – sẵn sàng khởi chạy!", flush=True)
    return not failed


# --------------------------------------------------------------------------------------
# Bước 5: khởi chạy Forge + gradio.live
# --------------------------------------------------------------------------------------
def build_args(cfg: Config) -> list[str]:
    # --skip-install: installer của extension đã chạy ở bước cài (prepare_forge_environment) → khởi động nhanh,
    # không bị pip đổi phiên bản gói mỗi lần chạy. Thêm extension mới thì chạy lại cell "Cài đặt".
    args = ["--listen", "--port", str(cfg.port), "--enable-insecure-extension-access", "--api",
            "--no-download-sd-model", "--skip-version-check", "--skip-python-version-check", "--skip-install",
            "--theme", cfg.theme]
    if cfg.tunnel in ("gradio.live", "both"):
        args.append("--share")
    if cfg.tunnel in ("ngrok", "both") and cfg.ngrok_token:
        args += ["--ngrok", cfg.ngrok_token]
    if cfg.gradio_auth:
        args += ["--gradio-auth", cfg.gradio_auth]
    if cfg.use_xformers and has_nvidia_gpu() and not cfg.test_mode:
        args.append("--xformers")
    if cfg.test_mode or not has_nvidia_gpu():
        args += ["--skip-torch-cuda-test", "--always-cpu"]
    else:
        args += gpu_plan(cfg)["flags"]
    if cfg.extra_args:
        args += cfg.extra_args.split()
    return args


URL_PATTERNS = [
    re.compile(r"https://[a-z0-9-]+\.gradio\.live"),
    re.compile(r"https://[a-z0-9.-]+\.ngrok(?:-free)?\.(?:app|io|dev)"),
]


LAUNCH_LOG = ROOT / "noobai_launch.log"

KNOWN_ERRORS = [
    ("Your device does not support the current version of Torch/CUDA",
     "Torch không thấy GPU. Kiểm tra Runtime ▸ Change runtime type = GPU, rồi chạy lại cell 3 (cài) và cell 6."),
    ("CUDA out of memory", "Hết VRAM. Giảm kích thước ảnh / tắt ControlNet, hoặc thêm `--always-low-vram` vào EXTRA_ARGS."),
    ("Cannot find empty port", "Cổng 7860 đang bị một Forge cũ chiếm. Chạy lại cell 6 (bộ cài sẽ tự dọn tiến trình cũ)."),
    ("address already in use", "Cổng 7860 đang bị chiếm. Chạy lại cell 6 (bộ cài sẽ tự dọn tiến trình cũ)."),
    ("ModuleNotFoundError", "Thiếu thư viện Python. Chạy lại cell 3 (cài đặt) rồi cell 6."),
    ("Could not create share link", "gradio.live không tạo được link. Dùng LINK DỰ PHÒNG (Colab proxy) in bên dưới, hoặc đổi TUNNEL=ngrok."),
    ("No module named 'xformers'", "xformers lỗi. Đặt USE_XFORMERS=False ở cell 1, chạy lại cell 1 và 6."),
    ("Killed", "Tiến trình bị hệ thống kill (hết RAM). Dùng runtime High-RAM hoặc tắt bớt extension."),
    ("No checkpoints found", "Chưa có model. Chạy lại cell 4 (tải tài nguyên)."),
    ("matplotlib_inline.backend_inline", "Biến MPLBACKEND của Colab rò sang Forge – bản lib mới đã sửa; chạy lại cell 2 rồi cell 6."),
]


def kill_stale_forge() -> None:
    """Dọn tiến trình Forge cũ còn chạy ngầm (ví dụ cell trước bị văng) để không kẹt cổng."""
    pids = forge_pids()
    for pid in pids:
        try:
            os.killpg(os.getpgid(pid), signal.SIGTERM)
        except Exception:  # noqa: BLE001
            try:
                os.kill(pid, signal.SIGTERM)
            except Exception:  # noqa: BLE001
                pass
    if pids:
        log(f"đã dừng {len(pids)} tiến trình Forge cũ")
        time.sleep(3)
        for pid in forge_pids():
            try:
                os.kill(pid, signal.SIGKILL)
            except Exception:  # noqa: BLE001
                pass


def colab_proxy_url(port: int) -> str | None:
    """Link dự phòng qua proxy của Colab (chỉ người đang đăng nhập Colab mở được)."""
    try:
        from google.colab.output import eval_js  # type: ignore
        return str(eval_js(f"google.colab.kernel.proxyPort({port})", timeout_sec=20)).rstrip("/")
    except Exception:  # noqa: BLE001
        return None


def diagnose(lines: list[str], returncode: int | None) -> None:
    print("\n" + "!" * 70, flush=True)
    print(f"!  Forge đã THOÁT (mã {returncode}). Log đầy đủ: {LAUNCH_LOG}", flush=True)
    text = "\n".join(lines)
    hints = [h for key, h in KNOWN_ERRORS if key in text]
    if hints:
        print("!  Nguyên nhân có thể:", flush=True)
        for h in dict.fromkeys(hints):
            print(f"!   • {h}", flush=True)
    err_lines = [ln for ln in lines if re.search(r"Traceback|Error|error:|Exception|Killed", ln)]
    if err_lines:
        print("!  Các dòng lỗi cuối:", flush=True)
        for ln in err_lines[-8:]:
            print("!    " + ln.rstrip()[:200], flush=True)
    print("!  → Chạy cell 🩺 Chẩn đoán để xem 80 dòng log cuối, hoặc gửi file log để được hỗ trợ.", flush=True)
    print("!" * 70 + "\n", flush=True)


URLS_FILE = ROOT / "noobai_urls.json"
PID_FILE = ROOT / "noobai_forge.pid"
# Sau khi UI đã lên, chỉ in các dòng log quan trọng (tránh làm nặng trình duyệt Colab → dễ mất kết nối)
IMPORTANT_LOG = re.compile(r"(error|exception|traceback|warning|out of memory|outofmemory|cuda|gradio\.live|ngrok|"
                           r"running on|total progress|model loaded|loading weights|interrupted|skipped|"
                           r"killed|stopped|restart)", re.I)


def forge_pids() -> list[int]:
    """PID các tiến trình Forge (launch.py / webui.py chạy trong thư mục Forge) đang chạy nền."""
    me = os.getpid()
    forge_dir = str(Paths.forge.resolve())
    pids = []
    for pid_dir in Path("/proc").glob("[0-9]*"):
        try:
            pid = int(pid_dir.name)
            if pid == me:
                continue
            cmd = (pid_dir / "cmdline").read_bytes().replace(b"\0", b" ").decode(errors="ignore")
            if "launch.py" not in cmd and "webui.py" not in cmd:
                continue
            if forge_dir in cmd or os.path.realpath(str(pid_dir / "cwd")) == forge_dir:
                pids.append(pid)
        except Exception:  # noqa: BLE001
            pass
    return pids


def port_open(port: int) -> bool:
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", int(port))) == 0


def forge_alive(cfg: Config | None = None) -> bool:
    cfg = cfg or Config.load()
    return bool(forge_pids()) and port_open(cfg.port)


def _save_urls(urls: list[str], proxy: str | None) -> None:
    try:
        URLS_FILE.write_text(json.dumps({"public": urls, "proxy": proxy, "time": time.time()}), encoding="utf-8")
    except Exception:  # noqa: BLE001
        pass


def saved_urls() -> dict:
    try:
        return json.loads(URLS_FILE.read_text(encoding="utf-8"))
    except Exception:  # noqa: BLE001
        return {}


def urls_from_log() -> list[str]:
    if not LAUNCH_LOG.exists():
        return []
    text = LAUNCH_LOG.read_text(encoding="utf-8", errors="replace")
    found: list[str] = []
    for pat in URL_PATTERNS:
        for m in pat.findall(text):
            if m not in found:
                found.append(m)
    return found


def show_ready(url: str, label: str = "GIAO DIỆN ĐÃ SẴN SÀNG") -> None:
    print("\n" + "★" * 70 + f"\n★  {label} → {url}\n" + "★" * 70 + "\n", flush=True)


def stop_forge() -> int:
    """Dừng hẳn Forge đang chạy nền."""
    pids = forge_pids()
    for pid in pids:
        try:
            os.killpg(os.getpgid(pid), signal.SIGTERM)
        except Exception:  # noqa: BLE001
            try:
                os.kill(pid, signal.SIGTERM)
            except Exception:  # noqa: BLE001
                pass
    time.sleep(3)
    for pid in forge_pids():
        try:
            os.kill(pid, signal.SIGKILL)
        except Exception:  # noqa: BLE001
            pass
    if pids:
        log(f"⏹ đã dừng Forge (pid {pids})")
    else:
        log("Forge không chạy")
    return len(pids)


def launch(cfg: Config, *, wait: bool = True, extra_args: Iterable[str] = (), ready_timeout: int = 0,
           quiet_after_ready: bool = True):
    """Khởi chạy Forge dưới dạng tiến trình nền ĐỘC LẬP (session riêng, log ghi thẳng ra file).

    - Cell Colab chỉ "theo dõi" log: trình duyệt mất kết nối / cell bị ngắt → Forge vẫn chạy, link vẫn sống.
    - Dùng reconnect() (cell 🔗) để lấy lại link, stop_forge() để dừng hẳn.
    """
    banner("Khởi chạy NoobAI-XL 1.1 (Forge)")
    py = Paths.python()
    if not py.exists() or not (Paths.forge / "launch.py").exists():
        raise RuntimeError("Chưa cài đặt – hãy chạy cell 3 (Cài đặt) trước.")
    kill_stale_forge()
    fix_opencv_if_no_libgl()
    args = build_args(cfg) + list(extra_args)
    plan = gpu_plan(cfg)
    log(f"GPU: {gpu_name()} | chế độ GPU: {plan['mode']}" + (f" (compute {plan['cc']})" if plan['cc'] else ""))
    for w in plan["why"]:
        log("   · " + w)
    log("COMMANDLINE_ARGS = " + " ".join(args))
    log(f"log được ghi vào {LAUNCH_LOG}")
    env = {**os.environ, **pip_env(), **forge_env(), "COMMANDLINE_ARGS": " ".join(args)}
    for k in list(env):
        if k.startswith(("JPY_", "IPY", "COLAB_BACKEND", "KERNEL_")):
            env.pop(k, None)
    if cfg.hf_token:
        env["HF_TOKEN"] = cfg.hf_token
    if URLS_FILE.exists():
        URLS_FILE.unlink()
    logf = LAUNCH_LOG.open("wb")
    proc = subprocess.Popen([str(py), str(Paths.forge / "launch.py")], cwd=str(Paths.forge), env=env,
                            stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
                            start_new_session=True)  # session riêng → không chết theo cell/kernel
    logf.close()
    PID_FILE.write_text(str(proc.pid), encoding="utf-8")
    log(f"Forge chạy nền, pid {proc.pid} (mất kết nối Colab cũng không tắt; dừng hẳn bằng cell 🔗 → Dừng)")
    public_urls = follow_log(cfg, proc, wait=wait, ready_timeout=ready_timeout, quiet_after_ready=quiet_after_ready)
    return proc, public_urls


def follow_log(cfg: Config, proc: "subprocess.Popen | None" = None, *, wait: bool = True, ready_timeout: int = 0,
               quiet_after_ready: bool = True, from_start: bool = True) -> list[str]:
    """Theo dõi noobai_launch.log, in link khi UI lên; trả về danh sách link công khai."""
    public_urls: list[str] = list(saved_urls().get("public", [])) if not from_start else []
    local_ready = not from_start and bool(saved_urls())
    share_failed = False
    proxy = None
    tail: list[str] = []
    t0 = time.time()
    buf = ""

    def alive() -> bool:
        if proc is not None:
            return proc.poll() is None
        return bool(forge_pids())

    def handle(line: str) -> bool:
        nonlocal local_ready, share_failed, proxy
        tail.append(line)
        if len(tail) > 400:
            del tail[:100]
        important = bool(IMPORTANT_LOG.search(line))
        if not local_ready or not quiet_after_ready or important:
            try:
                sys.stdout.write(line.rsplit("\r", 1)[-1] if "\r" in line else line)
                sys.stdout.flush()
            except Exception:  # noqa: BLE001
                pass
        for pat in URL_PATTERNS:
            for m in pat.findall(line):
                if m not in public_urls:
                    public_urls.append(m)
                    show_ready(m)
                    _save_urls(public_urls, proxy)
        if "Could not create share link" in line:
            share_failed = True
        if "Running on local URL" in line and not local_ready:
            local_ready = True
            proxy = colab_proxy_url(cfg.port)
            _save_urls(public_urls, proxy)
            if proxy:
                show_ready(proxy, "LINK DỰ PHÒNG (Colab proxy – mở bằng tài khoản đang chạy Colab)")
                print("   (link gradio.live sẽ hiện thêm bên dưới nếu tạo được – thường mất 10–60 s)\n", flush=True)
            if quiet_after_ready:
                print("   (từ đây chỉ in các dòng log quan trọng; log đầy đủ ở", LAUNCH_LOG, ")\n", flush=True)
        if share_failed and local_ready and cfg.tunnel in ("gradio.live", "both") and \
                "gradio.live" not in "".join(public_urls):
            print("\n⚠ gradio.live không tạo được link lúc này. Hãy dùng LINK DỰ PHÒNG ở trên, "
                  "hoặc đặt TUNNEL=ngrok (cell 1) rồi chạy lại cell 6.\n", flush=True)
            share_failed = False
        if not wait and (local_ready and (public_urls or cfg.tunnel == "none")):
            return False
        if ready_timeout and time.time() - t0 > ready_timeout:
            return False
        return True

    interrupted = False
    try:
        with LAUNCH_LOG.open("rb") as f:
            if not from_start:
                f.seek(0, os.SEEK_END)
            while True:
                chunk = f.read(65536)
                if chunk:
                    buf += chunk.decode("utf-8", errors="replace")
                    *lines, buf = buf.split("\n")
                    for ln in lines:
                        if not handle(ln + "\n"):
                            return public_urls
                    continue
                if not alive():
                    if buf:
                        handle(buf + "\n")
                    break
                if not wait and ready_timeout and time.time() - t0 > ready_timeout:
                    return public_urls
                time.sleep(0.5)
    except KeyboardInterrupt:
        interrupted = True
        print("\n⏸ Đã ngừng theo dõi log – Forge VẪN CHẠY NỀN, link vẫn dùng được.\n"
              "   Lấy lại link: cell 🔗 (Nối lại). Dừng hẳn Forge: cell 🔗 với ACTION = Dừng.", flush=True)
    if wait and not interrupted and not alive():
        rc = proc.returncode if proc is not None else None
        diagnose(tail, rc)
    return public_urls


def reconnect(cfg: Config | None = None, action: str = "link") -> None:
    """Cell 🔗: 'link' = in lại link (khởi chạy lại nếu Forge đã chết) | 'restart' | 'stop'."""
    cfg = cfg or Config.load()
    banner("🔗 Nối lại với Forge")
    if action == "stop":
        stop_forge()
        return
    if action == "restart":
        stop_forge()
        launch(cfg)
        return
    if forge_alive(cfg):
        info = saved_urls()
        urls = info.get("public") or urls_from_log()
        proxy = colab_proxy_url(cfg.port) or info.get("proxy")
        log(f"Forge đang chạy (pid {forge_pids()}, cổng {cfg.port} mở) – link hiện tại:")
        for u in urls:
            show_ready(u)
        if proxy:
            show_ready(proxy, "LINK DỰ PHÒNG (Colab proxy)")
        if not urls and cfg.tunnel != "none":
            log("⚠ chưa thấy link gradio.live trong log – gradio.live có thể đang lỗi; dùng link dự phòng, "
                "hoặc ACTION = Khởi động lại.")
        print("\nĐang tiếp tục theo dõi log (bấm ⏹ để ngừng theo dõi, Forge vẫn chạy)…\n", flush=True)
        follow_log(cfg, None, wait=True, from_start=False)
    else:
        if forge_pids():
            log(f"Forge đang khởi động (pid {forge_pids()}), cổng {cfg.port} chưa mở – theo dõi log:")
            follow_log(cfg, None, wait=True, from_start=True)
        else:
            log("Forge KHÔNG chạy (runtime có thể đã bị Colab thu hồi / khởi động lại) → khởi chạy lại…")
            if not (Paths.forge / "launch.py").exists() or not Paths.python().exists():
                log("⚠ môi trường không còn (Colab đã reset máy) → cần chạy lại cell 3 → 4 → 6.")
                return
            launch(cfg)


def doctor(cfg: Config | None = None, n: int = 80) -> None:
    """Cell 🩺: tóm tắt trạng thái + 80 dòng log cuối của lần khởi chạy gần nhất."""
    cfg = cfg or Config.load()
    banner("🩺 Chẩn đoán")
    log(f"Python hệ thống {platform.python_version()} | GPU: {gpu_name()} | Colab: {is_colab()}")
    log(f"Forge: {'có' if (Paths.forge / 'launch.py').exists() else 'CHƯA cài'} | venv: {'có' if Paths.python().exists() else 'CHƯA có'}")
    free = shutil.disk_usage(str(ROOT)).free
    log(f"đĩa trống: {human(free)}")
    try:
        mem = [ln for ln in Path("/proc/meminfo").read_text().splitlines() if ln.startswith(("MemTotal", "MemAvailable"))]
        log("RAM: " + " | ".join(" ".join(m.split()) for m in mem))
    except Exception:  # noqa: BLE001
        pass
    running = forge_pids()
    log(f"Forge đang chạy ngầm: {running or 'không'} | cổng {cfg.port}: {'mở' if port_open(cfg.port) else 'đóng'}")
    for u in saved_urls().get("public", []):
        log(f"link gần nhất: {u}")
    model = dest_for(MODEL_FILES["noobai_xl_eps_1_1"])
    log(f"model: {model} → {'OK ' + human(model.stat().st_size) if model.exists() else 'CHƯA có'}")
    if LAUNCH_LOG.exists():
        lines = LAUNCH_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
        print(f"\n--- {n} dòng cuối của {LAUNCH_LOG} ({len(lines)} dòng) ---")
        for ln in lines[-n:]:
            print(ln[:300])
        text = "\n".join(lines)
        hints = [h for key, h in KNOWN_ERRORS if key in text]
        if hints:
            print("\nGợi ý:")
            for h in dict.fromkeys(hints):
                print(" •", h)
    else:
        print("\nChưa có log khởi chạy – hãy chạy cell 6 trước.")


# --------------------------------------------------------------------------------------
# Luồng chính
# --------------------------------------------------------------------------------------
def install(cfg: Config) -> dict:
    banner(f"NoobAI-XL EPS 1.1 Installer v{LIB_VERSION} – cài môi trường")
    log(f"Python hệ thống: {platform.python_version()}  |  GPU: {gpu_name()}  |  thư mục: {ROOT}")
    ensure_system_tools()
    ensure_python_env(cfg)
    install_forge(cfg)
    install_extensions(cfg)
    prepare_forge_environment(cfg)
    install_tag_csv(cfg)
    install_localization(cfg)
    install_noob_tools(cfg)
    drive_info = setup_drive(cfg)
    write_settings(cfg, drive_info)
    (ROOT / "noobai_drive.json").write_text(json.dumps(drive_info), encoding="utf-8")
    log("✔ cài đặt môi trường hoàn tất")
    return drive_info


def download(cfg: Config) -> None:
    download_resources(cfg)
    # ghi lại config.json để chắc chắn tên model/VAE trỏ đúng file đã tải
    drive_path = ROOT / "noobai_drive.json"
    drive_info = json.loads(drive_path.read_text()) if drive_path.exists() else {}
    write_settings(cfg, drive_info)


def set_ui_language(cfg: Config, language: str | None = None) -> str:
    """Đổi ngôn ngữ giao diện (vi/en) mà không cần cài lại; có hiệu lực sau khi khởi động lại / Reload UI."""
    if language:
        cfg.ui_language = language
    if str(cfg.ui_language).lower().startswith("vi"):
        install_localization(cfg)
    drive_path = ROOT / "noobai_drive.json"
    drive_info = json.loads(drive_path.read_text()) if drive_path.exists() else {}
    write_settings(cfg, drive_info)
    return json.loads((Paths.forge / "config.json").read_text(encoding="utf-8")).get("localization", "None")


def main(argv: list[str]) -> int:
    cfg = Config.load()
    step = argv[1] if len(argv) > 1 else "all"
    if step == "lang":
        print("localization =", set_ui_language(cfg, argv[2] if len(argv) > 2 else None))
        return 0
    if step == "settings":  # ghi lại config.json / ui-config.json / noob_tools.json theo noobai_config.json hiện tại
        install_localization(cfg)
        install_noob_tools(cfg)
        drive_path = ROOT / "noobai_drive.json"
        write_settings(cfg, json.loads(drive_path.read_text()) if drive_path.exists() else {})
        return 0
    if step in ("install", "all"):
        install(cfg)
    if step in ("download", "all"):
        download(cfg)
    if step in ("test", "all"):
        if not self_test(cfg) and step == "all":
            print("Có mục kiểm tra chưa đạt – vẫn thử khởi chạy, xem log để sửa.")
    if step in ("launch", "all"):
        launch(cfg)
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


In [ ]:
#@title 3️⃣ Cài đặt môi trường (Python 3.10 + Forge + extensions) – ~3–5 phút
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
drive_info = L.install(cfg)


In [ ]:
#@title 4️⃣ Tải model / VAE / upscaler / ControlNet – ~2–6 phút tuỳ lựa chọn
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
L.download(cfg)


In [ ]:
#@title 5️⃣ Tự kiểm tra trước khi chạy
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
ok = L.self_test(cfg)
if not ok:
    print("\n⚠ Có mục chưa đạt. Thường chỉ cần chạy lại cell 3 hoặc 4. Nếu vẫn lỗi, copy log ở trên để được hỗ trợ.")


In [ ]:
#@title 6️⃣ 🚀 Khởi chạy giao diện (link gradio.live sẽ hiện ở đây)
#@markdown Forge chạy **nền độc lập** với cell này: trình duyệt mất kết nối hay cell bị ngắt thì Forge vẫn sống, link vẫn dùng được – chạy cell 🔗 để lấy lại link. Cell này chỉ theo dõi log (sau khi UI lên chỉ in dòng quan trọng). Muốn dừng hẳn Forge: cell 🔗 với ACTION = Dừng.
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
if L.forge_alive(cfg):
    print("Forge đang chạy sẵn → chỉ lấy lại link (muốn khởi động lại: cell 🔗, ACTION = Khởi động lại)")
    L.reconnect(cfg, "link")
else:
    L.launch(cfg)


In [ ]:
#@title 🔗 Nối lại sau khi Colab mất kết nối: lấy lại link / khởi động lại / dừng
#@markdown - **Lấy lại link**: Forge còn sống → in lại link; đã chết → tự khởi chạy lại (môi trường còn thì ~1 phút).
#@markdown - Nếu Colab đã **reset máy** (mất /content) cell sẽ báo → chạy lại cell 3 → 4 → 6 (model trên Drive không phải tải lại nếu đã bật LINK_DRIVE_MODELS và để model trong Drive/NoobAI/models).
ACTION = "Lấy lại link"  #@param ["Lấy lại link", "Khởi động lại", "Dừng"]
import importlib, sys
sys.path.insert(0, "/content")
try:
    import noobai_lib as L
except ModuleNotFoundError:
    raise SystemExit("⚠ /content/noobai_lib.py không còn → Colab đã reset máy. Chạy lại từ cell 1 (cell 2 → 3 → 4 → 6).")
importlib.reload(L)
L.reconnect(L.Config.load(), {"Lấy lại link": "link", "Khởi động lại": "restart", "Dừng": "stop"}[ACTION])


In [ ]:
#@title ⏱ Chống Colab tự ngắt kết nối (đọc kỹ rồi làm theo)
#@markdown Colab miễn phí ngắt phiên khi **tab Colab không có tương tác ~90 phút** (dù bạn đang dùng gradio.live ở tab khác), tối đa 12 h/phiên, và có thể thu hồi GPU bất kỳ lúc nào. Cell này in **trạng thái** và **đoạn JS giữ phiên** để bạn dán vào Console của trình duyệt (F12 → Console) **trên tab Colab**. Lưu ý: tự động hoá tương tác có thể vi phạm điều khoản Colab – dùng có chừng mực, tự chịu trách nhiệm; cách an toàn nhất là thỉnh thoảng quay lại tab Colab bấm vào đâu đó, hoặc dùng Colab Pro (background execution).
import importlib, sys, time
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
print("Forge:", "đang chạy ✅" if L.forge_alive(cfg) else "KHÔNG chạy ❌ (chạy cell 🔗)")
info = L.saved_urls()
for u in info.get("public", []):
    print("Link:", u)
try:
    import subprocess
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip())
except Exception:
    pass
print("""
──── Dán vào Console (F12) của TAB COLAB rồi Enter – cứ 60 s sẽ "chạm" vào trang để Colab không coi là bỏ không ────
(function(){
  if (window._noobKeep) clearInterval(window._noobKeep);
  window._noobKeep = setInterval(function(){
    try {
      var btn = document.querySelector("colab-connect-button");
      if (btn && btn.shadowRoot) { var b = btn.shadowRoot.querySelector("#connect"); if (b) b.click(); }
      document.dispatchEvent(new MouseEvent("mousemove", {bubbles:true}));
      console.log("NoobAI keep-alive", new Date().toLocaleTimeString());
    } catch(e) { console.log("keep-alive lỗi", e); }
  }, 60000);
  console.log("NoobAI keep-alive: BẬT (tắt bằng clearInterval(window._noobKeep))");
})();
────────────────────────────────────────────────────────────────────────────────────────────────────────
Mẹo thêm: • Giữ tab Colab mở (không thu nhỏ trình duyệt). • Ảnh đã lưu thẳng vào Drive/NoobAI/outputs nên mất phiên cũng không mất ảnh.
• Mất kết nối rồi nối lại được (Runtime ▸ Reconnect) → chạy cell 🔗. • Báo "Your session crashed after using all available RAM":
  giảm batch size, đóng tab/ứng dụng khác của Colab, hoặc thêm `--always-low-vram` vào EXTRA_ARGS.""")


In [ ]:
#@title 🩺 Chẩn đoán khi không thấy link / Forge thoát sớm
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
L.doctor(L.Config.load())


## 📖 Mẹo dùng NoobAI-XL EPS 1.1

**Prompt chuẩn (đã điền sẵn trong UI):**
```
masterpiece, best quality, newest, absurdres, highres, <tag Danbooru của bạn…>
```
**Negative chính thức:**
```
worst quality, old, early, low quality, lowres, signature, username, logo, bad hands, mutated hands, mammal, anthro, furry, ambiguous form, feral, semi-anthro
```
- Sampler **Euler a**, 24–32 steps, **CFG 5–7**, Clip skip 2, kích thước ≈ 1 MP (832×1216, 1024×1024, 1216×832…).
- Nhân vật: gõ tên theo Danbooru, ví dụ `hatsune_miku, vocaloid` – Tag Autocomplete sẽ gợi ý từ **349 714 tag** của repo (gõ `_` hoặc chữ cái đầu, Tab để chọn).
- Phong cách hoạ sĩ: `artist:xxx` hoặc chỉ tên hoạ sĩ theo Danbooru. Năm: `newest / recent / mid / early / old`.
- Hires fix: 4x-AnimeSharp × 1.5, denoise 0.35–0.45 (đã đặt mặc định). ADetailer 3 bộ mặt + mắt + tay đã bật sẵn (dropdown 🩹).
- ControlNet Union ProMax: chọn model `xinsir_controlnet_union_sdxl_promax`, chọn preprocessor tương ứng (canny/depth/openpose/lineart/tile…).

## ⚙️ Thông số khuyên dùng hay tự chỉnh?
- Cell 1 `PARAM_MODE`: **Khuyên dùng** (bộ NoobAI chuẩn, bỏ qua các ô chi tiết) hoặc **Tự chỉnh** (dùng các ô Width/Height/Steps/CFG/Sampler/Hires/ADetailer).
- Trong UI, dropdown **⚙️ Thông số tạo ảnh** nằm ngay trên *Sampling method* (txt2img và img2img). Mỗi bộ đặt **toàn bộ** sampler · scheduler · steps · CFG · kích thước · batch · Hires fix (bật/tắt, upscaler, ×, steps, denoise, CFG):

| Bộ (txt2img) | Khi nào dùng | Thông số |
|---|---|---|
| ✅ Khuyên dùng – NoobAI chuẩn | mặc định, đa số trường hợp | Euler a · Automatic · 28 · CFG 6 · 832×1216 · Hires 4x-AnimeSharp ×1.5, 14 steps, denoise 0.4 |
| ⚡ Nháp nhanh | thử prompt / tìm seed | Euler a · 20 · CFG 5.5 · Hires tắt (nhanh gấp ~3) |
| 💎 Chất lượng cao | ảnh cuối, in ấn | DPM++ 2M SDE · Karras · 32 · CFG 6.5 · Hires ×2, 16 steps, denoise 0.35 |
| 🎨 Sáng tạo | muốn đa dạng, ít bám prompt | Euler a · 30 · CFG 4.5 · Hires ×1.5, denoise 0.45 |
| 🎯 Chính xác | bám prompt chặt, ít ngẫu nhiên | DPM++ 2M · Karras · 30 · CFG 7.5 · Hires ×1.5, denoise 0.3 |
| ✏️ Tuỳ chỉnh | tự đặt | tự động chuyển sang mục này khi bạn sửa tay bất kỳ thông số nào |

img2img có bộ riêng: *Khuyên dùng* (denoise 0.5) · *Nhẹ* (0.3, giữ ảnh) · *Mạnh* (0.7, vẽ lại nhiều) · *Phóng to* (DPM++ 2M Karras, 0.3 – dùng với Ultimate SD Upscale).

Ý nghĩa từng thông số: **Steps** – số bước vẽ (20 nháp, 28 chuẩn, 32–40 kỹ; hơn 40 gần như không đẹp thêm). **CFG** – độ bám prompt (4–5 sáng tạo, 6 chuẩn, 7–8 chặt; >9 cháy màu). **Sampler** – Euler a mềm/anime, DPM++ 2M (SDE) Karras sắc nét hơn. **Denoise** (Hires/img2img) – mức được phép thay đổi ảnh (0.3 giữ, 0.5 vừa, 0.7 vẽ lại). **Clip skip** = 2 cho NoobAI (thanh trên cùng). **Batch count/size** – số ảnh mỗi lần bấm (T4: batch size 1–2).

## 📐 Độ phân giải chuẩn
NoobAI-XL (SDXL) được huấn luyện ở ~1 megapixel – vẽ đúng các cỡ này sẽ ít lỗi tay/chân/bố cục nhất:

| Dọc | Ngang | Vuông |
|---|---|---|
| **832×1216** (2:3, khuyên dùng) | **1216×832** (3:2) | **1024×1024** |
| 896×1152 (7:9) | 1152×896 (9:7) | |
| 768×1344 (9:16 điện thoại) | 1344×768 (16:9 màn hình) | |
| 640×1536 (5:12) | 1536×640 (12:5 banner) | |

Trong UI: dropdown **📐 Độ phân giải chuẩn** nằm ngay dưới Width/Height (txt2img và img2img) – chọn là tự điền; kéo tay Width/Height thì dropdown chuyển sang *Tuỳ chỉnh*. Mặc định đặt ở cell 1 (`DEFAULT_RESOLUTION`).
> Đừng vẽ thẳng 2048×2048 – model sẽ sinh thừa người/thừa chi. Muốn ảnh to hãy vẽ ở cỡ chuẩn rồi **tăng độ phân giải** như dưới.

## 🔍 Tăng độ phân giải cho ảnh (3 cách, từ nhanh đến kỹ)
1. **Hires. fix – ngay khi tạo ảnh (mặc định đã bật ✨)**: ảnh được vẽ ở cỡ chuẩn, phóng to bằng *4x-AnimeSharp* × **1.5** (832×1216 → 1248×1824) rồi vẽ thêm chi tiết với *Denoising* **0.4**.
   - Muốn to hơn: *Phóng to gấp* 2.0 (→ 1664×2432, T4 mất ~2 phút/ảnh). Denoise 0.3 giữ sát ảnh gốc, 0.5 thêm chi tiết nhưng có thể đổi nét mặt.
   - Tắt cho một ảnh: bỏ tích ở tiêu đề mục *Hires. fix*. Mẹo: vẽ nháp tắt Hires fix, tìm được seed ưng ý thì ♻️ dùng lại seed và bật Hires fix.
2. **Extras – phóng to ảnh có sẵn, không vẽ lại (vài giây)**: tab *Công cụ thêm (Extras)* → kéo ảnh vào → *Upscaler 1* = `4x-AnimeSharp` (anime) hoặc `4x-UltraSharp` (chi tiết/thực), *Phóng to gấp* 2–4 → *Tạo ảnh*. Nhanh, giữ nguyên nội dung, nhưng không thêm chi tiết mới.
3. **Ultimate SD Upscale – phóng to + vẽ lại chi tiết theo ô (kỹ nhất, 4K được)**: ở gallery bấm *Gửi sang img2img* → trong img2img: *Mức khử nhiễu* **0.25–0.35**, kéo xuống *Script* chọn **Ultimate SD upscale** → *Target size type* = *Scale from image size*, *Scale* = 2, *Upscaler* = 4x-AnimeSharp, *Tile width/height* 1024, *Padding* 32, *Seams fix* = *Half tile offset pass* → *Tạo ảnh*. Có thể bật thêm ControlNet **Tile** (model `noob_sdxl_controlnet_tile` nếu đã tải) để bám sát ảnh gốc hơn.

## ✨ Tự fix (đã bật sẵn, chỉnh ở cell 1 hoặc dropdown 🩹 trong UI)
- Dropdown **🩹 Tự sửa mặt / mắt / tay (ADetailer)** nằm ngay dưới ⚙️ Thông số tạo ảnh. Một cú chọn đặt model, prompt, confidence, dilation, blur, denoise, padding cho **3 bộ ADetailer**:

| Mức | Bộ 1 – mặt | Bộ 2 – mắt | Bộ 3 – tay |
|---|---|---|---|
| ✅ **Chuẩn** (mặc định) | `face_yolov8s` · denoise 0.4 · conf 0.3 | `Anzhc_Eyes_seg_hd` (YOLO mắt anime) · denoise 0.35 · dilation 8 | `hand_yolov8s` · denoise **0.5** · conf 0.25 · dilation 12 · padding 48 |
| 🔥 **Mạnh** | denoise 0.45 · dilation 8 | denoise 0.45 · dilation 12 | denoise **0.6** · conf 0.2 · dilation 16 · padding 64 |
| 🪶 Nhẹ | denoise 0.35 | – | – |
| 👥 Nhiều người | conf 0.2 | conf 0.2 | conf 0.15 · denoise 0.5 |
| ⛔ Tắt | – | – | – |

- Prompt mỗi bộ bắt đầu bằng `[PROMPT]` = ADetailer tự chèn prompt/negative chính của ảnh (giữ màu mắt, kiểu tóc, phong cách) rồi mới thêm thẻ bộ phận (`detailed eyes, bright pupils…`, `detailed hands, five fingers…`). Sửa được trong mục *ADetailer* của UI – sửa tay là dropdown tự chuyển sang *Tuỳ chỉnh*.
- Vùng inpaint của ADetailer được khớp với bucket SDXL theo tỉ lệ bbox (*ad_match_inpaint_bbox_size = Strict*) → tay/mắt được vẽ lại ở ~1024 px thay vì bị kéo méo theo khung ảnh.
- **Vì sao trước đây sửa không ăn?** Bộ cũ dùng `hand_yolov8n` (hay bỏ sót tay anime) và denoise 0.4 cho tay – quá thấp để đổi số ngón; mắt không có bộ riêng nên chỉ được vẽ lại chung với mặt ở độ phân giải thấp. Mức **Chuẩn** mới dùng model `s` + denoise 0.5 cho tay và bộ mắt riêng; vẫn chưa đủ → chọn **🔥 Mạnh**, hoặc *Gửi sang inpaint* → tô tay/mắt → *Chỉ vùng tô*, denoise 0.5–0.6, batch count 4 rồi chọn ảnh đẹp nhất.
- **Hires. fix** như mục trên. Cả hai đều tắt được từng lần trong UI; muốn tắt hẳn: `AUTO_HIRES_FIX = False` / `AUTO_ADETAILER = off` ở cell 1 → chạy lại cell 1 → `!python /content/noobai_lib.py settings` (hoặc chạy lại cell 3) → cell 6.
- Ô Negative prompt mặc định giờ là NoobAI chuẩn **+ khối anatomy gọn** (`bad anatomy, bad proportions, extra digits, fewer digits, extra fingers, missing fingers, fused fingers, bad feet, extra limbs, cross-eyed, uneven eyes, worst aesthetic`); prompt mặc định thêm `very awa` (thẻ thẩm mỹ top 5 % của NoobAI).

## 🩹 Thẻ sửa MẮT – TAY – CHÂN (Styles có sẵn)
Chọn ở ô **Styles** (cạnh nút Tạo ảnh), có thể chọn nhiều style cùng lúc. Bảng đầy đủ + mẹo inpaint: [docs/prompt_fix_mat_tay_chan.md](https://github.com/manhlee1196-boop/anime2/blob/main/docs/prompt_fix_mat_tay_chan.md).

| Style | Thêm vào prompt | Thêm vào negative |
|---|---|---|
| **Noob ✦ Fix mắt** | `detailed eyes, beautiful detailed eyes, looking at viewer, eye contact, eye focus` | `bad eyes, cross-eyed, uneven eyes, asymmetrical eyes, extra eyes, missing eye, lazy eye, empty eyes, deformed eyes, misaligned eyes, blurry eyes, extra pupils` |
| **Noob ✦ Fix tay** | `detailed hands, five fingers, fingernails` | `bad hands, mutated hands, malformed hands, deformed hands, poorly drawn hands, extra digits, fewer digits, extra fingers, missing fingers, fused fingers, too many fingers, long fingers, extra hands, extra arms, missing arms, disconnected limbs, floating limbs` |
| **Noob ✦ Fix chân** | `detailed legs, detailed feet, toes` | `bad feet, bad legs, extra legs, missing legs, deformed feet, malformed feet, poorly drawn feet, fused toes, extra toes, missing toes, twisted legs, bad knees, disconnected legs, extra limbs, bad proportions` |
| **Noob ✦ Fix mắt+tay+chân** | cả 3 dòng trên | `bad anatomy, bad proportions, bad perspective` + mắt + tay + chân (rút gọn) |
| **Noob ✦ Fix mắt+tay+chân (gọn)** | `detailed eyes, detailed hands, detailed feet` | `bad anatomy, bad hands, bad feet, extra digits, fewer digits, extra limbs, cross-eyed, uneven eyes` |

Mẹo quan trọng hơn cả negative: **ghi rõ tư thế tay/chân** trong prompt – `hand on hip`, `hands up`, `v`, `holding cup`, `arms behind back`, `hands in pockets`, `standing`, `sitting`, `crossed legs`, `seiza`… Tay có việc hoặc có tư thế cụ thể thì ít lỗi hẳn; ảnh không cần tay/chân → `upper body` / `cowboy shot` để cắt khung. Mắt lệch → thêm `looking at viewer, eye contact`, giảm CFG 5–6. Bàn chân nhỏ ở ảnh full body → giữ Hires fix hoặc inpaint vùng chân với `detailed feet, toes` denoise 0.45–0.55.

## 🚀 Dùng tối đa GPU
- Cell 1 `GPU_MODE = max` (mặc định): Forge chạy với `--always-high-vram` (T4 15 GB) hoặc `--always-gpu` (L4/A100 ≥ 20 GB) → model, text encoder, VAE **nằm hẳn trên GPU**, không hoán đổi ra RAM giữa các bước; thêm `--cuda-malloc` và `--vae-in-fp16` (giải mã ảnh nhanh gấp ~2 trên T4 nhờ VAE fp16-fix). Cell 6 in rõ chế độ và lý do.
- Thanh trên cùng của UI: **Trọng số trên GPU (GPU Weights, MB)** – để càng cao càng nhanh (mặc định = VRAM − 1 GB). Nếu gặp *CUDA out of memory* ở ảnh lớn: hạ xuống ~2–3 GB dưới mức tối đa, hoặc đổi `GPU_MODE = lowvram`.
- Kiểm tra GPU đang dùng: cell ⏱ in `nvidia-smi` (tên GPU, VRAM đang dùng, % tải). Khi đang vẽ, % tải phải ≈ 95–100 %; nếu thấp và cell 6 có dòng *Moving model to CPU / swapping* → VRAM không đủ cho cấu hình hiện tại, giảm độ phân giải Hires hoặc tắt ControlNet.
- Tăng số ảnh/giờ trên T4: giữ *Batch size* 1–2, tăng *Batch count*; dùng bộ ⚡ Nháp nhanh để tìm seed rồi mới Hires. Sampler 1 bước ≈ 1,0–1,3 s ở 832×1216 trên T4; ảnh 28 steps + Hires ×1.5 + ADetailer ≈ 60–80 s.
- xformers đã bật (`USE_XFORMERS`). Trên L4/A100 có thể thử thêm `--cuda-stream` vào `EXTRA_ARGS` (hoán đổi và tính toán song song) – không khuyến nghị trên T4.

## 🛠 Xử lý sự cố
| Hiện tượng | Cách xử lý |
|---|---|
| `CUDA out of memory` với GPU_MODE=max | Hạ *GPU Weights* trên thanh đầu UI 2–3 GB, hoặc `GPU_MODE = balanced`/`lowvram` rồi chạy lại cell 1 → `!python /content/noobai_lib.py settings` → cell 🔗 Khởi động lại |
| **Colab tự ngắt kết nối** | Phân biệt 2 trường hợp: (a) chỉ mất kết nối trình duyệt (góc phải báo *Reconnect*, RAM/Disk vẫn hiện) → bấm Reconnect, chạy cell **🔗** là có lại link, Forge chưa hề tắt; (b) runtime bị thu hồi (máy mới, /content trống) → chạy lại cell 2 → 3 → 4 → 6 (ảnh đã nằm trên Drive). Nguyên nhân thường gặp: tab Colab bỏ không ~90 phút trong khi bạn dùng gradio.live ở tab khác → xem cell **⏱** |
| Link gradio.live cũ không mở được | Forge đã tắt theo runtime → cell 🔗; nếu Forge còn sống mà gradio.live lỗi → cell 🔗 ACTION = Khởi động lại, hoặc dùng link dự phòng |
| Không thấy link gradio.live | Cell 6 luôn in thêm **LINK DỰ PHÒNG (Colab proxy)** ngay khi UI lên – dùng link đó. Nếu cell 6 *kết thúc* (không chạy mãi) tức Forge đã thoát: chạy cell 🩺 để xem lỗi, hoặc đổi `TUNNEL = ngrok` |
| `CUDA out of memory` | Giảm kích thước ảnh / batch, tắt ControlNet, hoặc thêm `--always-low-vram` vào EXTRA_ARGS |
| Model không hiện trong danh sách | Chạy lại cell 4 (tải sẽ tiếp tục nếu bị đứt), bấm 🔄 Refresh trong UI |
| Colab báo "disallowed code" | Giới hạn của Colab miễn phí với WebUI – cân nhắc Colab Pro |
| Muốn đổi UI sang tiếng Anh / Việt | Trong UI: **Cài đặt ▸ Giao diện người dùng ▸ Ngôn ngữ (Localization)** chọn `None` (Anh) hoặc `vi_VN` (Việt) → *Áp dụng cài đặt* → *Tải lại UI*. Hoặc đặt `UI_LANGUAGE` ở cell 1 rồi chạy lại cell 1 → 3 → 6 |
| Thêm extension mới qua UI | Sau khi cài trong tab Extensions, chạy lại cell 3 rồi cell 6 (để installer chạy với ràng buộc phiên bản) |

Mọi logic cài đặt nằm trong `colab/noobai_lib.py` của repo – notebook này được sinh tự động bởi `colab/build_notebook.py`.


In [ ]:
#@title 🧰 (Tuỳ chọn) Tải thêm model / LoRA / embedding bất kỳ lúc nào { display-mode: "form" }
#@markdown Dán link rồi chạy. Sau đó trong WebUI bấm nút 🔄 Refresh cạnh danh sách model/LoRA là thấy ngay (không cần khởi động lại).
URL = ""  #@param {type:"string"}
KIND = "Lora"  #@param ["Lora", "Stable-diffusion", "VAE", "ControlNet", "embeddings", "ESRGAN"]
FILENAME = ""  #@param {type:"string"}
import importlib, sys
sys.path.insert(0, "/content")
import noobai_lib as L
importlib.reload(L)
cfg = L.Config.load()
for u in L.split_urls(URL):
    u = L.normalize_url(u)
    name = FILENAME.strip() or L._guess_filename(u)
    dest = L.dest_for(dict(kind=KIND, name=name))
    L.download_file(u, dest, cfg)
    print("→", dest)


In [ ]:
#@title 💾 (Tuỳ chọn) Nén toàn bộ ảnh đã tạo thành ZIP (để tải về / lưu Drive)
import shutil, time, json, os
from pathlib import Path
info_path = Path("/content/noobai_drive.json")
outputs = Path(json.loads(info_path.read_text())["outputs"]) if info_path.exists() else Path("/content/noobai-forge/outputs")
zip_base = f"/content/noobai_outputs_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(zip_base, "zip", outputs)
print("✔ Đã nén:", zip_base + ".zip", f"({os.path.getsize(zip_base + '.zip')/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception:
    pass
